# 歌詞＋音楽 → 3Dダンスモーション生成（Google Colab）

楽曲の音声・歌詞・ジャンルから、**歌詞の意味**と**音楽のリズム**の両方を反映した 3D ダンスモーション（SMPL 形式）を作ります。
映像化（実写・CG 動画の生成）は行いません。確認用に骨格アニメーションとマネキンのプレビュー動画だけを描画します。

| # | 工程 | 使うもの | 出力 |
|---|---|---|---|
| 1 | ボーカル分離 | Demucs（htdemucs） | ボーカル音源 |
| 2 | 歌詞の時刻合わせ | wav2vec2（日本語）＋ WhisperX の整列機能 | 単語ごとのタイムスタンプ |
| 3 | ビート・ダウンビート検出 | beat_this | 拍・小節頭 |
| 4 | 8カウントのグリッド | — | 8カウント単位のスロット表 |
| 5 | 振付台本 | LLM（OpenRouter） | ジェスチャーの JSON |
| 6 | ベースダンス（1回目） | FlowerDance | 曲全体のモーション |
| 7 | 歌詞ジェスチャー | MotionGPT3 ＋ 逆運動学 | 「決め」を含むジェスチャー断片 |
| 8 | 拍に合わせて埋め込み | — | 指定済みモーションとマスク |
| 9 | つなぎの作り直し（2回目） | FlowerDance のマスク付き生成 | 完成モーション |
| 10 | 書き出しとプレビュー | — | SMPL `.pkl`・BVH・骨格アニメーション |
| 11 | マネキン動画 | PyTorch のレイマーチング | プレビュー動画 |

**方針**
- すべて学習済みモデルの**推論のみ**（学習・ファインチューニングはしません）。
- **Google Drive は使いません。** モデルと中間データは `/content` に置き、成果物はブラウザからダウンロードします（セッションごとにモデルを再ダウンロード）。
- **ソースからのビルドは行いません。** mamba-ssm / causal-conv1d はビルド済みホイール、PyTorch3D は回転変換のファイルだけを同梱します。
- モデルは同時にロードせず、工程ごとにロード → 解放します（VRAM 節約）。

## 0. 使い方

1. **ランタイム → ランタイムのタイプを変更** で GPU（**L4 または A100** 推奨）を選びます。
2. 左の 🔑（シークレット）に次を登録し、「ノートブックからのアクセス」をオンにします。
   - `OPENROUTER_API_KEY`（必須）: [OpenRouter](https://openrouter.ai/) の API キー
   - `HF_TOKEN`（任意）: Hugging Face のトークン（ダウンロードの速度制限よけ）
3. 上から順にセルを実行します。**3. 入力** で音源・歌詞・ジャンルを指定します。
4. 途中で確認・修正できる箇所があります（歌詞のタイミング、8カウント、振付台本、マスクの方式）。
5. 最後に成果物を ZIP でダウンロードします。

> ランタイムを再起動したときは **1-1** と **1-7** を実行し直してから、続きのセルを実行してください（中間結果は `/content/work` に保存されているので、終わった工程はやり直す必要はありません）。

**所要時間の目安（L4・4分の曲）**: 環境構築 5〜8分 / モデルのダウンロード 3〜5分 / 解析 2〜3分 / ベースダンス 1分前後 / ジェスチャー 2〜4分 / マスク付き生成 1〜3分 / プレビュー描画 数分。

## 1. 環境構築

FlowerDance の `requirements.txt` は mamba-ssm・causal-conv1d・PyTorch3D を Git の特定コミットからソースビルドする指定になっています。Colab では長時間かかり失敗もしやすいため、次のように置き換えます。

| 依存 | 対応 |
|---|---|
| PyTorch | `torch==2.7.1`（CUDA 12.8 版）を入れ直す。ビルド済みなのでダウンロードのみ |
| mamba-ssm / causal-conv1d | PyPI 版のインストーラーと**同じ規則**（torch・CUDA・Python・C++ABI の組み合わせ）で GitHub リリースのビルド済みホイール URL を組み立て、**存在を確認してから直接インストール**。ホイールが無い場合はビルドに進まず、その場で止まります |
| PyTorch3D | 使っているのは回転変換（`pytorch3d.transforms`）だけ。純粋な PyTorch のコードなので該当ファイル（BSD ライセンス）を同梱 |
| essentia / wandb | コード中で使われていないため入れない |
| allin1 / madmom | 使わない（beat_this と librosa で代替） |
| MotionGPT3 | `transformers==4.47.1` などの内部仕様に依存するため、専用フォルダに入れて **MotionGPT3 を動かす別プロセスだけ**に見せる（本体の環境は汚さない） |

最終手段として、Mamba の選択的スキャンを **純 PyTorch の実装**（mamba-ssm の参照実装と同じ計算をチャンク単位で行うもの）に差し替える仕組みも入れてあります。

In [ ]:
# @title 1-1. 作業フォルダと GPU の確認
import os, sys, subprocess, json, shutil, time

ROOT = "/content"
WORK = f"{ROOT}/work"            # 中間データ・成果物
MODELS = f"{ROOT}/models"        # 学習済みモデル（セッションごとに再ダウンロード）
REPOS = f"{ROOT}/repos"          # FlowerDance / MotionGPT3 のソースコード
PKG = f"{ROOT}/m2d_pkg"          # このノートブックのコード（m2d パッケージと PyTorch3D の代替）
MGPT_SITE = f"{ROOT}/mgpt_site"  # MotionGPT3 専用の上書きパッケージ
FD_REPO = f"{REPOS}/FlowerDance"
MGPT_REPO = f"{REPOS}/MotionGPT3"
for d in (WORK, MODELS, REPOS, f"{PKG}/m2d", f"{PKG}/pytorch3d/transforms", MGPT_SITE):
    os.makedirs(d, exist_ok=True)
os.environ["TORCH_HOME"] = f"{MODELS}/torch"
os.environ["HF_HOME"] = f"{MODELS}/huggingface"
os.environ["NLTK_DATA"] = f"{MODELS}/nltk"
os.environ["TOKENIZERS_PARALLELISM"] = "false"

try:
    smi = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total,driver_version", "--format=csv,noheader"],
                         capture_output=True, text=True)
    gpu_info = smi.stdout.strip() if smi.returncode == 0 else ""
except FileNotFoundError:
    gpu_info = ""
if not gpu_info:
    raise SystemExit("GPU が見つかりません。「ランタイム → ランタイムのタイプを変更」で L4 / A100 などの GPU を選んでください。")
print("GPU   :", gpu_info)
print("Python:", sys.version.split()[0])
print("disk  :", shutil.disk_usage(ROOT).free // 2**30, "GB free")

In [ ]:
# @title 1-2. PyTorch 2.7.1（CUDA 12.8）を入れ直す
from importlib.metadata import version, PackageNotFoundError

TORCH_PINS = {"torch": "2.7.1", "torchvision": "0.22.1", "torchaudio": "2.7.1"}

def installed_version(pkg):
    try:
        return version(pkg).split("+")[0]
    except PackageNotFoundError:
        return None

need = [p for p, v in TORCH_PINS.items() if installed_version(p) != v]
if need:
    print("installing:", need)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *[f"{p}=={v}" for p, v in TORCH_PINS.items()],
                    "--index-url", "https://download.pytorch.org/whl/cu128"], check=True)
# 以降の pip が PyTorch を上書きしないための制約ファイル
with open(f"{ROOT}/constraints.txt", "w") as f:
    f.write("".join(f"{p}=={v}\n" for p, v in TORCH_PINS.items()))

probe = subprocess.run([sys.executable, "-c",
                        "import torch;print(torch.__version__, torch.version.cuda, torch.cuda.is_available())"],
                       capture_output=True, text=True, check=True).stdout.split()
print("torch", probe[0], "| CUDA", probe[1], "| GPU available:", probe[2])
assert probe[0].startswith("2.7.1") and probe[2] == "True", "PyTorch 2.7.1 + CUDA の確認に失敗しました"
if need and "torch" in sys.modules:
    print("このカーネルでは以前の torch が読み込み済みです。ランタイムを再起動します。再起動後に 1-1 から実行し直してください。")
    time.sleep(2)
    os.kill(os.getpid(), 9)

In [ ]:
# @title 1-3. Mamba（mamba-ssm / causal-conv1d）をビルド済みホイールで入れる
import urllib.request

MAMBA_VERSION = "2.2.6.post3"    # FlowerDance の requirements.txt が指すコミット（v2.2.6.post3）
CAUSAL_CONV1D_VERSION = "1.5.4"  # 指定コミット（1.5.3.post2 と 1.5.4 の間）に最も近い、ホイールのある版
ALLOW_TORCH_MAMBA_FALLBACK = True  # ホイールが無い・読み込めない場合に純 PyTorch 実装へ切り替える（最終手段・低速）

tv, cuda, abi, py = subprocess.run(
    [sys.executable, "-c", "import sys,torch;print(torch.__version__.split('+')[0], torch.version.cuda, "
     "torch._C._GLIBCXX_USE_CXX11_ABI, f'cp{sys.version_info.major}{sys.version_info.minor}')"],
    capture_output=True, text=True, check=True).stdout.split()
torch_mm = ".".join(tv.split(".")[:2])
cuda_major = cuda.split(".")[0]
abi = str(abi == "True").upper()

def wheel_url(repo, pkg, ver):
    # mamba-ssm / causal-conv1d の setup.py の get_wheel_url() と同じ命名規則
    name = f"{pkg}-{ver}+cu{cuda_major}torch{torch_mm}cxx11abi{abi}-{py}-{py}-linux_x86_64.whl"
    return f"https://github.com/{repo}/releases/download/v{ver}/{name}"

def url_exists(url):
    req = urllib.request.Request(url, headers={"Range": "bytes=0-0"})
    try:
        with urllib.request.urlopen(req, timeout=60) as r:
            return r.status in (200, 206)
    except Exception:
        return False

wheels = {"causal_conv1d": wheel_url("Dao-AILab/causal-conv1d", "causal_conv1d", CAUSAL_CONV1D_VERSION),
          "mamba_ssm": wheel_url("state-spaces/mamba", "mamba_ssm", MAMBA_VERSION)}
missing = [k for k, u in wheels.items() if not url_exists(u)]
MAMBA_BACKEND = "cuda"
if missing:
    msg = ("ビルド済みホイールが見つかりません: " + ", ".join(f"{k} ({wheels[k]})" for k in missing) +
           "\nソースビルドは長時間止まるため行いません。")
    if not ALLOW_TORCH_MAMBA_FALLBACK:
        raise SystemExit(msg)
    print(msg + "\n→ 最終手段として純 PyTorch 版の Mamba を使います（生成は遅くなります）。")
    MAMBA_BACKEND = "torch"
else:
    for k, u in wheels.items():
        print(k, "<-", u)
    # ホイールの URL を直接渡すので、ソースビルドは起こり得ない
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--no-deps", "--no-build-isolation",
                    wheels["causal_conv1d"], wheels["mamba_ssm"]], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-c", f"{ROOT}/constraints.txt",
                    "einops", "ninja", "packaging", "transformers"], check=True)
json.dump({"MAMBA_BACKEND": MAMBA_BACKEND}, open(f"{WORK}/env_state.json", "w"))
print("MAMBA_BACKEND =", MAMBA_BACKEND)

In [ ]:
# @title 1-4. その他のライブラリ
pip = [sys.executable, "-m", "pip", "install", "-q"]
# 本体の環境（PyTorch は制約ファイルで固定）
subprocess.run(pip + ["-c", f"{ROOT}/constraints.txt",
                      "librosa==0.11.0", "soundfile", "soxr", "einops", "scipy", "huggingface_hub", "safetensors",
                      "demucs==4.1.0", "beat-this==1.1.0", "rotary-embedding-torch",
                      "pykakasi", "nltk", "pandas", "gdown", "opencv-python-headless", "ipywidgets", "requests"],
               check=True)
# WhisperX は整列機能だけを使うので依存なしで入れる（ASR 用の faster-whisper や torch 2.8 指定を避ける）
subprocess.run(pip + ["--no-deps", "whisperx==3.8.6"], check=True)
# MotionGPT3 専用（PYTHONPATH の先頭に置いて MotionGPT3 のプロセスだけに見せる）
subprocess.run(pip + ["--no-deps", "--upgrade", "--target", MGPT_SITE,
                      "transformers==4.47.1", "tokenizers==0.21.0", "huggingface_hub==0.26.5",
                      "pytorch-lightning==2.4.0", "lightning-utilities==0.11.9", "torchmetrics==1.6.0",
                      "omegaconf==2.3.0", "antlr4-python3-runtime==4.9.3"], check=True)
# プレビューの日本語表示用フォント
subprocess.run("apt-get -qq update > /dev/null && apt-get -qq install -y fonts-noto-cjk > /dev/null", shell=True)
print("done")

In [ ]:
# @title 1-5. FlowerDance / MotionGPT3 のソースを取得（動作確認したコミットに固定）
REPO_PINS = {
    FD_REPO: ("https://github.com/XulongT/FlowerDance", "4431435d17fb84e04057e9cf50a8e954434ced73"),
    MGPT_REPO: ("https://github.com/OpenMotionLab/MotionGPT3", "cf5bab7daa568ce472e7201cc2bf8b83d8c1b732"),
}

def git(*args, check=True):
    return subprocess.run(["git", *args], capture_output=True, text=True, check=check)

for dst, (url, commit) in REPO_PINS.items():
    if not os.path.isdir(f"{dst}/.git"):
        git("init", "-q", dst)
        git("-C", dst, "remote", "add", "origin", url)
    if git("-C", dst, "fetch", "-q", "--depth", "1", "origin", commit, check=False).returncode == 0:
        git("-C", dst, "checkout", "-q", "-f", commit)
    else:
        print(f"[warn] {url} の固定コミットを取得できないため既定ブランチを使います")
        git("-C", dst, "fetch", "-q", "--depth", "1", "origin")
        git("-C", dst, "checkout", "-q", "-f", "FETCH_HEAD")
    print(os.path.basename(dst), git("-C", dst, "rev-parse", "--short", "HEAD").stdout.strip())

### 1-6. パイプラインのコード（`m2d` パッケージ）

以下のセルは、このノートブックで使う処理を `/content/m2d_pkg` にファイルとして書き出します（実行しても何も起きないように見えますが、ファイルが作られます）。

| ファイル | 内容 |
|---|---|
| `pytorch3d/transforms/*` | PyTorch3D の回転変換（同梱、ビルド不要） |
| `m2d/smpl_utils.py` | SMPL 骨格（FlowerDance と同一の親子関係・オフセット）、順運動学、151 次元表現の変換、y-up ↔ z-up |
| `m2d/mamba_fallback.py` | 純 PyTorch 版 Mamba（最終手段。パラメータ名が同じなので重みをそのまま読める） |
| `m2d/flowerdance_runner.py` | FlowerDance の読み込み、音楽特徴量、曲全体・区間分割の生成、マスク付き生成 |
| `m2d/audio_analysis.py` | Demucs、beat_this、8カウントのグリッド |
| `m2d/lyrics_align.py` | 歌詞の強制整列（全体 CTC 整列 → WhisperX で行ごとに精密化）と手動補正 |
| `m2d/choreo_llm.py` | OpenRouter で振付台本を作る（JSON の検証と再試行） |
| `m2d/mgpt_runner.py` | MotionGPT3 を別プロセスで動かす |
| `m2d/motion_ik.py` | HumanML3D（22 関節位置・20fps）→ SMPL 回転（30fps）の変換（逆運動学） |
| `m2d/gesture_pipeline.py` | 候補の選択、「決め」の検出と切り出し、誇張 |
| `m2d/gesture_embed.py` | 拍に合わせた埋め込み、マスクとウィンドウの作成 |
| `m2d/export_render.py` | 書き出し（pkl / BVH）、骨格アニメーション、マネキン描画 |
| `m2d/viz.py` | 確認用の図と診断 |

In [ ]:
%%writefile /content/m2d_pkg/pytorch3d/__init__.py
# Minimal pure-PyTorch stand-in for PyTorch3D (only `pytorch3d.transforms` is provided).
# FlowerDance only uses rotation conversions from PyTorch3D, so building PyTorch3D from source is unnecessary.
__version__ = "0.7.8+m2d_shim"

In [ ]:
%%writefile /content/m2d_pkg/pytorch3d/transforms/__init__.py
# Pure-PyTorch subset of pytorch3d.transforms (BSD License, see rotation_conversions.py header).
import math

import torch

from .rotation_conversions import (
    axis_angle_to_matrix,
    axis_angle_to_quaternion,
    euler_angles_to_matrix,
    matrix_to_axis_angle,
    matrix_to_euler_angles,
    matrix_to_quaternion,
    matrix_to_rotation_6d,
    quaternion_apply,
    quaternion_invert,
    quaternion_multiply,
    quaternion_raw_multiply,
    quaternion_to_axis_angle,
    quaternion_to_matrix,
    random_quaternions,
    random_rotation,
    random_rotations,
    rotation_6d_to_matrix,
    standardize_quaternion,
)
from .rotation_conversions import _axis_angle_rotation


class RotateAxisAngle:
    """Small replacement for pytorch3d.transforms.RotateAxisAngle (only transform_points is supported).

    Like PyTorch3D, points are treated as row vectors, so the stored matrix is the transpose of the
    column-vector rotation matrix and `transform_points(p) == p @ R_row`.
    """

    def __init__(self, angle, axis="X", degrees=True, dtype=torch.float32, device=None):
        angle = torch.as_tensor(angle, dtype=dtype, device=device)
        if degrees:
            angle = angle / 180.0 * math.pi
        self._R = _axis_angle_rotation(axis, angle).transpose(-1, -2)

    def get_matrix(self):
        m = torch.eye(4, dtype=self._R.dtype, device=self._R.device)
        m[:3, :3] = self._R
        return m[None]

    def transform_points(self, points):
        return points @ self._R.to(points)


__all__ = [k for k in dir() if not k.startswith("_")]

In [ ]:
%%writefile /content/m2d_pkg/pytorch3d/transforms/rotation_conversions.py
# Vendored from facebookresearch/pytorch3d @ 75ebeeaea0908c5527e7b1e305fbc7681382db47
# (pytorch3d/transforms/rotation_conversions.py, BSD License). Pure PyTorch, no build needed.
# Only change: the `Device` type alias is defined locally instead of importing pytorch3d.common.
# Copyright (c) Meta Platforms, Inc. and affiliates.
# All rights reserved.
#
# This source code is licensed under the BSD-style license found in the
# LICENSE file in the root directory of this source tree.

# pyre-unsafe

from typing import Optional

import torch
import torch.nn.functional as F

from typing import Union

Device = Union[str, torch.device]  # m2d shim: replaces pytorch3d.common.datatypes.Device


"""
The transformation matrices returned from the functions in this file assume
the points on which the transformation will be applied are column vectors.
i.e. the R matrix is structured as

    R = [
            [Rxx, Rxy, Rxz],
            [Ryx, Ryy, Ryz],
            [Rzx, Rzy, Rzz],
        ]  # (3, 3)

This matrix can be applied to column vectors by post multiplication
by the points e.g.

    points = [[0], [1], [2]]  # (3 x 1) xyz coordinates of a point
    transformed_points = R * points

To apply the same matrix to points which are row vectors, the R matrix
can be transposed and pre multiplied by the points:

e.g.
    points = [[0, 1, 2]]  # (1 x 3) xyz coordinates of a point
    transformed_points = points * R.transpose(1, 0)
"""


def quaternion_to_matrix(quaternions: torch.Tensor) -> torch.Tensor:
    """
    Convert rotations given as quaternions to rotation matrices.

    Args:
        quaternions: quaternions with real part first,
            as tensor of shape (..., 4).

    Returns:
        Rotation matrices as tensor of shape (..., 3, 3).
    """
    r, i, j, k = torch.unbind(quaternions, -1)
    # pyre-fixme[58]: `/` is not supported for operand types `float` and `Tensor`.
    two_s = 2.0 / (quaternions * quaternions).sum(-1)

    o = torch.stack(
        (
            1 - two_s * (j * j + k * k),
            two_s * (i * j - k * r),
            two_s * (i * k + j * r),
            two_s * (i * j + k * r),
            1 - two_s * (i * i + k * k),
            two_s * (j * k - i * r),
            two_s * (i * k - j * r),
            two_s * (j * k + i * r),
            1 - two_s * (i * i + j * j),
        ),
        -1,
    )
    return o.reshape(quaternions.shape[:-1] + (3, 3))


def _copysign(a: torch.Tensor, b: torch.Tensor) -> torch.Tensor:
    """
    Return a tensor where each element has the absolute value taken from the,
    corresponding element of a, with sign taken from the corresponding
    element of b. This is like the standard copysign floating-point operation,
    but is not careful about negative 0 and NaN.

    Args:
        a: source tensor.
        b: tensor whose signs will be used, of the same shape as a.

    Returns:
        Tensor of the same shape as a with the signs of b.
    """
    signs_differ = (a < 0) != (b < 0)
    return torch.where(signs_differ, -a, a)


def _sqrt_positive_part(x: torch.Tensor) -> torch.Tensor:
    """
    Returns torch.sqrt(torch.max(0, x))
    but with a zero subgradient where x is 0.
    """
    ret = torch.zeros_like(x)
    positive_mask = x > 0
    if torch.is_grad_enabled():
        ret[positive_mask] = torch.sqrt(x[positive_mask])
    else:
        ret = torch.where(positive_mask, torch.sqrt(x), ret)
    return ret


def matrix_to_quaternion(matrix: torch.Tensor) -> torch.Tensor:
    """
    Convert rotations given as rotation matrices to quaternions.

    Args:
        matrix: Rotation matrices as tensor of shape (..., 3, 3).

    Returns:
        quaternions with real part first, as tensor of shape (..., 4).
    """
    if matrix.size(-1) != 3 or matrix.size(-2) != 3:
        raise ValueError(f"Invalid rotation matrix shape {matrix.shape}.")

    batch_dim = matrix.shape[:-2]
    m00, m01, m02, m10, m11, m12, m20, m21, m22 = torch.unbind(
        matrix.reshape(batch_dim + (9,)), dim=-1
    )

    q_abs = _sqrt_positive_part(
        torch.stack(
            [
                1.0 + m00 + m11 + m22,
                1.0 + m00 - m11 - m22,
                1.0 - m00 + m11 - m22,
                1.0 - m00 - m11 + m22,
            ],
            dim=-1,
        )
    )

    # we produce the desired quaternion multiplied by each of r, i, j, k
    quat_by_rijk = torch.stack(
        [
            # pyre-fixme[58]: `**` is not supported for operand types `Tensor` and
            #  `int`.
            torch.stack([q_abs[..., 0] ** 2, m21 - m12, m02 - m20, m10 - m01], dim=-1),
            # pyre-fixme[58]: `**` is not supported for operand types `Tensor` and
            #  `int`.
            torch.stack([m21 - m12, q_abs[..., 1] ** 2, m10 + m01, m02 + m20], dim=-1),
            # pyre-fixme[58]: `**` is not supported for operand types `Tensor` and
            #  `int`.
            torch.stack([m02 - m20, m10 + m01, q_abs[..., 2] ** 2, m12 + m21], dim=-1),
            # pyre-fixme[58]: `**` is not supported for operand types `Tensor` and
            #  `int`.
            torch.stack([m10 - m01, m20 + m02, m21 + m12, q_abs[..., 3] ** 2], dim=-1),
        ],
        dim=-2,
    )

    # We floor here at 0.1 but the exact level is not important; if q_abs is small,
    # the candidate won't be picked.
    flr = torch.tensor(0.1).to(dtype=q_abs.dtype, device=q_abs.device)
    quat_candidates = quat_by_rijk / (2.0 * q_abs[..., None].max(flr))

    # if not for numerical problems, quat_candidates[i] should be same (up to a sign),
    # forall i; we pick the best-conditioned one (with the largest denominator)
    out = quat_candidates[
        F.one_hot(q_abs.argmax(dim=-1), num_classes=4) > 0.5, :
    ].reshape(batch_dim + (4,))
    return standardize_quaternion(out)


def _axis_angle_rotation(axis: str, angle: torch.Tensor) -> torch.Tensor:
    """
    Return the rotation matrices for one of the rotations about an axis
    of which Euler angles describe, for each value of the angle given.

    Args:
        axis: Axis label "X" or "Y or "Z".
        angle: any shape tensor of Euler angles in radians

    Returns:
        Rotation matrices as tensor of shape (..., 3, 3).
    """

    cos = torch.cos(angle)
    sin = torch.sin(angle)
    one = torch.ones_like(angle)
    zero = torch.zeros_like(angle)

    if axis == "X":
        R_flat = (one, zero, zero, zero, cos, -sin, zero, sin, cos)
    elif axis == "Y":
        R_flat = (cos, zero, sin, zero, one, zero, -sin, zero, cos)
    elif axis == "Z":
        R_flat = (cos, -sin, zero, sin, cos, zero, zero, zero, one)
    else:
        raise ValueError("letter must be either X, Y or Z.")

    return torch.stack(R_flat, -1).reshape(angle.shape + (3, 3))


def euler_angles_to_matrix(euler_angles: torch.Tensor, convention: str) -> torch.Tensor:
    """
    Convert rotations given as Euler angles in radians to rotation matrices.

    Args:
        euler_angles: Euler angles in radians as tensor of shape (..., 3).
        convention: Convention string of three uppercase letters from
            {"X", "Y", and "Z"}.

    Returns:
        Rotation matrices as tensor of shape (..., 3, 3).
    """
    if euler_angles.dim() == 0 or euler_angles.shape[-1] != 3:
        raise ValueError("Invalid input euler angles.")
    if len(convention) != 3:
        raise ValueError("Convention must have 3 letters.")
    if convention[1] in (convention[0], convention[2]):
        raise ValueError(f"Invalid convention {convention}.")
    for letter in convention:
        if letter not in ("X", "Y", "Z"):
            raise ValueError(f"Invalid letter {letter} in convention string.")
    matrices = [
        _axis_angle_rotation(c, e)
        for c, e in zip(convention, torch.unbind(euler_angles, -1))
    ]
    # return functools.reduce(torch.matmul, matrices)
    return torch.matmul(torch.matmul(matrices[0], matrices[1]), matrices[2])


def _angle_from_tan(
    axis: str, other_axis: str, data, horizontal: bool, tait_bryan: bool
) -> torch.Tensor:
    """
    Extract the first or third Euler angle from the two members of
    the matrix which are positive constant times its sine and cosine.

    Args:
        axis: Axis label "X" or "Y or "Z" for the angle we are finding.
        other_axis: Axis label "X" or "Y or "Z" for the middle axis in the
            convention.
        data: Rotation matrices as tensor of shape (..., 3, 3).
        horizontal: Whether we are looking for the angle for the third axis,
            which means the relevant entries are in the same row of the
            rotation matrix. If not, they are in the same column.
        tait_bryan: Whether the first and third axes in the convention differ.

    Returns:
        Euler Angles in radians for each matrix in data as a tensor
        of shape (...).
    """

    i1, i2 = {"X": (2, 1), "Y": (0, 2), "Z": (1, 0)}[axis]
    if horizontal:
        i2, i1 = i1, i2
    even = (axis + other_axis) in ["XY", "YZ", "ZX"]
    if horizontal == even:
        return torch.atan2(data[..., i1], data[..., i2])
    if tait_bryan:
        return torch.atan2(-data[..., i2], data[..., i1])
    return torch.atan2(data[..., i2], -data[..., i1])


def _index_from_letter(letter: str) -> int:
    if letter == "X":
        return 0
    if letter == "Y":
        return 1
    if letter == "Z":
        return 2
    raise ValueError("letter must be either X, Y or Z.")


def matrix_to_euler_angles(matrix: torch.Tensor, convention: str) -> torch.Tensor:
    """
    Convert rotations given as rotation matrices to Euler angles in radians.

    Args:
        matrix: Rotation matrices as tensor of shape (..., 3, 3).
        convention: Convention string of three uppercase letters.

    Returns:
        Euler angles in radians as tensor of shape (..., 3).
    """
    if len(convention) != 3:
        raise ValueError("Convention must have 3 letters.")
    if convention[1] in (convention[0], convention[2]):
        raise ValueError(f"Invalid convention {convention}.")
    for letter in convention:
        if letter not in ("X", "Y", "Z"):
            raise ValueError(f"Invalid letter {letter} in convention string.")
    if matrix.size(-1) != 3 or matrix.size(-2) != 3:
        raise ValueError(f"Invalid rotation matrix shape {matrix.shape}.")
    i0 = _index_from_letter(convention[0])
    i2 = _index_from_letter(convention[2])
    tait_bryan = i0 != i2
    if tait_bryan:
        central_angle = torch.asin(
            matrix[..., i0, i2] * (-1.0 if i0 - i2 in [-1, 2] else 1.0)
        )
    else:
        central_angle = torch.acos(matrix[..., i0, i0])

    o = (
        _angle_from_tan(
            convention[0], convention[1], matrix[..., i2], False, tait_bryan
        ),
        central_angle,
        _angle_from_tan(
            convention[2], convention[1], matrix[..., i0, :], True, tait_bryan
        ),
    )
    return torch.stack(o, -1)


def random_quaternions(
    n: int, dtype: Optional[torch.dtype] = None, device: Optional[Device] = None
) -> torch.Tensor:
    """
    Generate random quaternions representing rotations,
    i.e. versors with nonnegative real part.

    Args:
        n: Number of quaternions in a batch to return.
        dtype: Type to return.
        device: Desired device of returned tensor. Default:
            uses the current device for the default tensor type.

    Returns:
        Quaternions as tensor of shape (N, 4).
    """
    if isinstance(device, str):
        device = torch.device(device)
    o = torch.randn((n, 4), dtype=dtype, device=device)
    s = (o * o).sum(1)
    o = o / _copysign(torch.sqrt(s), o[:, 0])[:, None]
    return o


def random_rotations(
    n: int, dtype: Optional[torch.dtype] = None, device: Optional[Device] = None
) -> torch.Tensor:
    """
    Generate random rotations as 3x3 rotation matrices.

    Args:
        n: Number of rotation matrices in a batch to return.
        dtype: Type to return.
        device: Device of returned tensor. Default: if None,
            uses the current device for the default tensor type.

    Returns:
        Rotation matrices as tensor of shape (n, 3, 3).
    """
    quaternions = random_quaternions(n, dtype=dtype, device=device)
    return quaternion_to_matrix(quaternions)


def random_rotation(
    dtype: Optional[torch.dtype] = None, device: Optional[Device] = None
) -> torch.Tensor:
    """
    Generate a single random 3x3 rotation matrix.

    Args:
        dtype: Type to return
        device: Device of returned tensor. Default: if None,
            uses the current device for the default tensor type

    Returns:
        Rotation matrix as tensor of shape (3, 3).
    """
    return random_rotations(1, dtype, device)[0]


def standardize_quaternion(quaternions: torch.Tensor) -> torch.Tensor:
    """
    Convert a unit quaternion to a standard form: one in which the real
    part is non negative.

    Args:
        quaternions: Quaternions with real part first,
            as tensor of shape (..., 4).

    Returns:
        Standardized quaternions as tensor of shape (..., 4).
    """
    return torch.where(quaternions[..., 0:1] < 0, -quaternions, quaternions)


def quaternion_raw_multiply(a: torch.Tensor, b: torch.Tensor) -> torch.Tensor:
    """
    Multiply two quaternions.
    Usual torch rules for broadcasting apply.

    Args:
        a: Quaternions as tensor of shape (..., 4), real part first.
        b: Quaternions as tensor of shape (..., 4), real part first.

    Returns:
        The product of a and b, a tensor of quaternions shape (..., 4).
    """
    aw, ax, ay, az = torch.unbind(a, -1)
    bw, bx, by, bz = torch.unbind(b, -1)
    ow = aw * bw - ax * bx - ay * by - az * bz
    ox = aw * bx + ax * bw + ay * bz - az * by
    oy = aw * by - ax * bz + ay * bw + az * bx
    oz = aw * bz + ax * by - ay * bx + az * bw
    return torch.stack((ow, ox, oy, oz), -1)


def quaternion_multiply(a: torch.Tensor, b: torch.Tensor) -> torch.Tensor:
    """
    Multiply two quaternions representing rotations, returning the quaternion
    representing their composition, i.e. the versor with nonnegative real part.
    Usual torch rules for broadcasting apply.

    Args:
        a: Quaternions as tensor of shape (..., 4), real part first.
        b: Quaternions as tensor of shape (..., 4), real part first.

    Returns:
        The product of a and b, a tensor of quaternions of shape (..., 4).
    """
    ab = quaternion_raw_multiply(a, b)
    return standardize_quaternion(ab)


def quaternion_invert(quaternion: torch.Tensor) -> torch.Tensor:
    """
    Given a quaternion representing rotation, get the quaternion representing
    its inverse.

    Args:
        quaternion: Quaternions as tensor of shape (..., 4), with real part
            first, which must be versors (unit quaternions).

    Returns:
        The inverse, a tensor of quaternions of shape (..., 4).
    """

    scaling = torch.tensor([1, -1, -1, -1], device=quaternion.device)
    return quaternion * scaling


def quaternion_apply(quaternion: torch.Tensor, point: torch.Tensor) -> torch.Tensor:
    """
    Apply the rotation given by a quaternion to a 3D point.
    Usual torch rules for broadcasting apply.

    Args:
        quaternion: Tensor of quaternions, real part first, of shape (..., 4).
        point: Tensor of 3D points of shape (..., 3).

    Returns:
        Tensor of rotated points of shape (..., 3).
    """
    if point.size(-1) != 3:
        raise ValueError(f"Points are not in 3D, {point.shape}.")
    real_parts = point.new_zeros(point.shape[:-1] + (1,))
    point_as_quaternion = torch.cat((real_parts, point), -1)
    out = quaternion_raw_multiply(
        quaternion_raw_multiply(quaternion, point_as_quaternion),
        quaternion_invert(quaternion),
    )
    return out[..., 1:]


def axis_angle_to_matrix(axis_angle: torch.Tensor) -> torch.Tensor:
    """
    Convert rotations given as axis/angle to rotation matrices.

    Args:
        axis_angle: Rotations given as a vector in axis angle form,
            as a tensor of shape (..., 3), where the magnitude is
            the angle turned anticlockwise in radians around the
            vector's direction.

    Returns:
        Rotation matrices as tensor of shape (..., 3, 3).
    """
    return quaternion_to_matrix(axis_angle_to_quaternion(axis_angle))


def matrix_to_axis_angle(matrix: torch.Tensor) -> torch.Tensor:
    """
    Convert rotations given as rotation matrices to axis/angle.

    Args:
        matrix: Rotation matrices as tensor of shape (..., 3, 3).

    Returns:
        Rotations given as a vector in axis angle form, as a tensor
            of shape (..., 3), where the magnitude is the angle
            turned anticlockwise in radians around the vector's
            direction.
    """
    return quaternion_to_axis_angle(matrix_to_quaternion(matrix))


def axis_angle_to_quaternion(axis_angle: torch.Tensor) -> torch.Tensor:
    """
    Convert rotations given as axis/angle to quaternions.

    Args:
        axis_angle: Rotations given as a vector in axis angle form,
            as a tensor of shape (..., 3), where the magnitude is
            the angle turned anticlockwise in radians around the
            vector's direction.

    Returns:
        quaternions with real part first, as tensor of shape (..., 4).
    """
    angles = torch.norm(axis_angle, p=2, dim=-1, keepdim=True)
    half_angles = angles * 0.5
    eps = 1e-6
    small_angles = angles.abs() < eps
    sin_half_angles_over_angles = torch.empty_like(angles)
    sin_half_angles_over_angles[~small_angles] = (
        torch.sin(half_angles[~small_angles]) / angles[~small_angles]
    )
    # for x small, sin(x/2) is about x/2 - (x/2)^3/6
    # so sin(x/2)/x is about 1/2 - (x*x)/48
    sin_half_angles_over_angles[small_angles] = (
        0.5 - (angles[small_angles] * angles[small_angles]) / 48
    )
    quaternions = torch.cat(
        [torch.cos(half_angles), axis_angle * sin_half_angles_over_angles], dim=-1
    )
    return quaternions


def quaternion_to_axis_angle(quaternions: torch.Tensor) -> torch.Tensor:
    """
    Convert rotations given as quaternions to axis/angle.

    Args:
        quaternions: quaternions with real part first,
            as tensor of shape (..., 4).

    Returns:
        Rotations given as a vector in axis angle form, as a tensor
            of shape (..., 3), where the magnitude is the angle
            turned anticlockwise in radians around the vector's
            direction.
    """
    norms = torch.norm(quaternions[..., 1:], p=2, dim=-1, keepdim=True)
    half_angles = torch.atan2(norms, quaternions[..., :1])
    angles = 2 * half_angles
    eps = 1e-6
    small_angles = angles.abs() < eps
    sin_half_angles_over_angles = torch.empty_like(angles)
    sin_half_angles_over_angles[~small_angles] = (
        torch.sin(half_angles[~small_angles]) / angles[~small_angles]
    )
    # for x small, sin(x/2) is about x/2 - (x/2)^3/6
    # so sin(x/2)/x is about 1/2 - (x*x)/48
    sin_half_angles_over_angles[small_angles] = (
        0.5 - (angles[small_angles] * angles[small_angles]) / 48
    )
    return quaternions[..., 1:] / sin_half_angles_over_angles


def rotation_6d_to_matrix(d6: torch.Tensor) -> torch.Tensor:
    """
    Converts 6D rotation representation by Zhou et al. [1] to rotation matrix
    using Gram--Schmidt orthogonalization per Section B of [1].
    Args:
        d6: 6D rotation representation, of size (*, 6)

    Returns:
        batch of rotation matrices of size (*, 3, 3)

    [1] Zhou, Y., Barnes, C., Lu, J., Yang, J., & Li, H.
    On the Continuity of Rotation Representations in Neural Networks.
    IEEE Conference on Computer Vision and Pattern Recognition, 2019.
    Retrieved from http://arxiv.org/abs/1812.07035
    """

    a1, a2 = d6[..., :3], d6[..., 3:]
    b1 = F.normalize(a1, dim=-1)
    b2 = a2 - (b1 * a2).sum(-1, keepdim=True) * b1
    b2 = F.normalize(b2, dim=-1)
    b3 = torch.cross(b1, b2, dim=-1)
    return torch.stack((b1, b2, b3), dim=-2)


def matrix_to_rotation_6d(matrix: torch.Tensor) -> torch.Tensor:
    """
    Converts rotation matrices to 6D rotation representation by Zhou et al. [1]
    by dropping the last row. Note that 6D representation is not unique.
    Args:
        matrix: batch of rotation matrices of size (*, 3, 3)

    Returns:
        6D rotation representation, of size (*, 6)

    [1] Zhou, Y., Barnes, C., Lu, J., Yang, J., & Li, H.
    On the Continuity of Rotation Representations in Neural Networks.
    IEEE Conference on Computer Vision and Pattern Recognition, 2019.
    Retrieved from http://arxiv.org/abs/1812.07035
    """
    batch_dim = matrix.size()[:-2]
    return matrix[..., :2, :].clone().reshape(batch_dim + (6,))

In [ ]:
%%writefile /content/m2d_pkg/m2d/__init__.py
"""music2dance (m2d): lyrics + music -> 3D dance (SMPL) pipeline used by music_to_dance.ipynb."""
__version__ = "1.0.0"

In [ ]:
%%writefile /content/m2d_pkg/m2d/smpl_utils.py
"""SMPL skeleton helpers shared by every stage.

Conventions
-----------
* Joint order / parents / rest offsets are exactly FlowerDance's ``vis.py`` (SMPL 24 joints).
  ``check_against_flowerdance`` verifies this against the cloned repository at runtime.
* Rotation matrices act on column vectors. 6D rotations follow PyTorch3D
  (``matrix_to_rotation_6d`` = first two *rows* of the matrix).
* "y-up" is the native SMPL / HumanML3D frame (x = body left, y = up, z = forward).
  "z-up" is FlowerDance's model frame: FlowerDance rotates the data by +90 deg about X
  ((x, y, z) -> (x, -z, y)) before training, so the root rotation is left-multiplied by
  ``R_ZUP_FROM_YUP`` and the root position is rotated by it. Local joint rotations are unchanged.
* FlowerDance motion vector (151 dims): [foot contacts (4) | root position (3) | 24 x 6D rotation].
"""
import numpy as np
import torch
import torch.nn.functional as F
from pytorch3d.transforms import (axis_angle_to_matrix, matrix_to_axis_angle,
                                  matrix_to_rotation_6d, rotation_6d_to_matrix)

FPS = 30
N_JOINTS = 24
MOTION_DIM = 151
CONTACT_SLICE = slice(0, 4)
ROOT_POS_SLICE = slice(4, 7)
ROT_OFFSET = 7

JOINT_NAMES = [
    "pelvis", "l_hip", "r_hip", "spine1", "l_knee", "r_knee", "spine2", "l_ankle",
    "r_ankle", "spine3", "l_foot", "r_foot", "neck", "l_collar", "r_collar", "head",
    "l_shoulder", "r_shoulder", "l_elbow", "r_elbow", "l_wrist", "r_wrist", "l_hand", "r_hand",
]

SMPL_PARENTS = [-1, 0, 0, 0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 9, 9, 12, 13, 14, 16, 17, 18, 19, 20, 21]

# Copied verbatim from FlowerDance/vis.py (smpl_offsets), y-up SMPL rest pose, metres.
SMPL_OFFSETS = np.array([
    [0.0, 0.0, 0.0],
    [0.05858135, -0.08228004, -0.01766408],
    [-0.06030973, -0.09051332, -0.01354254],
    [0.00443945, 0.12440352, -0.03838522],
    [0.04345142, -0.38646945, 0.008037],
    [-0.04325663, -0.38368791, -0.00484304],
    [0.00448844, 0.1379564, 0.02682033],
    [-0.01479032, -0.42687458, -0.037428],
    [0.01905555, -0.4200455, -0.03456167],
    [-0.00226458, 0.05603239, 0.00285505],
    [0.04105436, -0.06028581, 0.12204243],
    [-0.03483987, -0.06210566, 0.13032329],
    [-0.0133902, 0.21163553, -0.03346758],
    [0.07170245, 0.11399969, -0.01889817],
    [-0.08295366, 0.11247234, -0.02370739],
    [0.01011321, 0.08893734, 0.05040987],
    [0.12292141, 0.04520509, -0.019046],
    [-0.11322832, 0.04685326, -0.00847207],
    [0.2553319, -0.01564902, -0.02294649],
    [-0.26012748, -0.01436928, -0.03126873],
    [0.26570925, 0.01269811, -0.00737473],
    [-0.26910836, 0.00679372, -0.00602676],
    [0.08669055, -0.01063603, -0.01559429],
    [-0.0887537, -0.00865157, -0.01010708],
], dtype=np.float32)

FOOT_JOINTS = (7, 8, 10, 11)  # FlowerDance contact order: l_ankle, r_ankle, l_foot, r_foot
CONTACT_VEL_THRESHOLD = 0.01  # metres / frame, as in FlowerDance's dataset preprocessing

# Joints whose rotations are replaced for an `upper` gesture (wrists/hands/head keep the base dance:
# joint positions carry no information about them).
UPPER_JOINTS_DEFAULT = [3, 6, 9, 12, 13, 14, 16, 17, 18, 19]
# Joints whose rotations are replaced for a `full` gesture (root handled separately).
FULL_JOINTS_DEFAULT = [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 16, 17, 18, 19]

BONES = [(SMPL_PARENTS[j], j) for j in range(1, N_JOINTS)]
LEFT_JOINTS = {1, 4, 7, 10, 13, 16, 18, 20, 22}
RIGHT_JOINTS = {2, 5, 8, 11, 14, 17, 19, 21, 23}

R_ZUP_FROM_YUP = np.array([[1.0, 0.0, 0.0], [0.0, 0.0, -1.0], [0.0, 1.0, 0.0]], dtype=np.float32)


def children_of(j, limit=N_JOINTS):
    return [c for c in range(limit) if SMPL_PARENTS[c] == j]


def check_against_flowerdance(repo_dir):
    """Raise if the skeleton constants differ from the cloned FlowerDance repository."""
    import ast
    import os
    src = open(os.path.join(repo_dir, "vis.py"), encoding="utf-8").read()
    tree = ast.parse(src)
    found = {}
    for node in tree.body:
        if isinstance(node, ast.Assign) and len(node.targets) == 1 and isinstance(node.targets[0], ast.Name):
            if node.targets[0].id in ("smpl_parents", "smpl_offsets"):
                found[node.targets[0].id] = ast.literal_eval(node.value)
    if found.get("smpl_parents") != SMPL_PARENTS:
        raise RuntimeError("smpl_parents in FlowerDance/vis.py differ from m2d.smpl_utils")
    if not np.allclose(np.asarray(found.get("smpl_offsets"), dtype=np.float32), SMPL_OFFSETS, atol=1e-7):
        raise RuntimeError("smpl_offsets in FlowerDance/vis.py differ from m2d.smpl_utils")
    return True


def _t(x, device=None, dtype=torch.float32):
    if isinstance(x, torch.Tensor):
        return x.to(device=device if device is not None else x.device, dtype=dtype)
    return torch.as_tensor(np.asarray(x), dtype=dtype, device=device)


def offsets_tensor(like):
    return torch.as_tensor(SMPL_OFFSETS, dtype=like.dtype, device=like.device)


def fk(rotmat, root_pos, return_global=False):
    """Forward kinematics identical to FlowerDance's SMPLSkeleton (matrix form).

    rotmat: (..., 24, 3, 3) local rotations (joint 0 = global root orientation)
    root_pos: (..., 3)
    returns positions (..., 24, 3) [and global rotations (..., 24, 3, 3)]
    """
    off = offsets_tensor(rotmat)
    pos = [None] * N_JOINTS
    glob = [None] * N_JOINTS
    for j in range(N_JOINTS):
        p = SMPL_PARENTS[j]
        if p < 0:
            pos[j] = root_pos
            glob[j] = rotmat[..., 0, :, :]
        else:
            pos[j] = pos[p] + (glob[p] @ off[j].unsqueeze(-1)).squeeze(-1)
            glob[j] = glob[p] @ rotmat[..., j, :, :]
    pos = torch.stack(pos, dim=-2)
    if return_global:
        return pos, torch.stack(glob, dim=-3)
    return pos


def compute_contacts(positions):
    """FlowerDance contact labels: foot speed < 1 cm/frame (last frame copies the previous rule: v=0)."""
    feet = positions[..., FOOT_JOINTS, :]
    v = torch.zeros(feet.shape[:-1], dtype=positions.dtype, device=positions.device)
    v[..., :-1, :] = (feet[..., 1:, :, :] - feet[..., :-1, :, :]).norm(dim=-1)
    return (v < CONTACT_VEL_THRESHOLD).to(positions.dtype)


def encode(root_pos, rotmat, contacts=None):
    """(T,3), (T,24,3,3), (T,4) -> (T,151) un-normalised FlowerDance vector."""
    if contacts is None:
        contacts = compute_contacts(fk(rotmat, root_pos))
    r6 = matrix_to_rotation_6d(rotmat).reshape(*rotmat.shape[:-3], N_JOINTS * 6)
    return torch.cat([contacts, root_pos, r6], dim=-1)


def decode(vec):
    """(..., 151) -> dict(root_pos, rotmat, contacts)."""
    contacts = vec[..., CONTACT_SLICE]
    root_pos = vec[..., ROOT_POS_SLICE]
    r6 = vec[..., ROT_OFFSET:].reshape(*vec.shape[:-1], N_JOINTS, 6)
    return {"root_pos": root_pos, "rotmat": rotation_6d_to_matrix(r6), "contacts": contacts}


def joint_channels(joints):
    """Channel indices of the 6D rotation of the given joints inside the 151-dim vector."""
    out = []
    for j in joints:
        out += list(range(ROT_OFFSET + 6 * j, ROT_OFFSET + 6 * j + 6))
    return out


def rotmat_to_aa(rotmat):
    return matrix_to_axis_angle(rotmat)


def aa_to_rotmat(aa):
    return axis_angle_to_matrix(aa)


def yup_to_zup(root_rotmat, root_pos):
    R = torch.as_tensor(R_ZUP_FROM_YUP, dtype=root_rotmat.dtype, device=root_rotmat.device)
    return R @ root_rotmat, root_pos @ R.T


def zup_to_yup(root_rotmat, root_pos):
    R = torch.as_tensor(R_ZUP_FROM_YUP, dtype=root_rotmat.dtype, device=root_rotmat.device)
    return R.T @ root_rotmat, root_pos @ R


def rot_z(angle):
    """Rotation about +z (z-up world). angle: tensor (...)."""
    c, s = torch.cos(angle), torch.sin(angle)
    z, o = torch.zeros_like(angle), torch.ones_like(angle)
    return torch.stack([torch.stack([c, -s, z], -1), torch.stack([s, c, z], -1), torch.stack([z, z, o], -1)], -2)


def heading_zup(root_rotmat):
    """Yaw of the body's forward axis (local +z) projected on the ground plane of the z-up world."""
    fwd = root_rotmat[..., :, 2]
    return torch.atan2(fwd[..., 1], fwd[..., 0])


def split_yaw_tilt_zup(root_rotmat):
    """root = Rz(yaw) @ tilt, where tilt keeps the body's forward axis in the x-z plane rotated by yaw=0."""
    yaw = heading_zup(root_rotmat)
    tilt = rot_z(-yaw) @ root_rotmat
    return yaw, tilt


def unwrap(angle):
    a = angle.detach().cpu().numpy()
    return torch.as_tensor(np.unwrap(a, axis=0), dtype=angle.dtype, device=angle.device)


def minimal_rotation(a, b, eps=1e-8):
    """Smallest rotation matrices R with R a_hat = b_hat. a, b: (..., 3)."""
    a = F.normalize(a, dim=-1, eps=eps)
    b = F.normalize(b, dim=-1, eps=eps)
    v = torch.cross(a, b, dim=-1)
    c = (a * b).sum(-1).clamp(-1.0, 1.0)
    s = v.norm(dim=-1)
    # Antiparallel case: pick an axis orthogonal to a.
    helper = torch.where((a[..., :1].abs() < 0.9), torch.tensor([1.0, 0.0, 0.0], device=a.device, dtype=a.dtype).expand_as(a),
                         torch.tensor([0.0, 1.0, 0.0], device=a.device, dtype=a.dtype).expand_as(a))
    ortho = F.normalize(torch.cross(a, helper, dim=-1), dim=-1, eps=eps)
    axis = torch.where((s > 1e-6).unsqueeze(-1), v / s.clamp_min(1e-12).unsqueeze(-1), ortho)
    angle = torch.atan2(s, c)
    return axis_angle_to_matrix(axis * angle.unsqueeze(-1))


def kabsch(src, dst, weights=None):
    """Rotation R minimising sum_i w_i |R src_i - dst_i|^2. src, dst: (..., N, 3)."""
    if weights is None:
        weights = torch.ones(src.shape[:-1], dtype=src.dtype, device=src.device)
    H = (src * weights.unsqueeze(-1)).transpose(-1, -2) @ dst  # (...,3,3) = sum w src dst^T
    U, S, Vh = torch.linalg.svd(H)
    V = Vh.transpose(-1, -2)
    d = torch.sign(torch.det(V @ U.transpose(-1, -2)))
    d = torch.where(d == 0, torch.ones_like(d), d)
    D = torch.diag_embed(torch.stack([torch.ones_like(d), torch.ones_like(d), d], -1))
    return V @ D @ U.transpose(-1, -2)


def slerp_rotmat(R0, R1, w):
    """Geodesic interpolation between rotation matrices with weight w in [0,1] (broadcastable)."""
    rel = R0.transpose(-1, -2) @ R1
    aa = matrix_to_axis_angle(rel)
    return R0 @ axis_angle_to_matrix(aa * w)


def foot_floor_level(positions, q=5.0):
    """Robust floor height (z-up) estimated from the lowest foot joint per frame."""
    low = positions[..., list(FOOT_JOINTS), 2].min(dim=-1).values
    return float(np.percentile(low.detach().cpu().numpy(), q))

In [ ]:
%%writefile /content/m2d_pkg/m2d/mamba_fallback.py
"""Last-resort pure-PyTorch replacement for ``mamba_ssm.Mamba`` (no CUDA extension needed).

The module keeps exactly the parameter names/shapes of ``mamba_ssm.modules.mamba_simple.Mamba``
(in_proj, conv1d, x_proj, dt_proj, A_log, D, out_proj), so FlowerDance checkpoints load with
``strict=True``. The forward pass implements the same math as mamba_ssm's slow path
(``selective_scan_ref``) but evaluates the recurrence chunk-wise:

    inside a chunk:   h_t = exp(S_t) h_0 + sum_{s<=t} exp(S_t - S_s) dt_s B_s u_s,   S_t = sum_{k<=t} dt_k A
    across chunks:    the final state of a chunk is the initial state of the next one.

All exponents are <= 0, so it is numerically stable. It is slower than the fused CUDA kernel but
fast enough for FlowerDance's few sampling steps.

``install()`` registers a fake ``mamba_ssm`` package exposing this class so that
``from mamba_ssm import Mamba`` (used by FlowerDance/model/model.py) resolves to it.
"""
import math
import sys
import types

import torch
import torch.nn as nn
import torch.nn.functional as F

CHUNK = 32          # time steps per chunk (memory ~ B*D*N*CHUNK^2*GROUP_CHUNKS floats)
GROUP_CHUNKS = 8    # chunks processed in parallel for the intra-chunk part


def selective_scan_chunked(u, delta, A, B, C, D=None, z=None, delta_bias=None, delta_softplus=False,
                           chunk=CHUNK, group_chunks=GROUP_CHUNKS):
    """Same signature/semantics as mamba_ssm.ops.selective_scan_interface.selective_scan_ref
    for real-valued A (d, n) and input-dependent B, C of shape (b, n, l)."""
    dtype_in = u.dtype
    u = u.float()
    delta = delta.float()
    if delta_bias is not None:
        delta = delta + delta_bias[..., None].float()
    if delta_softplus:
        delta = F.softplus(delta)
    A = A.float()
    B = B.float()
    C = C.float()
    b, d, L = u.shape
    n = A.shape[1]
    pad = (-L) % chunk
    if pad:
        # Zero delta => no state change and zero input on padded steps (they are cut afterwards).
        u = F.pad(u, (0, pad))
        delta = F.pad(delta, (0, pad))
        B = F.pad(B, (0, pad))
        C = F.pad(C, (0, pad))
    Lp = L + pad
    nc = Lp // chunk
    # (b, d, nc, T) / (b, n, nc, T)
    u_c = u.view(b, d, nc, chunk)
    dt_c = delta.view(b, d, nc, chunk)
    B_c = B.view(b, n, nc, chunk)
    C_c = C.view(b, n, nc, chunk)

    causal = torch.tril(torch.ones(chunk, chunk, dtype=torch.bool, device=u.device))
    y = torch.empty(b, d, nc, chunk, dtype=torch.float32, device=u.device)
    h = torch.zeros(b, d, n, dtype=torch.float32, device=u.device)
    for g0 in range(0, nc, group_chunks):
        g1 = min(nc, g0 + group_chunks)
        dt = dt_c[:, :, g0:g1]                                    # (b,d,G,T)
        S = torch.cumsum(dt.unsqueeze(-1) * A[None, :, None, None, :], dim=3)  # (b,d,G,T,n)
        dBu = (dt * u_c[:, :, g0:g1]).unsqueeze(-1) * B_c[:, :, g0:g1].permute(0, 2, 3, 1).unsqueeze(1)  # (b,d,G,T,n)
        # W[t, s] = exp(S_t - S_s) for s <= t
        diff = S.unsqueeze(4) - S.unsqueeze(3)                    # (b,d,G,T(t),T(s),n)
        diff = diff.masked_fill(~causal[None, None, None, :, :, None], float("-inf"))
        h_local = torch.einsum("bdgtsn,bdgsn->bdgtn", torch.exp(diff), dBu)  # zero initial state
        del diff
        decay_in = torch.exp(S)                                   # (b,d,G,T,n) decay from chunk start
        Cg = C_c[:, :, g0:g1].permute(0, 2, 3, 1)                # (b,G,T,n)
        # Sequential carry of the state across the chunks of this group (cheap: b*d*n per chunk).
        for gi in range(g1 - g0):
            hs = h_local[:, :, gi] + decay_in[:, :, gi] * h.unsqueeze(2)   # (b,d,T,n)
            y[:, :, g0 + gi] = torch.einsum("bdtn,btn->bdt", hs, Cg[:, gi])
            h = hs[:, :, -1]
        del h_local, decay_in
    y = y.reshape(b, d, Lp)[..., :L]
    u = u[..., :L]
    out = y if D is None else y + u * D.float()[:, None]
    if z is not None:
        out = out * F.silu(z.float())
    return out.to(dtype_in)


def selective_scan_sequential(u, delta, A, B, C, D=None, z=None, delta_bias=None, delta_softplus=False):
    """Straight copy of mamba_ssm's selective_scan_ref math (real A, variable B/C). Used for testing."""
    dtype_in = u.dtype
    u = u.float()
    delta = delta.float()
    if delta_bias is not None:
        delta = delta + delta_bias[..., None].float()
    if delta_softplus:
        delta = F.softplus(delta)
    batch, dim, dstate = u.shape[0], A.shape[0], A.shape[1]
    B = B.float()
    C = C.float()
    x = A.new_zeros((batch, dim, dstate))
    ys = []
    deltaA = torch.exp(torch.einsum("bdl,dn->bdln", delta, A))
    deltaB_u = torch.einsum("bdl,bnl,bdl->bdln", delta, B, u)
    for i in range(u.shape[2]):
        x = deltaA[:, :, i] * x + deltaB_u[:, :, i]
        ys.append(torch.einsum("bdn,bn->bd", x, C[:, :, i]))
    y = torch.stack(ys, dim=2)
    out = y if D is None else y + u * D[:, None]
    if z is not None:
        out = out * F.silu(z)
    return out.to(dtype_in)


class Mamba(nn.Module):
    """Parameter-compatible pure-PyTorch version of mamba_ssm.modules.mamba_simple.Mamba (no caching)."""

    def __init__(self, d_model, d_state=16, d_conv=4, expand=2, dt_rank="auto", dt_min=0.001, dt_max=0.1,
                 dt_init="random", dt_scale=1.0, dt_init_floor=1e-4, conv_bias=True, bias=False,
                 use_fast_path=True, layer_idx=None, device=None, dtype=None):
        factory_kwargs = {"device": device, "dtype": dtype}
        super().__init__()
        self.d_model = d_model
        self.d_state = d_state
        self.d_conv = d_conv
        self.expand = expand
        self.d_inner = int(self.expand * self.d_model)
        self.dt_rank = math.ceil(self.d_model / 16) if dt_rank == "auto" else dt_rank
        self.layer_idx = layer_idx
        self.in_proj = nn.Linear(self.d_model, self.d_inner * 2, bias=bias, **factory_kwargs)
        self.conv1d = nn.Conv1d(self.d_inner, self.d_inner, bias=conv_bias, kernel_size=d_conv,
                                groups=self.d_inner, padding=d_conv - 1, **factory_kwargs)
        self.act = nn.SiLU()
        self.x_proj = nn.Linear(self.d_inner, self.dt_rank + self.d_state * 2, bias=False, **factory_kwargs)
        self.dt_proj = nn.Linear(self.dt_rank, self.d_inner, bias=True, **factory_kwargs)
        dt_init_std = self.dt_rank ** -0.5 * dt_scale
        if dt_init == "constant":
            nn.init.constant_(self.dt_proj.weight, dt_init_std)
        else:
            nn.init.uniform_(self.dt_proj.weight, -dt_init_std, dt_init_std)
        dt = torch.exp(torch.rand(self.d_inner, **factory_kwargs) * (math.log(dt_max) - math.log(dt_min))
                       + math.log(dt_min)).clamp(min=dt_init_floor)
        inv_dt = dt + torch.log(-torch.expm1(-dt))
        with torch.no_grad():
            self.dt_proj.bias.copy_(inv_dt)
        A = torch.arange(1, self.d_state + 1, dtype=torch.float32, device=device).repeat(self.d_inner, 1)
        self.A_log = nn.Parameter(torch.log(A))
        self.D = nn.Parameter(torch.ones(self.d_inner, device=device))
        self.out_proj = nn.Linear(self.d_inner, self.d_model, bias=bias, **factory_kwargs)
        self.scan_fn = selective_scan_chunked

    def forward(self, hidden_states, inference_params=None):
        batch, seqlen, _ = hidden_states.shape
        xz = (hidden_states @ self.in_proj.weight.T).transpose(1, 2)  # (b, 2*d_inner, l)
        if self.in_proj.bias is not None:
            xz = xz + self.in_proj.bias.to(xz.dtype)[:, None]
        A = -torch.exp(self.A_log.float())
        x, z = xz.chunk(2, dim=1)
        x = self.act(self.conv1d(x)[..., :seqlen])
        x_dbl = self.x_proj(x.transpose(1, 2).reshape(batch * seqlen, -1))
        dt, Bm, Cm = torch.split(x_dbl, [self.dt_rank, self.d_state, self.d_state], dim=-1)
        dt = (self.dt_proj.weight @ dt.T).reshape(self.d_inner, batch, seqlen).permute(1, 0, 2)
        Bm = Bm.reshape(batch, seqlen, self.d_state).transpose(1, 2).contiguous()
        Cm = Cm.reshape(batch, seqlen, self.d_state).transpose(1, 2).contiguous()
        y = self.scan_fn(x, dt, A, Bm, Cm, self.D.float(), z=z, delta_bias=self.dt_proj.bias.float(),
                         delta_softplus=True)
        return self.out_proj(y.transpose(1, 2))


def install():
    """Register a fake `mamba_ssm` package whose `Mamba` is the pure-PyTorch class above."""
    pkg = types.ModuleType("mamba_ssm")
    pkg.__path__ = []
    pkg.__version__ = "pure-pytorch-fallback"
    pkg.Mamba = Mamba
    modules = types.ModuleType("mamba_ssm.modules")
    modules.__path__ = []
    simple = types.ModuleType("mamba_ssm.modules.mamba_simple")
    simple.Mamba = Mamba
    sys.modules["mamba_ssm"] = pkg
    sys.modules["mamba_ssm.modules"] = modules
    sys.modules["mamba_ssm.modules.mamba_simple"] = simple
    return Mamba

In [ ]:
%%writefile /content/m2d_pkg/m2d/flowerdance_runner.py
"""FlowerDance inference wrapper: model loading, music features, whole/chunked sampling and masked editing.

The sampler and the editing loop reproduce FlowerDance's own code
(``model/flowmatching.py``: ``euler_sample`` and ``inpaint``) on top of ``DanceDecoder.infer_pred``;
the music features are computed with the functions of FlowerDance's ``inference.py``.

Masked generation (``inpaint``):
  * ``mask``  (T, 151) in [0, 1]: 1 = keep the given motion, 0 = generate.
  * ``decay`` (T, 151) bool: True  -> FlowerDance's time-decayed soft mask (weight = mask * t),
                             False -> strict mask (weight = mask, never released).
"""
import importlib
import math
import os
import sys
import time

import numpy as np
import torch
import torch.nn.functional as F

from . import smpl_utils as su

GENRES = ("Dai", "ShenYun", "Wei", "Korean", "Urban", "Hiphop", "Popping", "Miao", "HanTang",
          "Breaking", "Kun", "Locking", "Jazz", "Choreography", "Chinese", "DunHuang")
HF_REPO = "xlt99/FlowerDance"
HF_FILE = "train-3700.pt"
TRAIN_HORIZON = 1200  # 40 s at 30 fps: sequence length used by the released model configuration


def patch_transformers_for_mamba():
    """mamba_ssm.utils.generation imports names that newer transformers releases removed."""
    try:
        import transformers.generation as tg
    except Exception:
        return
    aliases = {
        "GreedySearchDecoderOnlyOutput": "GenerateDecoderOnlyOutput",
        "SampleDecoderOnlyOutput": "GenerateDecoderOnlyOutput",
    }
    for old, new in aliases.items():
        try:
            getattr(tg, old)
        except Exception:
            try:
                setattr(tg, old, getattr(tg, new))
            except Exception:
                pass


def genre_id(name):
    if isinstance(name, int):
        return name
    for i, g in enumerate(GENRES):
        if g.casefold() == str(name).strip().casefold():
            return i
    raise ValueError(f"Unknown FlowerDance genre {name!r}; choose one of {GENRES}")


class FlowerDance:
    def __init__(self, repo_dir, ckpt_path, device="cuda", mamba_backend="cuda", load_model=True):
        """mamba_backend: 'cuda' (mamba_ssm kernels) or 'torch' (pure-PyTorch fallback).
        load_model=False only loads the data normaliser (encode/decode without using the GPU)."""
        self.repo_dir = os.path.abspath(repo_dir)
        self.device = torch.device(device)
        if self.repo_dir not in sys.path:
            sys.path.insert(0, self.repo_dir)
        su.check_against_flowerdance(self.repo_dir)
        if not load_model:
            ckpt = torch.load(ckpt_path, map_location="cpu", weights_only=False)
            self.normalizer = ckpt["normalizer"]
            self.model = None
            self.mamba_backend = None
            return
        if mamba_backend == "torch":
            from . import mamba_fallback
            mamba_fallback.install()
        else:
            patch_transformers_for_mamba()
        for mod in ("model.model", "model"):
            sys.modules.pop(mod, None)
        DanceDecoder = importlib.import_module("model.model").DanceDecoder
        self.mamba_backend = mamba_backend
        # Same hyper-parameters as FlowerDance/EDGE.py
        self.model = DanceDecoder(nfeats=su.MOTION_DIM, seq_len=TRAIN_HORIZON, latent_dim=512, ff_size=1024,
                                  num_layers=8, num_heads=8, dropout=0.1, cond_feature_dim=35, activation=F.gelu)
        ckpt = torch.load(ckpt_path, map_location="cpu", weights_only=False)
        if "model_state_dict" not in ckpt or "normalizer" not in ckpt:
            raise KeyError("FlowerDance checkpoint must contain 'model_state_dict' and 'normalizer'")
        self.model.load_state_dict(ckpt["model_state_dict"], strict=True)
        self.normalizer = ckpt["normalizer"]
        self.model.to(self.device).eval()
        self.n_params = sum(p.numel() for p in self.model.parameters())

    # ------------------------------------------------------------------ normalisation
    def normalize(self, vec):
        x = vec.detach().float().cpu().clone()
        squeeze = x.dim() == 2
        if squeeze:
            x = x[None]
        x = self.normalizer.normalize(x)  # MinMaxScaler to [-1, 1] with clipping (in-place on the clone)
        return x[0] if squeeze else x

    def unnormalize(self, xn):
        x = xn.detach().float().cpu().clone()
        squeeze = x.dim() == 2
        if squeeze:
            x = x[None]
        x = self.normalizer.unnormalize(x)
        return x[0] if squeeze else x

    def decode(self, xn):
        """normalised (T,151) -> dict(root_pos, rotmat, contacts, positions) un-normalised, z-up."""
        m = su.decode(self.unnormalize(xn))
        m["positions"] = su.fk(m["rotmat"], m["root_pos"])
        return m

    def encode(self, root_pos, rotmat, contacts=None):
        return self.normalize(su.encode(root_pos.float().cpu(), rotmat.float().cpu(),
                                        None if contacts is None else contacts.float().cpu()))

    # ------------------------------------------------------------------ features
    def music_features(self, audio_path):
        """35-dim normalised features for the whole song, exactly as FlowerDance's inference.py."""
        inf = importlib.import_module("inference")
        import librosa
        if not hasattr(librosa.beat, "tempo"):  # removed in librosa 1.0 (FlowerDance pins 0.11)
            from librosa.feature.rhythm import tempo as _tempo
            librosa.beat.tempo = _tempo
        duration_total = librosa.get_duration(path=audio_path)
        frames = int(math.floor(duration_total * inf.FPS)) - 1
        frames = max(frames, 1)
        duration = frames / inf.FPS
        audio, tempo_audio, tempo_sr, frame_count = inf.load_audio_clip(audio_path, start=0.0, duration=duration)
        start_bpm = inf.estimate_tempo(tempo_audio, tempo_sr)
        raw = inf.extract_baseline_features(audio, frame_count, start_bpm)
        feats = inf.normalize_features(raw)
        return {"features": feats, "frames": frame_count, "fps": inf.FPS, "bpm_librosa": start_bpm}

    # ------------------------------------------------------------------ sampling
    def _pred(self, x, cond, genre, t, h):
        return self.model.infer_pred(x, cond, genre, t, h)

    @torch.no_grad()
    def sample(self, cond, genre, n_steps=21, seed=None, noise=None):
        """Plain generation (FlowerDance euler_sample). cond: (T,35) np/tensor. Returns normalised (T,151) cpu."""
        cond_t = torch.as_tensor(cond, dtype=torch.float32, device=self.device)[None]
        g = torch.tensor([genre_id(genre)], device=self.device)
        gen = torch.Generator(device=self.device)
        gen.manual_seed(int(seed) if seed is not None else int(time.time()))
        x = noise.to(self.device)[None] if noise is not None else torch.randn(
            (1, cond_t.shape[1], su.MOTION_DIM), device=self.device, generator=gen)
        t_values = torch.linspace(1, 0, n_steps, device=self.device)
        dt = 1.0 / (n_steps - 1)
        for i in range(n_steps - 1):
            t = torch.full((1,), float(t_values[i]), device=self.device)
            h = torch.full((1,), float(t_values[i] - t_values[i + 1]), device=self.device)
            v = self._pred(x, cond_t, g, t, h)
            x = x - v * dt
        return x[0].float().cpu()

    @torch.no_grad()
    def inpaint(self, x_known, mask, cond, genre, decay=None, n_steps=101, seed=None):
        """Masked generation (FlowerDance ``FlowMatching.inpaint`` with per-element soft/strict mask).

        x_known: normalised (T,151); mask: (T,151) in [0,1]; decay: (T,151) bool (True = soft, time-decayed).
        """
        dev = self.device
        x = torch.as_tensor(x_known, dtype=torch.float32, device=dev)[None]
        m = torch.as_tensor(mask, dtype=torch.float32, device=dev)[None]
        if decay is None:
            decay = torch.ones_like(m, dtype=torch.bool)
        dcy = torch.as_tensor(decay, dtype=torch.bool, device=dev).reshape(m.shape)
        cond_t = torch.as_tensor(cond, dtype=torch.float32, device=dev)[None]
        g = torch.tensor([genre_id(genre)], device=dev)
        gen = torch.Generator(device=dev)
        gen.manual_seed(int(seed) if seed is not None else int(time.time()))
        e = torch.randn(x.shape, device=dev, generator=gen)
        x_t = e.clone()
        t_values = torch.linspace(1.0, 0.0, n_steps, device=dev)
        for i in range(n_steps - 1):
            t_cur, t_next = t_values[i], t_values[i + 1]
            tb = torch.full((1,), float(t_cur), device=dev)
            hb = torch.full((1,), float(t_cur - t_next), device=dev)
            v = self._pred(x_t, cond_t, g, tb, hb)
            x_t = x_t + v * (t_next - t_cur)
            w = torch.where(dcy, m * t_next, m)
            x_t = w * ((1 - t_next) * x + t_next * e) + (1 - w) * x_t
        return x_t[0].float().cpu()

    def sample_long(self, cond, genre, mode="auto", n_steps=21, seed=0, window=960, overlap=150,
                    inpaint_steps=51, progress=print):
        """Generate a whole song.

        mode='whole'   : one pass over all frames (Mamba has no fixed horizon).
        mode='chunked' : windows of `window` frames; every window after the first is generated by masked
                         generation whose first `overlap` frames are strictly fixed to the previous window.
        mode='auto'    : 'whole' if the song fits in the training horizon (40 s), else 'chunked'.
        """
        T = len(cond)
        if mode == "auto":
            mode = "whole" if T <= TRAIN_HORIZON else "chunked"
        info = {"mode": mode, "frames": T, "windows": []}
        t0 = time.time()
        if mode == "whole" or T <= window:
            out = self.sample(cond, genre, n_steps=n_steps, seed=seed)
            info["windows"].append((0, T))
        elif mode == "chunked":
            if overlap >= window:
                raise ValueError("overlap must be smaller than window")
            out = torch.zeros(T, su.MOTION_DIM)
            out[:window] = self.sample(cond[:window], genre, n_steps=n_steps, seed=seed)
            info["windows"].append((0, window))
            filled = window
            k = 1
            while filled < T:
                e_ = min(T, filled - overlap + window)
                s_ = max(0, e_ - window)  # keep a full-length window at the end of the song
                known = torch.zeros(e_ - s_, su.MOTION_DIM)
                known[: filled - s_] = out[s_:filled]
                mask = torch.zeros_like(known)
                mask[: filled - s_] = 1.0
                seg = self.inpaint(known, mask, cond[s_:e_], genre, decay=torch.zeros_like(mask, dtype=torch.bool),
                                   n_steps=inpaint_steps, seed=seed + 1000 * k)
                out[filled:e_] = seg[filled - s_:]
                info["windows"].append((s_, e_))
                progress(f"  window {k}: frames {s_}-{e_} (fixed {filled - s_} frames of overlap)")
                filled = e_
                k += 1
        else:
            raise ValueError(f"unknown mode {mode}")
        info["seconds"] = time.time() - t0
        return out, info

    def free(self):
        if self.model is not None:
            self.model.cpu()
        self.model = None
        import gc
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()


def motion_stats(positions, fps=30, window_s=5.0):
    """Per-window mean joint speed (m/s) and foot-skating, to judge long-sequence quality."""
    pos = positions.detach().cpu()
    vel = (pos[1:] - pos[:-1]).norm(dim=-1) * fps  # (T-1, 24)
    speed = vel.mean(-1).numpy()
    win = int(window_s * fps)
    n = max(1, len(speed) // win)
    per_win = [float(speed[i * win:(i + 1) * win].mean()) for i in range(n)]
    contacts = su.compute_contacts(pos)
    feet = pos[:, list(su.FOOT_JOINTS)]
    fv = torch.zeros(feet.shape[:2])
    fv[:-1] = (feet[1:, :, :2] - feet[:-1, :, :2]).norm(dim=-1)
    skate = float((fv * contacts).sum() / contacts.sum().clamp_min(1)) * fps
    up = (pos[:, 12] - pos[:, 0]).mean(0)
    up_cos = float(up[2] / up.norm().clamp_min(1e-9))  # body axis vs +z: should be close to 1 in FlowerDance's frame
    return {"speed_per_window": per_win, "mean_speed": float(speed.mean()), "foot_skate_mps": skate,
            "root_travel_m": float((pos[1:, 0, :2] - pos[:-1, 0, :2]).norm(dim=-1).sum()), "up_axis_cos": up_cos}

In [ ]:
%%writefile /content/m2d_pkg/m2d/audio_analysis.py
"""Steps 1, 3, 4: vocal separation (Demucs), beats/downbeats (beat_this) and the 8-count grid."""
import gc
import json
import os

import numpy as np

COUNT_NAMES = [f"{i}{s}" for i in range(1, 9) for s in ("", "&")]  # '1', '1&', ..., '8&'


def _free():
    gc.collect()
    try:
        import torch
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
    except Exception:
        pass


def load_audio(path, sr=None, mono=False):
    import librosa
    y, sr = librosa.load(path, sr=sr, mono=mono)
    return y, sr


# ---------------------------------------------------------------------------- step 1: Demucs
def separate_vocals(audio_path, out_dir, model_name="htdemucs", device="cuda", shifts=1, overlap=0.25):
    """Returns paths of vocals.wav and accompaniment.wav (44.1 kHz stereo)."""
    import soundfile as sf
    import torch
    from demucs.apply import apply_model
    from demucs.pretrained import get_model

    os.makedirs(out_dir, exist_ok=True)
    model = get_model(model_name)
    model.eval()
    y, _ = load_audio(audio_path, sr=model.samplerate, mono=False)
    if y.ndim == 1:
        y = np.stack([y, y])
    if y.shape[0] == 1:
        y = np.concatenate([y, y])
    y = y[: model.audio_channels]
    wav = torch.as_tensor(y, dtype=torch.float32)
    ref = wav.mean(0)
    mean, std = ref.mean(), ref.std() + 1e-8
    with torch.no_grad():
        sources = apply_model(model, ((wav - mean) / std)[None], device=device, shifts=shifts, split=True,
                              overlap=overlap, progress=True)[0]
    sources = sources * std + mean
    names = list(model.sources)
    vocals = sources[names.index("vocals")].cpu().numpy()
    accomp = sum(sources[i] for i, n in enumerate(names) if n != "vocals").cpu().numpy()
    vp = os.path.join(out_dir, "vocals.wav")
    ap = os.path.join(out_dir, "accompaniment.wav")
    sf.write(vp, vocals.T, model.samplerate)
    sf.write(ap, accomp.T, model.samplerate)
    del model, sources, wav
    _free()
    return vp, ap


# ---------------------------------------------------------------------------- step 3: beat_this
# The authors' public share (Nextcloud): WebDAV endpoint (used by beat_this itself) and the share link.
BEAT_THIS_URLS = (
    "https://cloud.cp.jku.at/public.php/dav/files/7ik4RrBKTS273gp/{name}.ckpt",
    "https://cloud.cp.jku.at/index.php/s/7ik4RrBKTS273gp/download?path=%2F&files={name}.ckpt",
)
# Mirrors of the unmodified checkpoint (MIT licensed weights), used when the authors' host is down.
# Every download is accepted only if its SHA-256 equals the original file's hash.
BEAT_THIS_MIRRORS = {
    "final0": (
        "https://github.com/zyziyun/reelfold/releases/download/models-beat-this-final0/final0.ckpt",
        "https://huggingface.co/AEmotionStudio/muscriptor-models/resolve/main/shared/beat_this_final0.ckpt",
    ),
}
BEAT_THIS_SHA256 = {
    "final0": "8c328b45f59d8dd3dff219253ff6a8d6482be57d0133a29140e2febbf8eb8331",
}


def _sha256(path):
    import hashlib
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1 << 20), b""):
            h.update(chunk)
    return h.hexdigest()


def beat_this_cache_path(name="final0"):
    """Where torch.hub (and therefore beat_this) looks for the downloaded checkpoint."""
    import torch
    return os.path.join(torch.hub.get_dir(), "checkpoints", f"beat_this-{name}.ckpt")


def _valid_checkpoint(path, name=None):
    import torch
    if not os.path.isfile(path) or os.path.getsize(path) < 1_000_000:
        return False
    if name in BEAT_THIS_SHA256 and _sha256(path) != BEAT_THIS_SHA256[name]:
        return False
    try:
        ck = torch.load(path, map_location="cpu", weights_only=False)
        return isinstance(ck, dict) and "state_dict" in ck
    except Exception:
        return False


def ensure_beat_this_checkpoint(name="final0", retries=5, timeout=120, log=print):
    """Download a beat_this checkpoint into the torch.hub cache: authors' host first, then mirrors, with retries.

    beat_this then finds it in the cache and never touches the network. Returns the cached path.
    """
    import time

    import requests
    dst = beat_this_cache_path(name)
    if _valid_checkpoint(dst, name):
        return dst
    os.makedirs(os.path.dirname(dst), exist_ok=True)
    errors = []
    for attempt in range(retries):
        for url in [u.format(name=name) for u in BEAT_THIS_URLS] + list(BEAT_THIS_MIRRORS.get(name, ())):
            try:
                with requests.get(url, stream=True, timeout=timeout) as r:
                    r.raise_for_status()
                    with open(dst + ".part", "wb") as f:
                        for chunk in r.iter_content(1 << 20):
                            f.write(chunk)
                os.replace(dst + ".part", dst)
                if _valid_checkpoint(dst, name):
                    log(f"beat_this {name}: downloaded from {url.split('/')[2]} (attempt {attempt + 1}, SHA-256 OK)")
                    return dst
                errors.append(f"{url}: not a valid checkpoint (size/SHA-256 mismatch)")
                os.remove(dst)
            except Exception as e:  # network errors, HTTP 5xx ...
                errors.append(f"{url}: {e!r}")
        wait = min(60, 5 * 2 ** attempt)
        log(f"beat_this {name}: download failed ({errors[-1][:120]}); retrying in {wait}s "
            f"({attempt + 1}/{retries})")
        time.sleep(wait)
    raise RuntimeError(
        f"beat_this の重み {name}.ckpt を作者のサーバーからもミラーからも取得できませんでした。\n"
        f"  1) 時間をおいて再実行する\n"
        f"  2) ブラウザで次のどれかから {name}.ckpt をダウンロードし、{dst} にアップロードする\n"
        f"     https://cloud.cp.jku.at/index.php/s/7ik4RrBKTS273gp\n"
        + "".join(f"     {u}\n" for u in BEAT_THIS_MIRRORS.get(name, ())) +
        f"  3) 4-4 で BEAT_TRACKER = 'librosa' を選ぶ（精度は下がります）\n"
        f"last errors: {errors[-2:]}")


def detect_beats(audio_path, checkpoint="final0", device="cuda", dbn=False):
    from beat_this.inference import Audio2Beats
    ensure_beat_this_checkpoint(checkpoint)
    y, sr = load_audio(audio_path, sr=None, mono=True)
    a2b = Audio2Beats(checkpoint_path=checkpoint, device=device, dbn=dbn)
    beats, downbeats = a2b(y, sr)
    del a2b
    _free()
    return np.asarray(beats, dtype=np.float64), np.asarray(downbeats, dtype=np.float64)


def detect_beats_librosa(audio_path, beats_per_bar=4):
    """Last-resort fallback: librosa beat tracking + downbeat phase from low-frequency onset strength."""
    import librosa
    y, sr = librosa.load(audio_path, sr=22050, mono=True)
    hop = 512
    _, perc = librosa.effects.hpss(y)
    env = librosa.onset.onset_strength(y=perc, sr=sr, hop_length=hop)
    _, beat_frames = librosa.beat.beat_track(onset_envelope=env, sr=sr, hop_length=hop, tightness=100)
    beats = librosa.frames_to_time(beat_frames, sr=sr, hop_length=hop)
    if len(beats) < beats_per_bar * 2:
        return beats, beats[::beats_per_bar]
    # kick drums mostly land on the first beat of a bar: compare low-band onset strength per phase
    S = np.abs(librosa.stft(perc, hop_length=hop))
    freqs = librosa.fft_frequencies(sr=sr)
    low = librosa.onset.onset_strength(S=librosa.amplitude_to_db(S[freqs < 150]), sr=sr, hop_length=hop)
    strength = low[np.clip(beat_frames, 0, len(low) - 1)]
    phase = int(np.argmax([strength[p::beats_per_bar].mean() for p in range(beats_per_bar)]))
    return np.asarray(beats, dtype=np.float64), np.asarray(beats[phase::beats_per_bar], dtype=np.float64)


def tempo_from_beats(beats):
    if len(beats) < 2:
        return float("nan")
    return float(60.0 / np.median(np.diff(beats)))


# ---------------------------------------------------------------------------- step 4: 8 counts
def _nearest(arr, x):
    i = int(np.searchsorted(arr, x))
    best = None
    for j in (i - 1, i):
        if 0 <= j < len(arr) and (best is None or abs(arr[j] - x) < abs(arr[best] - x)):
            best = j
    return best


def _build_eights(beats, is_down, start_idx):
    """Group beats into eights. A downbeat that would fall on a count other than 1 or 5 (an irregular
    bar or a tempo/meter change) closes the current eight early and restarts counting on that downbeat."""
    eights = []
    cur = None
    for i in range(start_idx, len(beats)):
        if cur is None:
            cur = {"beats": [i], "irregular": False}
            continue
        k = len(cur["beats"])  # the next beat would be count k+1
        if k == 8:
            eights.append(cur)
            cur = {"beats": [i], "irregular": False}
        elif is_down[i] and k != 4:
            cur["irregular"] = True
            eights.append(cur)
            cur = {"beats": [i], "irregular": False}
        else:
            cur["beats"].append(i)
    if cur is not None and cur["beats"]:
        eights.append(cur)
    return eights


def build_grid(beats, downbeats, duration, line_starts=None, phase="auto", tol=0.07):
    """8-count grid (two 4/4 bars per unit, starting on a downbeat).

    phase: 'auto' chooses whether eights start on even or odd bars so that lyric lines begin on (or
    just before) count 1; or an int bar offset (0/1) to force it.
    Returns dict with 'eights' (index, start, end, beat_times, counts{name: time}), 'beats', 'downbeats'.
    """
    beats = np.asarray(beats, dtype=np.float64)
    downbeats = np.asarray(downbeats, dtype=np.float64)
    if len(beats) < 8:
        raise ValueError("Too few beats were detected to build 8 counts")
    is_down = np.zeros(len(beats), dtype=bool)
    for d in downbeats:
        j = _nearest(beats, d)
        if j is not None and abs(beats[j] - d) <= tol:
            is_down[j] = True
    down_idx = np.flatnonzero(is_down)
    if len(down_idx) == 0:
        down_idx = np.array([0])
        is_down[0] = True
    period = float(np.median(np.diff(beats)))

    def make(start_bar):
        start = int(down_idx[min(start_bar, len(down_idx) - 1)])
        ev = _build_eights(beats, is_down, start)
        return start, ev

    def score(ev):
        if not line_starts:
            return 0.0
        ones = np.array([beats[e["beats"][0]] for e in ev])
        fives = np.array([beats[e["beats"][4]] for e in ev if len(e["beats"]) > 4])
        sc = 0.0
        for ls in line_starts:
            d1 = np.min(np.abs(ones - ls)) if len(ones) else 9e9
            dpk = np.min(np.abs(ones - period - ls)) if len(ones) else 9e9  # pickup on count 8
            dph = np.min(np.abs(ones - 0.5 * period - ls)) if len(ones) else 9e9  # pickup on 8&
            d5 = np.min(np.abs(fives - ls)) if len(fives) else 9e9
            sc += 1.0 * (d1 < 0.35 * period) + 0.6 * (min(dpk, dph) < 0.3 * period) + 0.3 * (d5 < 0.35 * period)
        return sc

    if phase == "auto":
        options = [make(0), make(1)]
        scores = [float(score(ev)) for _, ev in options]
        choice = int(np.argmax(scores)) if scores[1] > scores[0] else 0
    else:
        choice = int(phase)
        scores = None
    start_idx, ev = make(choice)
    eights = []
    for n, e in enumerate(ev):
        bi = e["beats"]
        times = [float(beats[i]) for i in bi]
        nxt = float(beats[bi[-1] + 1]) if bi[-1] + 1 < len(beats) else float(times[-1] + period)
        local_p = float(np.median(np.diff(times + [nxt]))) if len(times) > 0 else period
        counts = {}
        for k, t in enumerate(times):
            t_next = float(beats[bi[k] + 1]) if bi[k] + 1 < len(beats) else t + local_p
            counts[f"{k + 1}"] = t
            counts[f"{k + 1}&"] = 0.5 * (t + t_next)
        end = nxt if len(times) == 8 else min(duration, times[-1] + local_p)
        eights.append({"index": n, "start": times[0], "end": float(end), "beat_times": times, "counts": counts,
                       "n_beats": len(times), "irregular": bool(e["irregular"]), "beat_period": local_p})
    return {"eights": eights, "beats": beats.tolist(), "downbeats": downbeats.tolist(), "bpm": tempo_from_beats(beats),
            "beat_period": period, "phase": choice, "phase_scores": scores, "pickup_beats": int(start_idx)}


def count_time(grid, eight, count):
    e = grid["eights"][int(eight)]
    count = str(count).strip()
    if count not in e["counts"]:
        raise KeyError(f"count {count!r} does not exist in eight {eight} (has {list(e['counts'])})")
    return e["counts"][count]


def nearest_slot(grid, t):
    """(eight index, count name, slot time) of the 8-count slot closest to time t."""
    best = None
    for e in grid["eights"]:
        if t < e["start"] - 2.0 or t > e["end"] + 2.0:
            continue
        for name, ct in e["counts"].items():
            d = abs(ct - t)
            if best is None or d < best[0]:
                best = (d, e["index"], name, ct)
    if best is None:
        return None
    return best[1], best[2], best[3]


def attach_lyrics(grid, lines, words):
    """Annotate eights with lyric words on their nearest count and lines with the eights they cover."""
    for e in grid["eights"]:
        e["words"] = []
        e["lines"] = []
    for w in words:
        if w.get("start") is None:
            continue
        slot = nearest_slot(grid, w["start"])
        if slot is None:
            continue
        ei, cname, _ = slot
        grid["eights"][ei]["words"].append({"count": cname, "text": w["text"], "line": w["line"]})
    for ln in lines:
        if ln.get("start") is None:
            ln["eights"] = []
            continue
        ln["eights"] = [e["index"] for e in grid["eights"] if e["end"] > ln["start"] and e["start"] < ln["end"]]
        for ei in ln["eights"]:
            grid["eights"][ei]["lines"].append(ln["index"])
    return grid


def grid_table(grid, lines=None):
    """Rows for display: one per eight."""
    rows = []
    for e in grid["eights"]:
        wtxt = " ".join(f"{w['count']}:{w['text']}" for w in e.get("words", []))
        rows.append({"eight": e["index"], "start": round(e["start"], 2), "end": round(e["end"], 2),
                     "beats": e["n_beats"], "irregular": e["irregular"], "lines": ",".join(map(str, e.get("lines", []))),
                     "lyrics_on_counts": wtxt})
    return rows


def save_json(obj, path):
    with open(path, "w", encoding="utf-8") as f:
        json.dump(obj, f, ensure_ascii=False, indent=1)

In [ ]:
%%writefile /content/m2d_pkg/m2d/lyrics_align.py
"""Step 2: forced alignment of the given lyrics to the separated vocals (wav2vec2 via WhisperX).

Two stages, so that whole songs fit in memory:
  A. global pass: wav2vec2 CTC log-probabilities over the whole vocal track (computed window by window)
     and one Viterbi CTC forced alignment of the complete lyrics -> coarse line timings;
  B. WhisperX's ``align`` per lyric line inside its coarse window (+ padding) -> refined timings.
     Lines where stage B fails or looks implausible keep the stage-A result.

Japanese: the wav2vec2 vocabulary does not contain every kanji, so each pykakasi segment whose
characters are not all in the vocabulary is aligned through its hiragana reading and mapped back to
the original characters. Words = pykakasi segments (Japanese) or whitespace tokens (other languages).
"""
import re
import unicodedata

import numpy as np

SECTION_RE = re.compile(r"^\s*([\[【(（<].*[\]】)）>])\s*$")
SAMPLE_RATE = 16000


# ---------------------------------------------------------------------------- lyrics parsing
def parse_lyrics(text):
    lines, section = [], None
    for raw in text.splitlines():
        s = unicodedata.normalize("NFKC", raw).strip()
        if not s:
            continue
        m = SECTION_RE.match(s)
        if m:
            section = m.group(1).strip("[]【】()（）<>").strip()
            continue
        lines.append({"index": len(lines), "text": s, "section": section})
    if not lines:
        raise ValueError("The lyrics are empty")
    return lines


def _kakasi():
    import pykakasi
    return pykakasi.kakasi()


def _katakana_to_hiragana(s):
    return "".join(chr(ord(c) - 0x60) if "ァ" <= c <= "ヶ" else c for c in s)


def build_tokens(lines, dictionary, language="ja"):
    """Alignment units. Returns (tokens, words): tokens carry (line, char_start, char_end, label_id, char)."""
    tokens, words = [], []
    dic = dictionary
    kks = _kakasi() if language == "ja" else None
    for ln in lines:
        text = ln["text"]
        if language == "ja":
            pos = 0
            for seg in kks.convert(text):
                orig = seg["orig"]
                start = text.find(orig, pos)
                if start < 0:
                    start = pos
                end = start + len(orig)
                pos = end
                if not orig.strip():
                    continue
                lead = len(orig) - len(orig.lstrip())
                orig = orig.strip()
                start = start + lead
                end = start + len(orig)
                w = {"line": ln["index"], "text": orig, "char_start": start, "char_end": end}
                content = [(k, c) for k, c in enumerate(orig)
                           if not unicodedata.category(c).startswith(("P", "Z", "S", "C"))]
                if not content:
                    continue  # punctuation only
                if all(c.lower() in dic for _, c in content):
                    for k, c in content:
                        tokens.append({"line": ln["index"], "c0": start + k, "c1": start + k + 1,
                                       "id": dic[c.lower()], "char": c})
                else:
                    reading = _katakana_to_hiragana(seg.get("hira", "") or "")
                    rd = [c for c in reading if c.lower() in dic]
                    if not rd:  # try katakana reading
                        rd = [c for c in (seg.get("kana", "") or "") if c.lower() in dic]
                    for c in rd:
                        tokens.append({"line": ln["index"], "c0": start, "c1": end, "id": dic[c.lower()], "char": c})
                words.append(w)
        else:
            sep = dic.get("|")
            for m in re.finditer(r"\S+", text):
                w = {"line": ln["index"], "text": m.group(0), "char_start": m.start(), "char_end": m.end()}
                words.append(w)
                if sep is not None and tokens and tokens[-1]["line"] == ln["index"]:
                    tokens.append({"line": ln["index"], "c0": m.start(), "c1": m.start(), "id": sep, "char": "|",
                                   "is_sep": True})
                for k, c in enumerate(m.group(0)):
                    if c.lower() in dic:
                        tokens.append({"line": ln["index"], "c0": m.start() + k, "c1": m.start() + k + 1,
                                       "id": dic[c.lower()], "char": c.lower()})
    return tokens, words


# ---------------------------------------------------------------------------- emissions
def load_align_model(language, device):
    """WhisperX's alignment model loader (falls back to transformers if WhisperX cannot be imported)."""
    try:
        import whisperx
        model, meta = whisperx.load_align_model(language_code=language, device=device)
        meta = dict(meta)
        meta["loader"] = "whisperx"
        return model, meta
    except Exception as e:  # pragma: no cover - depends on the environment
        print(f"[warn] whisperx.load_align_model failed ({e!r}); loading the model with transformers")
        from transformers import Wav2Vec2ForCTC, Wav2Vec2Processor
        name = {"ja": "jonatasgrosman/wav2vec2-large-xlsr-53-japanese",
                "en": "facebook/wav2vec2-base-960h"}.get(language)
        if name is None:
            raise
        proc = Wav2Vec2Processor.from_pretrained(name)
        model = Wav2Vec2ForCTC.from_pretrained(name).to(device)
        vocab = proc.tokenizer.get_vocab()
        return model, {"language": language, "dictionary": {k.lower(): v for k, v in vocab.items()},
                       "type": "huggingface", "loader": "transformers"}


def blank_id(meta):
    d = meta["dictionary"]
    for k in ("[pad]", "<pad>"):
        if k in d:
            return d[k]
    return 0


def compute_emissions(model, meta, audio16k, device, window_s=20.0, pad_s=1.0):
    """Log-probabilities (T_frames, vocab) for the whole signal; returns (logp, seconds_per_frame)."""
    import torch
    n = len(audio16k)
    win, pad = int(window_s * SAMPLE_RATE), int(pad_s * SAMPLE_RATE)
    outs = []
    frame_dur = None
    model.eval()
    for s in range(0, n, win):
        a, b = max(0, s - pad), min(n, s + win + pad)
        seg = np.asarray(audio16k[a:b], dtype=np.float32)
        if len(seg) < 400:
            seg = np.pad(seg, (0, 400 - len(seg)))
        x = torch.as_tensor(seg)[None].to(device)
        with torch.inference_mode():
            if meta.get("type") == "torchaudio":
                em, _ = model(x)
            else:
                xn = (x - x.mean()) / (x.std() + 1e-7)
                em = model(xn).logits
            em = torch.log_softmax(em.float(), dim=-1)[0].cpu().numpy()
        fd = (b - a) / SAMPLE_RATE / em.shape[0]
        frame_dur = fd if frame_dur is None else frame_dur
        k0 = int(round((s - a) / SAMPLE_RATE / fd))
        k1 = k0 + int(round((min(n, s + win) - s) / SAMPLE_RATE / fd))
        outs.append(em[k0:min(k1, em.shape[0])])
    return np.concatenate(outs, axis=0), frame_dur


# ---------------------------------------------------------------------------- Viterbi CTC
def ctc_force_align(logp, labels, blank, free_start=False, free_end=False):
    """Viterbi forced alignment. logp: (T, V); labels: (N,). Returns per-label (start, end, score) frames.

    free_start / free_end: the audio may begin after the first / stop before the last label (a cut-out
    section of the song). Labels outside the audio are left unaligned (start = end = -1).
    """
    T, N = logp.shape[0], len(labels)
    S = 2 * N + 1
    ext = np.full(S, blank, dtype=np.int64)
    ext[1::2] = labels
    allow2 = np.zeros(S, dtype=bool)
    allow2[3::2] = np.asarray(labels[1:]) != np.asarray(labels[:-1])
    if T < N and not (free_start or free_end):
        raise ValueError(f"audio too short for the lyrics ({T} frames < {N} tokens)")
    neg = -1e30
    dp = np.full(S, neg)
    if free_start:
        dp[:] = logp[0, ext]  # may enter the label sequence anywhere
    else:
        dp[0] = logp[0, blank]
        dp[1] = logp[0, ext[1]]
    bp = np.zeros((T, S), dtype=np.int8)
    for t in range(1, T):
        c0 = dp
        c1 = np.concatenate([[neg], dp[:-1]])
        c2 = np.concatenate([[neg, neg], dp[:-2]])
        c2 = np.where(allow2, c2, neg)
        stack = np.stack([c0, c1, c2])
        arg = stack.argmax(0)
        dp = stack[arg, np.arange(S)] + logp[t, ext]
        bp[t] = arg
    if free_end:
        s = int(np.argmax(dp))  # may leave the label sequence anywhere
    else:
        s = int(S - 1 if dp[S - 1] >= dp[S - 2] else S - 2)
    path = np.empty(T, dtype=np.int64)
    for t in range(T - 1, -1, -1):
        path[t] = s
        s -= int(bp[t, s])
    starts = np.full(N, -1)
    ends = np.full(N, -1)
    scores = np.zeros(N)
    counts = np.zeros(N)
    for t in range(T):
        st = path[t]
        if st % 2 == 1:
            k = (st - 1) // 2
            if starts[k] < 0:
                starts[k] = t
            ends[k] = t + 1
            scores[k] += logp[t, ext[st]]
            counts[k] += 1
    scores = scores / np.maximum(counts, 1)
    return starts, ends, scores


def _assign_times(tokens, words, lines, frame_dur, t_offset=0.0):
    """Derive word/line times from token times (characters without tokens are interpolated)."""
    for ln in lines:
        lt = [t for t in tokens if t["line"] == ln["index"] and t.get("start") is not None]
        ln["start"] = float(min(t["start"] for t in lt) + t_offset) if lt else None
        ln["end"] = float(max(t["end"] for t in lt) + t_offset) if lt else None
        ln["score"] = float(np.mean([t["score"] for t in lt])) if lt else float("nan")
    for w in words:
        wt = [t for t in tokens if t["line"] == w["line"] and t.get("start") is not None
              and t["c0"] < w["char_end"] and t["c1"] > w["char_start"] and not t.get("is_sep")]
        w["start"] = float(min(t["start"] for t in wt) + t_offset) if wt else None
        w["end"] = float(max(t["end"] for t in wt) + t_offset) if wt else None
    # interpolate missing word times inside each line
    for ln in lines:
        ws = [w for w in words if w["line"] == ln["index"]]
        known = [i for i, w in enumerate(ws) if w["start"] is not None]
        if not known:
            continue
        for i, w in enumerate(ws):
            if w["start"] is None:
                prev = max([k for k in known if k < i], default=None)
                nxt = min([k for k in known if k > i], default=None)
                if prev is not None and nxt is not None:
                    w["start"] = ws[prev]["end"]
                    w["end"] = ws[nxt]["start"]
                elif prev is not None:
                    w["start"] = w["end"] = ws[prev]["end"]
                else:
                    w["start"] = w["end"] = ws[nxt]["start"]
    return words, lines


def global_align(tokens, words, lines, logp, frame_dur, blank, free_start=False, free_end=False):
    if not tokens:
        raise ValueError("No lyric character exists in the alignment model's vocabulary")
    labels = np.array([t["id"] for t in tokens], dtype=np.int64)
    st, en, sc = ctc_force_align(logp, labels, blank, free_start, free_end)
    for t, a, b, s in zip(tokens, st, en, sc):
        t["start"], t["end"], t["score"] = (float(a * frame_dur), float(b * frame_dur), float(s)) if a >= 0 else (None, None, -99.0)
    return _assign_times(tokens, words, lines, frame_dur)


# ---------------------------------------------------------------------------- WhisperX refinement
def whisperx_refine(model, meta, audio16k, tokens, words, lines, device, language="ja", pad=0.4, max_shift=1.0,
                    log=print):
    """Re-align each line with whisperx.align inside its coarse window. Returns number of refined lines."""
    import whisperx
    dur = len(audio16k) / SAMPLE_RATE
    refined = 0
    for ln in lines:
        if ln.get("start") is None:
            continue
        ltoks = [t for t in tokens if t["line"] == ln["index"] and not t.get("is_sep")]
        if not ltoks or any(t.get("start") is None for t in ltoks):
            continue  # partly outside the processed section: keep the global alignment
        text = "".join(t["char"] for t in ltoks) if language == "ja" else ln["text"]
        seg = [{"start": max(0.0, ln["start"] - pad), "end": min(dur, ln["end"] + pad), "text": text}]
        try:
            res = whisperx.align(seg, model, meta, audio16k, device, return_char_alignments=False)
        except Exception as e:  # keep the global result
            log(f"  line {ln['index']}: whisperx.align failed ({e!r}); keeping the global alignment")
            continue
        wlist = [w for s in res.get("segments", []) for w in s.get("words", [])]
        backup = [(t["start"], t["end"]) for t in ltoks]
        if language == "ja":
            chars = [w for w in wlist if w.get("word", "").strip()]
            if len(chars) != len(ltoks) or any(c["word"].strip() != t["char"] for c, t in zip(chars, ltoks)):
                log(f"  line {ln['index']}: whisperx output does not match the tokens; keeping the global alignment")
                continue
            ok = [c for c in chars if c.get("start") is not None]
            if len(ok) < max(1, len(chars) // 2):
                continue
            for c, t in zip(chars, ltoks):
                if c.get("start") is not None:
                    t["start"], t["end"] = float(c["start"]), float(c["end"])
        else:
            lw = [w for w in words if w["line"] == ln["index"]]
            if len(wlist) != len(lw):
                log(f"  line {ln['index']}: word count mismatch; keeping the global alignment")
                continue
            for w, r in zip(lw, wlist):
                if r.get("start") is not None:
                    for t in ltoks:
                        if t["c0"] >= w["char_start"] and t["c1"] <= w["char_end"]:
                            t["start"], t["end"] = float(r["start"]), float(r["end"])
        # plausibility: keep the global result if the refined line collapsed or jumped away
        new = [(t["start"], t["end"]) for t in ltoks]
        old_span = max(e for _, e in backup) - min(s_ for s_, _ in backup)
        new_span = max(e for _, e in new) - min(s_ for s_, _ in new)
        shift = abs(min(s_ for s_, _ in new) - min(s_ for s_, _ in backup))
        if new_span < 0.3 * old_span or shift > max_shift:
            for t, (s0, e0) in zip(ltoks, backup):
                t["start"], t["end"] = s0, e0
            log(f"  line {ln['index']}: whisperx result implausible (span {new_span:.2f}s vs {old_span:.2f}s, "
                f"shift {shift:.2f}s); keeping the global alignment")
            continue
        refined += 1
    _assign_times(tokens, words, lines, None)
    return refined


# ---------------------------------------------------------------------------- manual corrections
def apply_overrides(lines, words, line_overrides=None, global_offset=0.0):
    """line_overrides: {line_index: (start, end)}; words inside a line are rescaled linearly."""
    line_overrides = line_overrides or {}
    for ln in lines:
        if ln.get("start") is None:
            continue
        s0, e0 = ln["start"], ln["end"]
        s1, e1 = line_overrides.get(ln["index"], (s0, e0))
        s1, e1 = s1 + global_offset, e1 + global_offset
        scale = (e1 - s1) / max(e0 - s0, 1e-6)
        for w in words:
            if w["line"] == ln["index"] and w.get("start") is not None:
                w["start"] = s1 + (w["start"] - s0) * scale
                w["end"] = s1 + (w["end"] - s0) * scale
        ln["start"], ln["end"] = s1, e1
        ln["manual"] = ln["index"] in line_overrides
    return lines, words


def align_lyrics(vocals_path, lyrics_text, language="ja", device="cuda", refine=True, free_start=False,
                 free_end=False, log=print):
    """Full step 2. Returns (lines, words, info).

    free_start / free_end: set when only a section of the song is processed, so that lyric lines sung
    before / after that section stay unaligned instead of being squeezed into it.
    """
    import gc

    import librosa
    import torch
    lines = parse_lyrics(lyrics_text)
    audio, _ = librosa.load(vocals_path, sr=SAMPLE_RATE, mono=True)
    model, meta = load_align_model(language, device)
    tokens, words = build_tokens(lines, meta["dictionary"], language)
    missing = [ln["index"] for ln in lines if not any(t["line"] == ln["index"] for t in tokens)]
    if missing:
        log(f"[warn] lines without alignable characters: {missing}")
    logp, frame_dur = compute_emissions(model, meta, audio, device)
    global_align(tokens, words, lines, logp, frame_dur, blank_id(meta), free_start, free_end)
    outside = [ln["index"] for ln in lines if ln.get("start") is None]
    if outside:
        log(f"lines not found in this section of the song (left unaligned): {outside}")
    n_ref = 0
    if refine:
        try:
            n_ref = whisperx_refine(model, meta, audio, tokens, words, lines, device, language, log=log)
        except ImportError as e:
            log(f"[warn] whisperx is not importable ({e!r}); using the global alignment only")
    info = {"tokens": len(tokens), "frames": int(logp.shape[0]), "frame_dur": frame_dur, "refined_lines": n_ref,
            "unaligned_lines": outside,
            "loader": meta.get("loader"), "model_type": meta.get("type")}
    del model, logp
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    return lines, words, info

In [ ]:
%%writefile /content/m2d_pkg/m2d/choreo_llm.py
"""Step 5: choreography script from lyrics + timing + 8 counts + genre, written by an LLM via OpenRouter."""
import json
import re
import time

import requests

OPENROUTER_URL = "https://openrouter.ai/api/v1/chat/completions"
OPENROUTER_MODELS_URL = "https://openrouter.ai/api/v1/models"
COUNTS = [f"{i}{s}" for i in range(1, 9) for s in ("", "&")]
COUNT_ORDER = {c: i for i, c in enumerate(COUNTS)}

SYSTEM_PROMPT = """You are an experienced choreographer who designs lyric-driven dance choreography.
You receive song lyrics with timings, an 8-count grid (2 bars of 4/4 = counts 1..8, off-beats are "&"),
and the dance genre. A dance model will dance the whole song; you only choose a FEW lyric moments where a
gesture should express the meaning of the words (mainly in the chorus or emotionally strong lines).
Do NOT put a gesture on every line. Leave space between gestures.

Each gesture will be generated by a text-to-motion model trained on HumanML3D, so `motion_description`
must be a short, concrete, physical English sentence in HumanML3D caption style, starting with
"a person", describing body parts and directions (e.g. "a person raises both arms high above the head
and spreads them wide", "a person points forward with the right hand", "a person crouches down then jumps up").
No metaphors, no emotions, no props, no camera, at most 25 words.

`scope` is "upper" if the gesture is expressed with arms/hands/torso while the legs keep dancing,
"full" if it needs the whole body (jump, crouch, turn, lunge, kneel).

The gesture's key pose ("決め") lands exactly on `hit_count` of `eight`. `start_count` and `end_count`
delimit the gesture inside the same eight (start <= hit <= end). Prefer hits on strong counts that
coincide with the important word.

Answer with ONE JSON object only, no markdown, following this schema:
{"concept": "<1-2 sentences: overall idea, in the language of the lyrics>",
 "gestures": [
   {"line_index": <int>, "lyric": "<the words expressed>", "meaning": "<why this gesture, in the language of the lyrics>",
    "motion_description": "<English HumanML3D-style sentence>",
    "eight": <int>, "start_count": "<count>", "hit_count": "<count>", "end_count": "<count>",
    "scope": "upper" | "full", "priority": <1 (must) | 2 | 3 (optional)>}
 ]}
Valid counts: 1, 1&, 2, 2&, ..., 8, 8&."""


def build_user_prompt(lines, grid, genre, max_gestures, min_gap_beats, extra_instructions=""):
    out = [f"Dance genre: {genre}", f"Tempo: {grid['bpm']:.1f} BPM, beat = {grid['beat_period']:.3f} s",
           f"Number of eights: {len(grid['eights'])} (indices 0..{len(grid['eights']) - 1})",
           f"Maximum number of gestures: {max_gestures}; at least {min_gap_beats} beats between two gestures' hits.", ""]
    out.append("LYRICS (index | section | start-end seconds | eights | text):")
    for ln in lines:
        if ln.get("start") is None:
            out.append(f"{ln['index']} | {ln.get('section') or '-'} | (not aligned) | - | {ln['text']}")
            continue
        out.append(f"{ln['index']} | {ln.get('section') or '-'} | {ln['start']:.2f}-{ln['end']:.2f} | "
                   f"{','.join(map(str, ln.get('eights', [])))} | {ln['text']}")
    out.append("")
    out.append("8-COUNT GRID (eight | start-end seconds | lyric words on the count where they start):")
    for e in grid["eights"]:
        words = " ".join(f"[{w['count']}]{w['text']}" for w in e.get("words", []))
        flag = " (irregular)" if e.get("irregular") else ""
        out.append(f"E{e['index']} | {e['start']:.2f}-{e['end']:.2f}{flag} | {words}")
    if extra_instructions:
        out += ["", "ADDITIONAL INSTRUCTIONS FROM THE USER:", extra_instructions]
    return "\n".join(out)


def list_models(api_key=None, timeout=30):
    headers = {"Authorization": f"Bearer {api_key}"} if api_key else {}
    r = requests.get(OPENROUTER_MODELS_URL, headers=headers, timeout=timeout)
    r.raise_for_status()
    return [m["id"] for m in r.json().get("data", [])]


def call_openrouter(messages, model, api_key, temperature=0.7, json_mode=True, timeout=180, max_tokens=4000):
    headers = {"Authorization": f"Bearer {api_key}", "Content-Type": "application/json",
               "HTTP-Referer": "https://colab.research.google.com", "X-Title": "music2dance"}
    body = {"model": model, "messages": messages, "temperature": temperature, "max_tokens": max_tokens}
    if json_mode:
        body["response_format"] = {"type": "json_object"}
    for attempt in range(5):
        r = requests.post(OPENROUTER_URL, headers=headers, json=body, timeout=timeout)
        if r.status_code == 400 and json_mode and "response_format" in r.text:
            body.pop("response_format", None)  # provider without JSON mode
            continue
        if r.status_code in (429, 500, 502, 503, 504):
            time.sleep(2 ** attempt * 2)
            continue
        if r.status_code != 200:
            raise RuntimeError(f"OpenRouter error {r.status_code}: {r.text[:500]}")
        data = r.json()
        if "choices" not in data or not data["choices"]:
            raise RuntimeError(f"OpenRouter returned no choices: {str(data)[:500]}")
        return data["choices"][0]["message"]["content"] or ""
    raise RuntimeError("OpenRouter: too many retries")


def extract_json(text):
    text = text.strip()
    text = re.sub(r"^```(?:json)?\s*|\s*```$", "", text)
    try:
        return json.loads(text)
    except json.JSONDecodeError:
        pass
    start = text.find("{")
    if start < 0:
        raise ValueError("no JSON object found in the answer")
    depth = 0
    for i in range(start, len(text)):
        if text[i] == "{":
            depth += 1
        elif text[i] == "}":
            depth -= 1
            if depth == 0:
                return json.loads(text[start:i + 1])
    raise ValueError("no JSON object found in the answer")


def _norm_count(c):
    c = str(c).strip().replace("＆", "&").replace(" ", "").lower().replace("and", "&")
    if c.endswith(".0"):
        c = c[:-2]
    return c


def validate_script(obj, grid, lines, max_gestures, min_gap_beats, count_time):
    """Return (clean gestures, warnings). Raises ValueError if the answer is unusable."""
    if not isinstance(obj, dict) or not isinstance(obj.get("gestures"), list):
        raise ValueError('top-level object must contain a "gestures" list')
    warnings, clean, errors = [], [], []
    n_eights = len(grid["eights"])
    for k, g in enumerate(obj["gestures"]):
        try:
            e = int(g["eight"])
            sc, hc, ec = (_norm_count(g.get(x, "")) for x in ("start_count", "hit_count", "end_count"))
            if not (0 <= e < n_eights):
                raise ValueError(f"eight {e} out of range 0..{n_eights - 1}")
            for c in (sc, hc, ec):
                if c not in COUNT_ORDER:
                    raise ValueError(f"invalid count {c!r}")
                if c not in grid["eights"][e]["counts"]:
                    raise ValueError(f"count {c} does not exist in the (short) eight {e}")
            if not (COUNT_ORDER[sc] <= COUNT_ORDER[hc] <= COUNT_ORDER[ec]):
                sc, ec = min(sc, hc, key=COUNT_ORDER.get), max(ec, hc, key=COUNT_ORDER.get)
                warnings.append(f"gesture {k}: start/hit/end reordered")
            desc = re.sub(r"\s+", " ", str(g["motion_description"])).strip().rstrip(".")
            if not re.match(r"^[\x20-\x7E]+$", desc):
                raise ValueError("motion_description must be plain English ASCII")
            if not desc.lower().startswith("a person"):
                desc = "a person " + desc[0].lower() + desc[1:]
            if len(desc.split()) > 30:
                desc = " ".join(desc.split()[:30])
                warnings.append(f"gesture {k}: description truncated")
            scope = str(g.get("scope", "upper")).lower().strip()
            if scope not in ("upper", "full"):
                warnings.append(f"gesture {k}: scope {scope!r} -> upper")
                scope = "upper"
            li = g.get("line_index")
            li = int(li) if li is not None and str(li).lstrip("-").isdigit() else None
            clean.append({"line_index": li, "lyric": str(g.get("lyric", "")), "meaning": str(g.get("meaning", "")),
                          "motion_description": desc, "eight": e, "start_count": sc, "hit_count": hc, "end_count": ec,
                          "scope": scope, "priority": int(g.get("priority", 2) or 2),
                          "hit_time": float(count_time(grid, e, hc)), "start_time": float(count_time(grid, e, sc)),
                          "end_time": float(count_time(grid, e, ec))})
        except (KeyError, TypeError, ValueError) as ex:
            errors.append(f"gesture {k} is invalid: {ex}")
    if errors and len(errors) * 2 > len(obj["gestures"]):
        raise ValueError("; ".join(errors[:5]))
    warnings += [e + " (dropped)" for e in errors]
    if not clean:
        raise ValueError("the script contains no gesture")
    # enforce spacing and maximum count (keep higher priority first, then earlier)
    gap = min_gap_beats * grid["beat_period"]
    kept = []
    for g in sorted(clean, key=lambda d: (d["priority"], d["hit_time"])):
        if len(kept) >= max_gestures:
            warnings.append(f"dropped (max {max_gestures}): {g['lyric']!r}")
            continue
        if any(abs(g["hit_time"] - h["hit_time"]) < gap for h in kept):
            warnings.append(f"dropped (too close to another gesture): {g['lyric']!r} at {g['hit_time']:.2f}s")
            continue
        kept.append(g)
    kept.sort(key=lambda d: d["hit_time"])
    for i, g in enumerate(kept):
        g["id"] = i
    return kept, warnings


def generate_script(lines, grid, genre, api_key, model, count_time, max_gestures=10, min_gap_beats=8,
                    temperature=0.7, retries=4, extra_instructions="", log=print):
    messages = [{"role": "system", "content": SYSTEM_PROMPT},
                {"role": "user", "content": build_user_prompt(lines, grid, genre, max_gestures, min_gap_beats,
                                                              extra_instructions)}]
    last_err = None
    for attempt in range(1, retries + 1):
        text = call_openrouter(messages, model, api_key, temperature=temperature)
        try:
            obj = extract_json(text)
            gestures, warns = validate_script(obj, grid, lines, max_gestures, min_gap_beats, count_time)
            for w in warns:
                log("  [note] " + w)
            return {"concept": obj.get("concept", ""), "gestures": gestures, "raw": text, "model": model,
                    "attempts": attempt}
        except (ValueError, json.JSONDecodeError) as ex:
            last_err = ex
            log(f"  attempt {attempt}: could not use the answer ({ex}); retrying")
            messages = messages[:2] + [{"role": "assistant", "content": text[:6000]},
                                       {"role": "user", "content": f"Your answer was invalid: {ex}. "
                                        "Reply again with ONE corrected JSON object following the schema exactly."}]
    raise RuntimeError(f"LLM script generation failed after {retries} attempts: {last_err}")

In [ ]:
%%writefile /content/m2d_pkg/m2d/mgpt_runner.py
"""MotionGPT3 text-to-motion in a separate process (step 7).

MotionGPT3 needs transformers==4.47.1 internals (its MoT-GPT2 code) and a few old packages; they are
installed in an override directory that is put first on PYTHONPATH only for this process, so the main
notebook keeps its own versions. Only the generation path is used: the evaluation metrics, renderers
and datasets of the repository are replaced by stubs (they need extra downloads and are not used).

Usage:
  python mgpt_runner.py prepare --repo REPO --gpt2 GPT2_DIR
  python mgpt_runner.py generate --repo REPO --ckpt CKPT --requests REQ.json --out OUT_DIR [--seed 0]
REQ.json: [{"id": 0, "text": "a person raises both arms", "n": 4, "seconds": 4.0}, ...]
Output : OUT_DIR/gesture_<id>.npz with joints (n, L, 22, 3) y-up at 20 fps and feats (n, L, 263).
"""
import argparse
import importlib
import json
import os
import shutil
import sys
import types

PROMPT_TEMPLATE = "Generate motion: {}"


def _stub(name):
    import importlib.machinery
    mod = types.ModuleType(name)
    mod.__spec__ = importlib.machinery.ModuleSpec(name, None)
    sys.modules[name] = mod
    return mod


def install_stubs():
    import torch.nn as nn
    # Evaluation metrics (bert_score, spacy models, TM2T evaluators...) are not needed for generation.
    metrics = _stub("motGPT.metrics")

    class BaseMetrics(nn.Module):
        def __init__(self, *args, **kwargs):
            super().__init__()

    metrics.BaseMetrics = BaseMetrics
    # Renderers need pyrender / moviepy.editor.
    render_utils = _stub("motGPT.utils.render_utils")
    render_utils.render_motion = lambda *a, **k: None
    try:
        import spacy  # noqa: F401
    except Exception:
        _stub("spacy")  # only imported (never used) by the training datasets
    try:
        import scipy.ndimage.filters  # noqa: F401
    except Exception:
        import scipy.ndimage
        sys.modules["scipy.ndimage.filters"] = scipy.ndimage
    try:
        import rich  # noqa: F401
        import rich.progress  # noqa: F401
    except Exception:
        rich = _stub("rich")
        prog = _stub("rich.progress")
        prog.track = lambda it, *a, **k: it
        rich.progress = prog


def build_cfg():
    from omegaconf import OmegaConf
    from motGPT.config import get_module_config
    try:
        OmegaConf.register_new_resolver("eval", eval)
    except ValueError:
        pass
    cfg_assets = OmegaConf.load("./configs/assets.yaml")
    cfg_base = OmegaConf.load(os.path.join(cfg_assets.CONFIG_FOLDER, "default.yaml"))
    cfg_exp = OmegaConf.merge(cfg_base, OmegaConf.load("./configs/webui.yaml"))
    if not cfg_exp.FULL_CONFIG:
        cfg_exp = get_module_config(cfg_exp, cfg_assets.CONFIG_FOLDER)
    cfg = OmegaConf.merge(cfg_exp, cfg_assets)
    cfg.DEBUG = False
    cfg.METRIC.TYPE = []
    return cfg


def load_model(repo, ckpt, device):
    os.chdir(repo)
    sys.path.insert(0, repo)
    install_stubs()
    import torch
    from motGPT.data.build_data import build_data
    from motGPT.models.build_model import build_model
    base = importlib.import_module("motGPT.models.base")
    # The evaluation metrics are replaced by an empty module (BaseModel code still refers to self.metrics).
    base.BaseModel.configure_metrics = lambda self: setattr(self, "metrics", sys.modules["motGPT.metrics"].BaseMetrics())
    cfg = build_cfg()
    datamodule = build_data(cfg, phase="test")
    model = build_model(cfg, datamodule).eval()
    if ckpt:
        state = torch.load(ckpt, map_location="cpu", weights_only=False)
        state = state.get("state_dict", state)
        # BaseModel.load_state_dict needs the real metrics and returns nothing, so use nn.Module's loader
        # on the network weights only (evaluation-metric and loss buffers are not used for generation).
        ignore = ("metrics.", "_losses.")
        state = {k: v for k, v in state.items() if not k.startswith(ignore) and "Metrics" not in k}
        res = torch.nn.Module.load_state_dict(model, state, strict=False)
        missing = [k for k in res.missing_keys if not k.startswith(ignore)]
        unexpected = list(res.unexpected_keys)
        print(f"[MotionGPT3] loaded {ckpt}: missing={len(missing)} unexpected={len(unexpected)}")
        core_missing = [k for k in missing if k.startswith(("lm.", "vae."))]
        if core_missing:
            raise RuntimeError(f"checkpoint does not match the model: missing {core_missing[:10]}")
        if missing or unexpected:
            print(f"[MotionGPT3] note: missing={missing[:5]} unexpected={unexpected[:5]}")
    model.to(device)
    return model, datamodule


def generate(model, datamodule, text, n, seconds, seed):
    import torch
    lm, vae = model.lm, model.vae
    L = max(40, min(196, int(round(seconds * datamodule.fps / 4.0)) * 4))
    torch.manual_seed(seed)
    with torch.no_grad():
        try:
            # Official test-time path ('Generate motion: <Caption_Placeholder>' template).
            outputs = lm.generate_conditional(texts=[text] * n, lengths=[L] * n, task="t2m", stage="test")
        except Exception as e:  # pragma: no cover
            print(f"[MotionGPT3] generate_conditional failed ({e!r}); using generate_direct_motion")
            outputs = lm.generate_direct_motion([PROMPT_TEMPLATE.format(text) + " \n"] * n, motion_tokens=None)
        lat, _ = lm.sample_tokens(outputs, lm.device, temperature=1.0, cfg=model.guidance_scale,
                                  vae_mean_std_inv=vae.mean_std_inv)
        lat = lat.reshape(n, vae.latent_size, -1).permute(1, 0, 2)
        feats = vae.decode(lat, lengths=[L] * n)
        joints = datamodule.feats2joints(feats)
        feats = datamodule.denormalize(feats)
    return joints.float().cpu().numpy(), feats.float().cpu().numpy()


def cmd_prepare(args):
    """scripts/gen_mot_gpt.py of MotionGPT3, with explicit paths."""
    import torch
    from transformers import GPT2LMHeadModel
    target = os.path.join(args.repo, "deps", "mot-gpt2")
    model = GPT2LMHeadModel.from_pretrained(args.gpt2).eval()
    new_sd = {}
    for name, p in model.state_dict().items():
        name = name.replace("attn.c_attn", "c_attn.fn.0").replace("attn.c_proj", "c_proj.fn.0")
        name = name.replace("mlp", "mlp.fn.0").replace("ln_f", "ln_f.fn.0").replace("ln_1", "ln_1.fn.0")
        name = name.replace("ln_2", "ln_2.fn.0")
        new_sd[name] = p
    os.makedirs(target, exist_ok=True)
    torch.save(new_sd, os.path.join(target, "model_state_dict.pth"))
    model.config.to_json_file(os.path.join(target, "config.json"))
    model.generation_config.to_json_file(os.path.join(target, "generation_config.json"))
    for f in ("tokenizer.json", "tokenizer_config.json", "vocab.json", "merges.txt"):
        if os.path.exists(os.path.join(args.gpt2, f)):
            shutil.copy(os.path.join(args.gpt2, f), os.path.join(target, f))
    print(f"[MotionGPT3] prepared {target}")


def cmd_generate(args):
    import numpy as np
    import torch
    device = "cuda" if torch.cuda.is_available() else "cpu"
    reqs = json.load(open(args.requests, encoding="utf-8"))
    model, dm = load_model(os.path.abspath(args.repo), args.ckpt, device)
    os.makedirs(args.out, exist_ok=True)
    for k, r in enumerate(reqs):
        joints, feats = generate(model, dm, r["text"], int(r.get("n", 4)), float(r.get("seconds", 4.0)),
                                 args.seed + 1000 * int(r["id"]) + k)
        np.savez(os.path.join(args.out, f"gesture_{r['id']}.npz"), joints=joints, feats=feats, text=r["text"],
                 fps=dm.fps)
        print(f"[MotionGPT3] gesture {r['id']}: {joints.shape} '{r['text']}'", flush=True)


def main():
    ap = argparse.ArgumentParser()
    sub = ap.add_subparsers(dest="cmd", required=True)
    p = sub.add_parser("prepare")
    p.add_argument("--repo", required=True)
    p.add_argument("--gpt2", required=True)
    g = sub.add_parser("generate")
    g.add_argument("--repo", required=True)
    g.add_argument("--ckpt", default="")
    g.add_argument("--requests", required=True)
    g.add_argument("--out", required=True)
    g.add_argument("--seed", type=int, default=0)
    args = ap.parse_args()
    {"prepare": cmd_prepare, "generate": cmd_generate}[args.cmd](args)


if __name__ == "__main__":
    main()

In [ ]:
%%writefile /content/m2d_pkg/m2d/motion_ik.py
"""HumanML3D joints -> FlowerDance motion (the hardest conversion of the pipeline).

Input : MotionGPT3 output, 22 joint positions, y-up, 20 fps (HumanML3D skeleton/proportions).
Output: SMPL 24 local rotations + root position at 30 fps on FlowerDance's skeleton (y-up here;
        the z-up conversion happens when the clip is embedded).

Steps
  1. chirality check (left x up must point forward, using the toe direction) and 20 -> 30 fps
     cubic-spline resampling of the joint positions;
  2. analytic IK: Kabsch fit of the pelvis (hips + spine) and of spine3 (neck + collars), minimal
     ("swing") rotations along every single-child bone;
  3. optimisation refinement (Adam on 6D rotations): bone-direction error + anatomical hinge priors
     for elbows/knees (no sideways bending / hyper-extension) + temporal smoothness + small prior to
     the analytic solution. Bone directions are scale-free, so the different body proportions of
     HumanML3D and SMPL do not matter; the root translation is rescaled by the leg-length ratio.
Joints without positional information (head, wrists, hands, toes) get identity local rotations and
are never copied into the dance (see smpl_utils.UPPER_JOINTS_DEFAULT / FULL_JOINTS_DEFAULT).
"""
import numpy as np
import torch
import torch.nn.functional as F
from pytorch3d.transforms import (axis_angle_to_matrix, matrix_to_axis_angle,
                                  matrix_to_rotation_6d, rotation_6d_to_matrix)

from . import smpl_utils as su

HML_FPS = 20
N22 = 22
OPT_JOINTS = [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 12, 13, 14, 16, 17, 18, 19]
TARGET_BONES = [(su.SMPL_PARENTS[c], c) for c in range(1, N22)]
KNEES = (4, 5)
ELBOWS = (18, 19)


# ---------------------------------------------------------------------------- preprocessing
def chirality_sign(P):
    """+1 if (left x up) points to the toes (SMPL convention), -1 if the skeleton is mirrored."""
    P = np.asarray(P, dtype=np.float64)
    left = P[:, 1] - P[:, 2]
    up = np.array([0.0, 1.0, 0.0])
    fwd = (P[:, 10] - P[:, 7]) + (P[:, 11] - P[:, 8])
    fwd[:, 1] = 0.0
    s = np.einsum("ti,ti->t", np.cross(left, up), fwd)
    return 1.0 if np.median(s) >= 0 else -1.0


def fix_chirality(P):
    """Mirror x if needed so that the joint labels match SMPL's handedness. Returns (P, mirrored)."""
    if chirality_sign(P) < 0:
        P = np.array(P, copy=True)
        P[..., 0] *= -1.0
        return P, True
    return np.asarray(P), False


def resample(P, src_fps=HML_FPS, dst_fps=su.FPS):
    from scipy.interpolate import CubicSpline
    P = np.asarray(P, dtype=np.float64)
    T = P.shape[0]
    if T < 4:
        idx = np.round(np.linspace(0, T - 1, max(1, int(round(T * dst_fps / src_fps))))).astype(int)
        return P[idx]
    t_src = np.arange(T) / src_fps
    t_dst = np.arange(0.0, t_src[-1] + 1e-9, 1.0 / dst_fps)
    return CubicSpline(t_src, P, axis=0)(t_dst)


def body_scale(P):
    """SMPL leg length / HumanML3D leg length (both legs, median over time)."""
    O = su.SMPL_OFFSETS
    smpl_leg = 0.5 * (np.linalg.norm(O[4]) + np.linalg.norm(O[7]) + np.linalg.norm(O[5]) + np.linalg.norm(O[8]))
    P = np.asarray(P)
    hml = 0.5 * (np.linalg.norm(P[:, 4] - P[:, 1], axis=-1) + np.linalg.norm(P[:, 7] - P[:, 4], axis=-1)
                 + np.linalg.norm(P[:, 5] - P[:, 2], axis=-1) + np.linalg.norm(P[:, 8] - P[:, 5], axis=-1))
    if not np.isfinite(np.median(hml)) or np.median(hml) < 1e-3:
        return 1.0  # degenerate skeleton (e.g. collapsed legs): keep metres as they are
    return float(np.clip(smpl_leg / np.median(hml), 0.5, 2.0))


# ---------------------------------------------------------------------------- analytic IK
def analytic_ik(P):
    """P: (T,22,3) torch y-up -> local rotations (T,24,3,3)."""
    T = P.shape[0]
    dt, dev = P.dtype, P.device
    O = su.offsets_tensor(P)
    R = torch.eye(3, dtype=dt, device=dev).expand(T, su.N_JOINTS, 3, 3).clone()
    G = [None] * su.N_JOINTS
    src = F.normalize(O[[1, 2, 3]], dim=-1).expand(T, 3, 3)
    dst = F.normalize(P[:, [1, 2, 3]] - P[:, [0]], dim=-1)
    w = torch.tensor([1.0, 1.0, 1.5], dtype=dt, device=dev).expand(T, 3)
    G[0] = su.kabsch(src, dst, w)
    R[:, 0] = G[0]
    for j in range(1, su.N_JOINTS):
        Gp = G[su.SMPL_PARENTS[j]]
        ch = su.children_of(j, N22) if j < N22 else []
        if not ch:
            G[j] = Gp
            continue
        if len(ch) == 1:
            c = ch[0]
            a = (Gp @ O[c].unsqueeze(-1)).squeeze(-1)
            b = P[:, c] - P[:, j]
            Gj = su.minimal_rotation(a, b) @ Gp
        else:
            a = F.normalize((Gp.unsqueeze(1) @ O[ch].unsqueeze(-1)).squeeze(-1), dim=-1)
            b = F.normalize(P[:, ch] - P[:, j:j + 1], dim=-1)
            Gj = su.kabsch(a, b) @ Gp
        R[:, j] = Gp.transpose(-1, -2) @ Gj
        G[j] = Gj
    return R


# ---------------------------------------------------------------------------- refinement
def _bone_dirs(X):
    idx_p = [p for p, _ in TARGET_BONES]
    idx_c = [c for _, c in TARGET_BONES]
    return F.normalize(X[:, idx_c] - X[:, idx_p], dim=-1)


def refine_ik(P, R_init, iters=300, lr=0.02, w_dir=1.0, w_hinge=0.05, w_hyper=0.5, w_smooth=0.5, w_prior=0.01,
              verbose=False):
    """Optimise local rotations of OPT_JOINTS (see module docstring). P: (T,22,3), R_init: (T,24,3,3)."""
    T = P.shape[0]
    dev, dt = P.device, P.dtype
    target = _bone_dirs(P)
    r6_init = matrix_to_rotation_6d(R_init[:, OPT_JOINTS]).detach()
    r6 = r6_init.clone().requires_grad_(True)
    opt = torch.optim.Adam([r6], lr=lr)
    ex = torch.tensor([1.0, 0.0, 0.0], dtype=dt, device=dev)
    ey = torch.tensor([0.0, 1.0, 0.0], dtype=dt, device=dev)
    O = su.offsets_tensor(P)
    shin = {4: F.normalize(O[7], dim=0), 5: F.normalize(O[8], dim=0)}
    fore = {18: F.normalize(O[20], dim=0), 19: F.normalize(O[21], dim=0)}
    pos_in_opt = {j: i for i, j in enumerate(OPT_JOINTS)}
    zero_root = torch.zeros(T, 3, dtype=dt, device=dev)
    history = []
    for it in range(iters):
        Rm = rotation_6d_to_matrix(r6)
        R_full = R_init.detach().clone()
        R_full[:, OPT_JOINTS] = Rm
        X = su.fk(R_full, zero_root)
        l_dir = (1.0 - (_bone_dirs(X) * target).sum(-1)).mean()
        l_hinge = 0.0
        l_hyper = 0.0
        for k in KNEES:
            Rk = Rm[:, pos_in_opt[k]]
            l_hinge = l_hinge + ((Rk @ ex) - ex).pow(2).sum(-1).mean()
            l_hyper = l_hyper + F.relu((Rk @ shin[k])[:, 2] - shin[k][2]).mean()
        for k in ELBOWS:
            Re = Rm[:, pos_in_opt[k]]
            l_hinge = l_hinge + ((Re @ ey) - ey).pow(2).sum(-1).mean()
            l_hyper = l_hyper + F.relu(fore[k][2] - (Re @ fore[k])[:, 2]).mean()
        l_smooth = (r6[2:] - 2 * r6[1:-1] + r6[:-2]).pow(2).sum(-1).mean() if T > 2 else torch.zeros((), device=dev)
        l_prior = (r6 - r6_init).pow(2).sum(-1).mean()
        loss = w_dir * l_dir + w_hinge * l_hinge + w_hyper * l_hyper + w_smooth * l_smooth + w_prior * l_prior
        opt.zero_grad()
        loss.backward()
        opt.step()
        if verbose and (it % 50 == 0 or it == iters - 1):
            history.append((it, float(l_dir), float(l_hinge), float(l_hyper), float(l_smooth)))
    with torch.no_grad():
        R_out = R_init.clone()
        R_out[:, OPT_JOINTS] = rotation_6d_to_matrix(r6)
    return R_out, history


# ---------------------------------------------------------------------------- evaluation
def retarget_positions(P, root_pos):
    """Ideal SMPL-proportioned joints: the target bone directions laid out with SMPL bone lengths."""
    O = su.offsets_tensor(P)
    out = torch.zeros(P.shape[0], N22, 3, dtype=P.dtype, device=P.device)
    out[:, 0] = root_pos
    for c in range(1, N22):
        p = su.SMPL_PARENTS[c]
        out[:, c] = out[:, p] + F.normalize(P[:, c] - P[:, p], dim=-1) * O[c].norm()
    return out


def ik_errors(P, R, root_pos):
    """Per-joint errors between the IK result and the (retargeted) target. Returns numpy dict."""
    with torch.no_grad():
        X = su.fk(R, root_pos)[:, :N22]
        Y = retarget_positions(P, root_pos)
        pos_err = (X - Y).norm(dim=-1)  # metres
        cosv = (_bone_dirs(X) * _bone_dirs(P)).sum(-1).clamp(-1, 1)
        ang = torch.rad2deg(torch.acos(cosv))
    return {"pos_err_m": pos_err.cpu().numpy(), "bone_angle_deg": ang.cpu().numpy(),
            "fk_positions": X.cpu().numpy(), "target_positions": Y.cpu().numpy()}


# ---------------------------------------------------------------------------- full conversion
def hml_to_smpl(joints20, refine=True, iters=300, device="cpu"):
    """joints20: (T,22,3) numpy, y-up, 20 fps -> dict with 30 fps SMPL motion (y-up)."""
    P, mirrored = fix_chirality(np.asarray(joints20, dtype=np.float64))
    P30 = resample(P, HML_FPS, su.FPS)
    scale = body_scale(P30)
    Pt = torch.as_tensor(P30, dtype=torch.float32, device=device)
    R0 = analytic_ik(Pt)
    root = Pt[:, 0] * scale
    if refine:
        R, hist = refine_ik(Pt, R0, iters=iters, verbose=True)
    else:
        R, hist = R0, []
    err0 = ik_errors(Pt, R0, root)
    err = ik_errors(Pt, R, root)
    return {"rotmat": R.cpu(), "root_pos": root.cpu(), "target30": P30 * scale, "scale": scale,
            "mirrored": mirrored, "errors_analytic": err0, "errors": err, "history": hist}


# ---------------------------------------------------------------------------- keyframes & editing
def _smooth(x, sigma):
    if sigma <= 0:
        return x
    r = int(3 * sigma + 0.5)
    k = np.exp(-0.5 * (np.arange(-r, r + 1) / sigma) ** 2)
    k /= k.sum()
    xp = np.pad(x, (r, r), mode="edge")
    return np.convolve(xp, k, mode="valid")


def detect_keyframes(P30, scope="upper", fps=su.FPS, peak_window_s=0.8, sigma=1.5, edge_s=0.2):
    """'決め' candidates: local minima of end-effector speed that follow a strong movement.

    P30: (T,22,3) y-up positions at 30 fps. Returns list of dicts sorted by score (best first).
    """
    P = np.asarray(P30, dtype=np.float64)
    T = len(P)
    rel = P - P[:, :1]
    if scope == "full":
        eff, w = [20, 21, 18, 19, 7, 8], np.array([1.0, 1.0, 0.5, 0.5, 0.7, 0.7])
    else:
        eff, w = [20, 21, 18, 19], np.array([1.0, 1.0, 0.5, 0.5])
    v = np.linalg.norm(np.diff(rel[:, eff], axis=0), axis=-1) * fps  # (T-1, k)
    s = (v * w).sum(-1) / w.sum()
    if scope == "full":
        s = s + 0.7 * np.linalg.norm(np.diff(P[:, 0], axis=0), axis=-1) * fps
    s = np.concatenate([s[:1], s])
    s = _smooth(s, sigma)
    arm = np.linalg.norm(su.SMPL_OFFSETS[18]) + np.linalg.norm(su.SMPL_OFFSETS[20])
    scale = body_scale(P) if T > 0 else 1.0
    W = max(2, int(peak_window_s * fps))
    lo, hi = int(edge_s * fps), T - 1 - int(edge_s * fps)
    cands = []
    for i in range(max(1, lo), min(T - 1, hi + 1)):
        # local minimum; on a plateau only its first frame counts
        if s[i] < s[i - 1] and s[i] <= s[i + 1]:
            peak = s[max(0, i - W):i + 1].max()
            drop = peak - s[i]
            # the pose is "set" as soon as the deceleration is (almost) finished
            j = i
            while j - 1 >= lo and s[j - 1] <= s[i] + 0.1 * drop and s[j - 1] < peak:
                j -= 1
            i = j
            ext = np.linalg.norm(rel[i, [20, 21]] - rel[i, [9, 9]], axis=-1).mean() * scale / arm
            height = 0.0
            if scope == "full":
                height = abs(P[i, 0, 1] - P[0, 0, 1]) * scale
            score = drop * (0.5 + ext + 2.0 * height)
            cands.append({"frame": i, "score": float(score), "speed": float(s[i]), "peak": float(peak),
                          "extension": float(ext)})
    if not cands:  # no clear stop: take the most extended pose
        ext_all = np.linalg.norm(rel[:, [20, 21]] - rel[:, [9, 9]], axis=-1).mean(-1)
        i = int(np.argmax(ext_all[lo:hi + 1]) + lo) if hi > lo else T // 2
        cands.append({"frame": i, "score": 0.0, "speed": float(s[i]), "peak": float(s[i]),
                      "extension": float(ext_all[i] * scale / arm)})
    cands.sort(key=lambda c: -c["score"])
    return cands, s


def cut_clip(arr, key, pre, post):
    """Frames [key-pre, key+post] of arr (time on axis 0); edges are padded by repeating frames."""
    T = arr.shape[0]
    idx = np.clip(np.arange(key - pre, key + post + 1), 0, T - 1)
    return arr[idx]


def exaggerate(R, joints, factor, ref_frame=0, max_angle=np.pi * 0.95):
    """Scale each joint's rotation *relative to a reference frame* (the neutral start of the clip)."""
    if abs(factor - 1.0) < 1e-6:
        return R
    R = R.clone()
    ref = R[ref_frame:ref_frame + 1, joints]
    rel = ref.transpose(-1, -2) @ R[:, joints]
    aa = matrix_to_axis_angle(rel) * factor
    ang = aa.norm(dim=-1, keepdim=True)
    aa = torch.where(ang > max_angle, aa / ang.clamp_min(1e-8) * max_angle, aa)
    R[:, joints] = ref @ axis_angle_to_matrix(aa)
    return R

In [ ]:
%%writefile /content/m2d_pkg/m2d/gesture_pipeline.py
"""Step 7: choose the best MotionGPT3 candidate, find its key pose, cut it around the key and convert it."""
import json
import os

import numpy as np
import torch

from . import motion_ik as ik
from . import smpl_utils as su

EXAGGERATE_JOINTS_DEFAULT = [3, 6, 9, 13, 14, 16, 17, 18, 19]


def make_requests(gestures, n_candidates=4, seconds=4.0):
    return [{"id": int(g["id"]), "text": g["motion_description"], "n": int(n_candidates), "seconds": float(seconds)}
            for g in gestures]


def write_requests(gestures, path, **kw):
    with open(path, "w", encoding="utf-8") as f:
        json.dump(make_requests(gestures, **kw), f, ensure_ascii=False, indent=1)


def load_candidates(out_dir, gid):
    d = np.load(os.path.join(out_dir, f"gesture_{gid}.npz"))
    return d["joints"], int(d["fps"])


def score_candidate(P30, scope, pre, post):
    """Best key pose of one candidate (positions at 30 fps, SMPL scale)."""
    cands, speed = ik.detect_keyframes(P30, scope)
    T = len(P30)
    best = None
    for c in cands[:5]:
        k = c["frame"]
        context_ok = (k - pre >= 0) and (k + post <= T - 1)
        s = c["score"] * (1.0 if context_ok else 0.7)
        # gestures limited to the upper body should not walk away
        if scope == "upper":
            travel = np.linalg.norm(P30[-1, 0, [0, 2]] - P30[0, 0, [0, 2]])
            s *= 1.0 / (1.0 + 2.0 * max(0.0, travel - 0.3))
        if best is None or s > best["score"]:
            best = dict(c, score=float(s), context_ok=bool(context_ok))
    return best, speed


def process_gesture(g, joints20, beat_period, pre_beats=1.0, post_beats=1.0, exaggeration=1.15,
                    exaggerate_joints=None, ik_iters=300, device="cpu", fps=su.FPS):
    """g: script gesture (needs id, scope, hit_time). joints20: (n, L, 22, 3) MotionGPT3 candidates."""
    pre = int(round(pre_beats * beat_period * fps))
    post = int(round(post_beats * beat_period * fps))
    per_cand = []
    for ci, J in enumerate(joints20):
        P, mirrored = ik.fix_chirality(J)
        P30 = ik.resample(P) * ik.body_scale(ik.resample(P))
        best, speed = score_candidate(P30, g["scope"], pre, post)
        per_cand.append({"candidate": ci, "key": best["frame"], "score": best["score"], "context_ok": best["context_ok"],
                         "speed": speed, "mirrored": mirrored})
    chosen = max(per_cand, key=lambda c: c["score"])
    conv = ik.hml_to_smpl(joints20[chosen["candidate"]], refine=True, iters=ik_iters, device=device)
    R = conv["rotmat"]
    joints = list(exaggerate_joints if exaggerate_joints is not None else EXAGGERATE_JOINTS_DEFAULT)
    R = ik.exaggerate(R, joints, exaggeration, ref_frame=0)
    key = chosen["key"]
    rot_clip = torch.as_tensor(ik.cut_clip(R.numpy(), key, pre, post))
    root_clip = torch.as_tensor(ik.cut_clip(conv["root_pos"].numpy(), key, pre, post))
    return {
        "id": int(g["id"]), "scope": g["scope"], "hit_frame": int(round(g["hit_time"] * fps)),
        "rotmat": rot_clip.float(), "root_pos": root_clip.float(), "key": pre,
        "ref_height": float(conv["root_pos"][0, 1]),  # standing pelvis height (y-up == z-up after rotation)
        "info": {"candidates": [{k: v for k, v in c.items() if k != "speed"} for c in per_cand],
                 "chosen": chosen["candidate"], "key_frame_in_candidate": key, "pre": pre, "post": post,
                 "mirrored": conv["mirrored"], "scale": conv["scale"],
                 "ik_bone_angle_deg_mean": float(conv["errors"]["bone_angle_deg"].mean()),
                 "ik_bone_angle_deg_max": float(conv["errors"]["bone_angle_deg"].max()),
                 "ik_pos_err_cm_mean": float(conv["errors"]["pos_err_m"].mean() * 100),
                 "ik_pos_err_cm_max": float(conv["errors"]["pos_err_m"].max() * 100)},
        "debug": {"conv": conv, "speeds": [c["speed"] for c in per_cand], "rotmat_full": R},
    }

In [ ]:
%%writefile /content/m2d_pkg/m2d/gesture_embed.py
"""Step 8/9: put gesture clips on the beat inside the base dance and build the masks for re-generation.

Per frame / channel, the mask passed to ``FlowerDance.inpaint`` means
  1 = fixed (gesture channels inside a gesture, or the base dance elsewhere), 0 = re-generated.
Transitions of ``trans_pre`` / ``trans_post`` frames before/after every gesture are re-generated
(preparation + recovery). ``decay`` selects FlowerDance's time-decayed soft mask (True) or a strict
mask (False) per element; window borders are always strict so that the re-generated windows can be
pasted back into the base dance seamlessly.
"""
import numpy as np
import torch

from . import smpl_utils as su


def _region_channels(scope, joints):
    ch = su.joint_channels(joints)
    if scope == "full":
        ch = ch + list(range(4, 7)) + su.joint_channels([0])
    return sorted(set(ch))


def place_root_full(base_rot0, base_pos, clip_rot0_y, clip_pos_y, key_idx, ref_height, mode="hybrid", floor_z=None):
    """Root rotation/position (z-up) for a `full` gesture.

    mode 'base'  : root position and orientation of the base dance (the plan's literal rule).
    mode 'hybrid': standing position (x, y) and facing (yaw) follow the base dance; the gesture adds
                   its own height change (jump / crouch), its tilt and its yaw change relative to the
                   key frame (turns). Keeps the dancer where the base dance is, without losing the motion.
    """
    if mode == "base":
        return base_rot0.clone(), base_pos.clone()
    g_rot, g_pos = su.yup_to_zup(clip_rot0_y, clip_pos_y)
    yaw_g, tilt_g = su.split_yaw_tilt_zup(g_rot)
    yaw_g = su.unwrap(yaw_g)
    yaw_b, _ = su.split_yaw_tilt_zup(base_rot0)
    yaw_b = su.unwrap(yaw_b)
    yaw = yaw_b + (yaw_g - yaw_g[key_idx])
    rot0 = su.rot_z(yaw) @ tilt_g
    pos = base_pos.clone()
    pos[:, 2] = base_pos[:, 2] + (g_pos[:, 2] - ref_height)
    return rot0, pos


def assemble(base, gestures, trans_frames, upper_joints=None, full_joints=None, full_root_mode="hybrid",
             gesture_mask_mode="soft", base_mask_mode="soft", transition_regen="replaced", min_gap_frames=0,
             log=print):
    """Build the motion with gestures pasted in, plus mask/decay tensors.

    base: dict root_pos (T,3), rotmat (T,24,3,3), contacts (T,4) -- z-up, un-normalised.
    gestures: list of dicts with keys
        id, scope ('upper'|'full'), hit_frame, rotmat (L,24,3,3) y-up local, root_pos (L,3) y-up (SMPL scale),
        key (index of the key frame inside the clip), ref_height (standing pelvis height of the clip, z-up)
        and optional trans_pre / trans_post (frames).
    """
    upper_joints = list(upper_joints or su.UPPER_JOINTS_DEFAULT)
    full_joints = list(full_joints or su.FULL_JOINTS_DEFAULT)
    T = base["rotmat"].shape[0]
    rot = base["rotmat"].clone()
    pos = base["root_pos"].clone()
    cont = base["contacts"].clone()
    mask = torch.ones(T, su.MOTION_DIM)
    changed = torch.zeros(T, su.MOTION_DIM, dtype=torch.bool)  # channels whose values come from a gesture
    decay = torch.full((T, su.MOTION_DIM), base_mask_mode == "soft", dtype=torch.bool)
    occupied = np.zeros(T, dtype=bool)
    floor_z = su.foot_floor_level(su.fk(base["rotmat"], base["root_pos"]))
    regions = []
    for g in sorted(gestures, key=lambda d: d["hit_frame"]):
        L = g["rotmat"].shape[0]
        a = int(g["hit_frame"] - g["key"])
        b = a + L
        lo, hi = max(0, a), min(T, b)
        if hi - lo < 2:
            log(f"[skip] gesture {g['id']}: outside the song (frames {a}-{b})")
            continue
        guard_lo, guard_hi = max(0, lo - min_gap_frames), min(T, hi + min_gap_frames)
        if occupied[guard_lo:guard_hi].any():
            log(f"[skip] gesture {g['id']}: overlaps an earlier gesture (frames {lo}-{hi})")
            continue
        ci = torch.arange(lo - a, hi - a)
        scope = g["scope"]
        joints = full_joints if scope == "full" else upper_joints
        rot[lo:hi][:, joints] = g["rotmat"][ci][:, joints]
        if scope == "full":
            r0, p0 = place_root_full(base["rotmat"][lo:hi, 0], base["root_pos"][lo:hi], g["rotmat"][ci, 0],
                                     g["root_pos"][ci], int(g["key"] - (lo - a)), g["ref_height"], full_root_mode)
            rot[lo:hi, 0] = r0
            pos[lo:hi] = p0
            # keep the feet above the floor of the base dance
            X = su.fk(rot[lo:hi], pos[lo:hi])
            low = X[:, list(su.FOOT_JOINTS), 2].min(-1).values
            lift = torch.clamp(floor_z - low, min=0.0)
            pos[lo:hi, 2] += lift
            cont[lo:hi] = su.compute_contacts(su.fk(rot[lo:hi], pos[lo:hi]))
        ch = _region_channels(scope, joints)
        mask[lo:hi, ch] = 1.0
        changed[lo:hi, ch] = True
        decay[lo:hi, ch] = gesture_mask_mode == "soft"
        if scope == "full":
            mask[lo:hi, 0:4] = 0.0  # foot contacts are re-generated for whole-body gestures
        occupied[lo:hi] = True
        tp = int(g.get("trans_pre", trans_frames))
        tq = int(g.get("trans_post", trans_frames))
        regions.append({"id": g["id"], "scope": scope, "start": lo, "end": hi, "hit": int(g["hit_frame"]),
                        "pre": (max(0, lo - tp), lo), "post": (hi, min(T, hi + tq)), "channels": ch})
    # transitions (never over a gesture region)
    all_ch = list(range(su.MOTION_DIM))
    for r in regions:
        regen = all_ch if (transition_regen == "all" or r["scope"] == "full") else r["channels"]
        for s, e in (r["pre"], r["post"]):
            if e <= s:
                continue
            frames = np.arange(s, e)
            frames = frames[~occupied[s:e]]
            if len(frames):
                idx = torch.as_tensor(frames)
                mask[idx[:, None], torch.as_tensor(regen)[None, :]] = 0.0
    return {"root_pos": pos, "rotmat": rot, "contacts": cont, "changed": changed}, mask, decay, regions


def edited_vector(fd, base_norm, assembled):
    """Normalised vector for masked generation: the base dance's own values everywhere, except the
    channels replaced by gestures (so the untouched parts are bit-identical to the base generation)."""
    enc = fd.encode(assembled["root_pos"], assembled["rotmat"], assembled["contacts"])
    ch = assembled["changed"]
    clipped = (enc.abs() >= 0.999) & ch  # values at the edge of the training range (clipped by the normaliser)
    frac = float(clipped.sum()) / max(1, int(ch.sum()))
    return torch.where(ch, enc, base_norm), frac


def edit_windows(regions, T, context_frames, min_len, edge_lock):
    """Merge gesture neighbourhoods into windows for masked generation. Returns [(s, e)]."""
    spans = []
    for r in regions:
        s = r["pre"][0] - context_frames
        e = r["post"][1] + context_frames
        spans.append([max(0, s), min(T, e)])
    spans.sort()
    merged = []
    for s, e in spans:
        if merged and s <= merged[-1][1]:
            merged[-1][1] = max(merged[-1][1], e)
        else:
            merged.append([s, e])
    out = []
    for s, e in merged:
        if e - s < min_len:
            c = (s + e) // 2
            s, e = max(0, c - min_len // 2), min(T, c - min_len // 2 + min_len)
            s = max(0, e - min_len)
        out.append((int(s), int(e)))
    # merging may be needed again after enlarging
    out.sort()
    final = []
    for s, e in out:
        if final and s <= final[-1][1]:
            final[-1] = (final[-1][0], max(final[-1][1], e))
        else:
            final.append((s, e))
    return final


def lock_window_edges(mask, decay, windows, edge_lock, T):
    """Strictly fix `edge_lock` frames at each inner window border (they hold the base dance)."""
    mask = mask.clone()
    decay = decay.clone()
    for s, e in windows:
        if s > 0:
            mask[s:s + edge_lock] = 1.0
            decay[s:s + edge_lock] = False
        if e < T:
            mask[e - edge_lock:e] = 1.0
            decay[e - edge_lock:e] = False
    return mask, decay


def run_edit(fd, base_norm, edited_norm, mask, decay, windows, cond, genre, n_steps=101, seed=0, log=print):
    """Masked generation on every window; everything outside the windows stays the base dance."""
    out = base_norm.clone()
    for k, (s, e) in enumerate(windows):
        log(f"  window {k + 1}/{len(windows)}: frames {s}-{e} ({(e - s) / su.FPS:.1f}s), "
            f"fixed {float(mask[s:e].mean()) * 100:.0f}% of channels")
        out[s:e] = fd.inpaint(edited_norm[s:e], mask[s:e], cond[s:e], genre, decay=decay[s:e], n_steps=n_steps,
                              seed=seed + k)
    return out

In [ ]:
%%writefile /content/m2d_pkg/m2d/export_render.py
"""Step 10/11: exports (SMPL .pkl, joints .npz, BVH) and preview videos (skeleton / mannequin)."""
import math
import os
import pickle
import subprocess

import numpy as np
import torch

from . import smpl_utils as su

FONT_CANDIDATES = [
    "/usr/share/fonts/opentype/noto/NotoSansCJK-Regular.ttc",
    "/usr/share/fonts/opentype/noto/NotoSansCJKjp-Regular.otf",
    "/usr/share/fonts/truetype/noto/NotoSansCJK-Regular.ttc",
    "/usr/share/fonts/noto-cjk/NotoSansCJK-Regular.ttc",
    "/usr/share/fonts/truetype/fonts-japanese-gothic.ttf",
    "/usr/share/fonts/truetype/dejavu/DejaVuSans.ttf",
]


# ---------------------------------------------------------------------------- exports
def export_smpl_pkl(motion, path, frame="zup"):
    """motion: dict root_pos (T,3), rotmat (T,24,3,3) in FlowerDance's z-up frame.

    frame='zup': identical layout to FlowerDance's inference output
                 {smpl_poses (T,72) axis-angle, smpl_trans (T,3), full_pose (T,24,3)}.
    frame='yup': rotated back to the SMPL/AIST++ y-up frame, AIST++-style keys
                 {smpl_poses, smpl_trans, smpl_scaling=[1.0]} (+ full_pose in y-up).
    """
    rot = motion["rotmat"].float().clone()
    pos = motion["root_pos"].float().clone()
    if frame == "yup":
        rot[:, 0], pos = su.zup_to_yup(rot[:, 0], pos)
    joints = su.fk(rot, pos)
    data = {"smpl_poses": su.rotmat_to_aa(rot).reshape(-1, 72).numpy().astype(np.float32),
            "smpl_trans": pos.numpy().astype(np.float32), "full_pose": joints.numpy().astype(np.float32)}
    if frame == "yup":
        data["smpl_scaling"] = np.array([1.0], dtype=np.float32)
    with open(path, "wb") as f:
        pickle.dump(data, f)
    return path


def export_bvh(motion, path, fps=su.FPS, scale=100.0):
    """BVH (y-up, centimetres, ZXY Euler) of the SMPL skeleton for Blender/MotionBuilder."""
    from scipy.spatial.transform import Rotation
    rot = motion["rotmat"].float().clone()
    pos = motion["root_pos"].float().clone()
    rot[:, 0], pos = su.zup_to_yup(rot[:, 0], pos)
    T = rot.shape[0]
    children = {j: su.children_of(j) for j in range(su.N_JOINTS)}
    lines = ["HIERARCHY"]

    def write(j, depth):
        ind = "  " * depth
        off = su.SMPL_OFFSETS[j] * scale
        if j == 0:
            lines.append(f"ROOT {su.JOINT_NAMES[j]}")
        else:
            lines.append(f"{ind}JOINT {su.JOINT_NAMES[j]}")
        lines.append(f"{ind}{{")
        lines.append(f"{ind}  OFFSET {off[0]:.6f} {off[1]:.6f} {off[2]:.6f}")
        if j == 0:
            lines.append(f"{ind}  CHANNELS 6 Xposition Yposition Zposition Zrotation Xrotation Yrotation")
        else:
            lines.append(f"{ind}  CHANNELS 3 Zrotation Xrotation Yrotation")
        if children[j]:
            for c in children[j]:
                write(c, depth + 1)
        else:
            lines.append(f"{ind}  End Site")
            lines.append(f"{ind}  {{")
            lines.append(f"{ind}    OFFSET 0.000000 0.000000 0.000000")
            lines.append(f"{ind}  }}")
        lines.append(f"{ind}}}")

    write(0, 0)
    order = []

    def dfs(j):
        order.append(j)
        for c in children[j]:
            dfs(c)

    dfs(0)
    lines += ["MOTION", f"Frames: {T}", f"Frame Time: {1.0 / fps:.6f}"]
    eul = {j: Rotation.from_matrix(rot[:, j].numpy()).as_euler("ZXY", degrees=True) for j in order}
    p = pos.numpy() * scale
    for t in range(T):
        vals = [f"{v:.4f}" for v in p[t]]
        for j in order:
            vals += [f"{v:.4f}" for v in eul[j][t]]
        lines.append(" ".join(vals))
    with open(path, "w") as f:
        f.write("\n".join(lines) + "\n")
    return path


# ---------------------------------------------------------------------------- camera / overlays
class Camera:
    def __init__(self, positions, root_rot, width=640, height=640, fov_deg=40.0, distance=4.2, elevation_deg=12.0,
                 azimuth_offset_deg=0.0, follow=True, smooth_s=1.5, fps=su.FPS, floor_z=None):
        pos = positions.detach().cpu().numpy() if torch.is_tensor(positions) else np.asarray(positions)
        fwd = root_rot[:, :, 2].detach().cpu().numpy() if torch.is_tensor(root_rot) else root_rot[:, :, 2]
        self.W, self.H = width, height
        self.f = 0.5 * height / math.tan(math.radians(fov_deg) / 2)
        self.floor_z = float(floor_z if floor_z is not None else np.percentile(pos[:, list(su.FOOT_JOINTS), 2].min(-1), 5))
        az = math.atan2(fwd[:, 1].mean(), fwd[:, 0].mean()) + math.radians(azimuth_offset_deg)
        self.view_dir = np.array([math.cos(az), math.sin(az), 0.0])  # camera sits in front of the dancer
        root = pos[:, 0].copy()
        if follow:
            k = max(1, int(smooth_s * fps))
            ker = np.ones(k) / k
            for d in range(2):
                root[:, d] = np.convolve(np.pad(root[:, d], (k // 2, k - 1 - k // 2), mode="edge"), ker, mode="valid")
        else:
            root[:, :2] = root[:, :2].mean(0)
        self.target = np.stack([root[:, 0], root[:, 1], np.full(len(root), self.floor_z + 0.95)], -1)
        el = math.radians(elevation_deg)
        self.offset = distance * np.array([math.cos(el) * self.view_dir[0], math.cos(el) * self.view_dir[1], math.sin(el)])

    def pose(self, t):
        eye = self.target[t] + self.offset
        z = self.target[t] - eye
        z /= np.linalg.norm(z)
        x = np.cross(z, np.array([0.0, 0.0, 1.0]))
        x /= np.linalg.norm(x)
        y = np.cross(x, z)  # up in image
        return eye, np.stack([x, y, z])  # rows: right, up, forward

    def project(self, pts, t):
        eye, R = self.pose(t)
        c = (np.asarray(pts) - eye) @ R.T
        zc = np.maximum(c[..., 2], 1e-3)
        u = self.W / 2 + self.f * c[..., 0] / zc
        v = self.H / 2 - self.f * c[..., 1] / zc
        return np.stack([u, v], -1), c[..., 2]


def find_font():
    for p in FONT_CANDIDATES:
        if os.path.exists(p):
            return p
    return None


def make_overlay_fn(grid=None, lines=None, regions=None, fps=su.FPS, start_frame=0):
    eights = grid["eights"] if grid else []
    regs = regions or []

    def info(frame):
        t = (frame + start_frame) / fps
        out = [f"{t:6.2f}s"]
        for e in eights:
            if e["start"] <= t < e["end"]:
                cnt = "-"
                for name, ct in sorted(e["counts"].items(), key=lambda kv: kv[1]):
                    if ct <= t + 1e-6:
                        cnt = name
                out[0] += f"   8-count E{e['index']}  [{cnt}]"
                break
        lyr = ""
        for ln in lines or []:
            if ln.get("start") is not None and ln["start"] - 0.2 <= t <= ln["end"] + 0.2:
                lyr = ln["text"]
        if lyr:
            out.append(lyr)
        for r in regs:
            if r["start"] <= frame + start_frame < r["end"]:
                hit = " *HIT*" if abs(frame + start_frame - r["hit"]) <= 2 else ""
                out.append(f"gesture #{r['id']} ({r['scope']}){hit}: {r.get('label', '')}")
        return out

    return info


def _draw_text(img, texts, font_path, size=20):
    from PIL import Image, ImageDraw, ImageFont
    if not texts:
        return img
    im = Image.fromarray(img)
    dr = ImageDraw.Draw(im)
    try:
        font = ImageFont.truetype(font_path, size) if font_path else ImageFont.load_default()
    except Exception:
        font = ImageFont.load_default()
    y = 8
    for s in texts:
        dr.text((10, y), s, fill=(255, 255, 255), font=font, stroke_width=2, stroke_fill=(0, 0, 0))
        y += size + 6
    return np.asarray(im)


class FFmpegWriter:
    def __init__(self, path, width, height, fps, audio=None, audio_start=0.0):
        cmd = ["ffmpeg", "-y", "-loglevel", "error", "-f", "rawvideo", "-pix_fmt", "rgb24", "-s", f"{width}x{height}",
               "-r", str(fps), "-i", "-"]
        if audio:
            cmd += ["-ss", f"{audio_start:.3f}", "-i", audio, "-map", "0:v", "-map", "1:a", "-c:a", "aac", "-b:a", "192k",
                    "-shortest"]
        cmd += ["-c:v", "libx264", "-pix_fmt", "yuv420p", "-crf", "20", "-preset", "veryfast", path]
        self.proc = subprocess.Popen(cmd, stdin=subprocess.PIPE)

    def write(self, frame):
        self.proc.stdin.write(np.ascontiguousarray(frame, dtype=np.uint8).tobytes())

    def close(self):
        self.proc.stdin.close()
        if self.proc.wait() != 0:
            raise RuntimeError("ffmpeg failed")


# ---------------------------------------------------------------------------- skeleton video
BONE_COLORS = {"left": (70, 140, 255), "right": (255, 90, 90), "center": (230, 230, 230)}


def _bone_color(j):
    if j in su.LEFT_JOINTS:
        return BONE_COLORS["left"]
    if j in su.RIGHT_JOINTS:
        return BONE_COLORS["right"]
    return BONE_COLORS["center"]


def draw_skeleton_frame(cam, t, positions_list, labels=None, floor_extent=3.0):
    import cv2
    W, H = cam.W, cam.H
    img = np.zeros((H, W, 3), dtype=np.uint8)
    img[:] = np.linspace(40, 15, H, dtype=np.uint8)[:, None, None]
    c = cam.target[t]
    g = np.arange(-floor_extent, floor_extent + 1e-6, 0.5)
    for k in g:
        for a, b in (((c[0] + k, c[1] - floor_extent), (c[0] + k, c[1] + floor_extent)),
                     ((c[0] - floor_extent, c[1] + k), (c[0] + floor_extent, c[1] + k))):
            pts = np.array([[a[0], a[1], cam.floor_z], [b[0], b[1], cam.floor_z]])
            uv, z = cam.project(pts, t)
            if (z > 0.05).all():
                cv2.line(img, tuple(np.round(uv[0]).astype(int)), tuple(np.round(uv[1]).astype(int)), (70, 70, 70), 1,
                         cv2.LINE_AA)
    for pi, P in enumerate(positions_list):
        uv, z = cam.project(P[t], t)
        shift = np.zeros(2)
        order = sorted(range(1, su.N_JOINTS), key=lambda j: -z[j])
        for j in order:
            p = su.SMPL_PARENTS[j]
            col = _bone_color(j)
            if pi > 0:
                col = tuple(int(0.55 * x) for x in col)
            cv2.line(img, tuple(np.round(uv[p] + shift).astype(int)), tuple(np.round(uv[j] + shift).astype(int)), col,
                     4 if pi == 0 else 2, cv2.LINE_AA)
        for j in range(su.N_JOINTS):
            cv2.circle(img, tuple(np.round(uv[j]).astype(int)), 4 if j != 15 else 9, (250, 250, 250) if pi == 0 else (140, 140, 140),
                       -1 if j != 15 else 2, cv2.LINE_AA)
    return img


def render_skeleton_video(out_path, positions, root_rot, audio=None, start_frame=0, n_frames=None, overlay=None,
                          ghost_positions=None, width=640, height=640, fps=su.FPS, font_path=None, progress=None):
    """positions (T,24,3) z-up; ghost_positions (optional, e.g. the base dance) is drawn dimmed behind."""
    pos = positions.detach().cpu().numpy() if torch.is_tensor(positions) else np.asarray(positions)
    T = pos.shape[0]
    n = T - start_frame if n_frames is None else min(n_frames, T - start_frame)
    sl = slice(start_frame, start_frame + n)
    plist = [pos[sl]]
    if ghost_positions is not None:
        gp = ghost_positions.detach().cpu().numpy() if torch.is_tensor(ghost_positions) else np.asarray(ghost_positions)
        plist.append(gp[sl])
    cam = Camera(pos[sl], root_rot[sl], width, height, fps=fps)
    font_path = font_path or find_font()
    wr = FFmpegWriter(out_path, width, height, fps, audio, start_frame / fps)
    try:
        for t in range(n):
            img = draw_skeleton_frame(cam, t, plist)
            if overlay is not None:
                img = _draw_text(img, overlay(t), font_path)
            wr.write(img)
            if progress and t % 300 == 0:
                progress(t, n)
    finally:
        wr.close()
    return out_path


# ---------------------------------------------------------------------------- mannequin (GPU ray marching)
# (joint a, joint b, radius a, radius b, material) ; material 0 = body, 1 = joint accent, 2 = head
MANNEQUIN_CONES = [
    (0, 3, 0.115, 0.105, 0), (1, 2, 0.095, 0.095, 0), (3, 6, 0.105, 0.11, 0), (6, 9, 0.115, 0.125, 0),
    (13, 14, 0.075, 0.075, 0), (9, 12, 0.07, 0.05, 0), (12, 15, 0.048, 0.045, 0),
    (9, 16, 0.075, 0.06, 0), (9, 17, 0.075, 0.06, 0),
    (16, 18, 0.052, 0.041, 0), (17, 19, 0.052, 0.041, 0), (18, 20, 0.04, 0.031, 0), (19, 21, 0.04, 0.031, 0),
    (20, 22, 0.03, 0.034, 0), (21, 23, 0.03, 0.034, 0),
    (1, 4, 0.085, 0.058, 0), (2, 5, 0.085, 0.058, 0), (4, 7, 0.054, 0.04, 0), (5, 8, 0.054, 0.04, 0),
    (7, 10, 0.042, 0.03, 0), (8, 11, 0.042, 0.03, 0),
]
MANNEQUIN_SPHERES = [(16, 0.058, 1), (17, 0.058, 1), (18, 0.046, 1), (19, 0.046, 1), (20, 0.034, 1), (21, 0.034, 1),
                     (4, 0.062, 1), (5, 0.062, 1), (7, 0.045, 1), (8, 0.045, 1)]
MATERIAL_COLORS = torch.tensor([[0.86, 0.72, 0.56], [0.55, 0.42, 0.32], [0.88, 0.75, 0.6]])


def _round_cone(p, a, b, r1, r2):
    """Signed distance of points p (N,1,3) to round cones (1,M,...) -- Inigo Quilez's sdRoundCone."""
    ba = b - a
    l2 = (ba * ba).sum(-1).clamp_min(1e-10)
    rr = r1 - r2
    a2 = l2 - rr * rr
    il2 = 1.0 / l2
    pa = p - a
    y = (pa * ba).sum(-1)
    z = y - l2
    q = pa * l2.unsqueeze(-1) - ba * y.unsqueeze(-1)
    x2 = (q * q).sum(-1)
    y2 = y * y * l2
    z2 = z * z * l2
    k = torch.sign(rr) * rr * rr * x2
    d1 = torch.sqrt(x2 + z2) * il2 - r2
    d2 = torch.sqrt(x2 + y2) * il2 - r1
    d3 = (torch.sqrt((x2 * a2 * il2).clamp_min(0)) + y * rr) * il2 - r1
    return torch.where(torch.sign(z) * a2 * z2 > k, d1, torch.where(torch.sign(y) * a2 * y2 < k, d2, d3))


class MannequinRenderer:
    def __init__(self, width=640, height=640, device="cuda", steps=64, shadow_steps=28, smooth_k=0.012):
        self.W, self.H, self.dev = width, height, torch.device(device)
        self.steps, self.shadow_steps, self.k = steps, shadow_steps, smooth_k
        self.light = torch.tensor([0.45, -0.35, 0.82], device=self.dev)
        self.light = self.light / self.light.norm()

    def _prims(self, P, G):
        """P: (24,3) positions, G: (24,3,3) global rotations (z-up). Returns cone/sphere tensors."""
        dev = self.dev
        ia = torch.tensor([c[0] for c in MANNEQUIN_CONES], device=dev)
        ib = torch.tensor([c[1] for c in MANNEQUIN_CONES], device=dev)
        A, B = P[ia], P[ib]
        r1 = torch.tensor([c[2] for c in MANNEQUIN_CONES], device=dev)
        r2 = torch.tensor([c[3] for c in MANNEQUIN_CONES], device=dev)
        cm = torch.tensor([c[4] for c in MANNEQUIN_CONES], device=dev)
        sc = [P[s[0]] for s in MANNEQUIN_SPHERES]
        sr = [s[1] for s in MANNEQUIN_SPHERES]
        sm = [s[2] for s in MANNEQUIN_SPHERES]
        head = P[15] + G[15] @ torch.tensor([0.0, 0.085, 0.02], device=dev)  # SMPL local y = up
        nose = head + G[15] @ torch.tensor([0.0, -0.005, 0.1], device=dev)   # local +z = face direction
        sc += [head, nose]
        sr += [0.105, 0.028]
        sm += [2, 1]
        return (A, B, r1, r2, cm, torch.stack(sc), torch.tensor(sr, device=dev), torch.tensor(sm, device=dev))

    def _sdf(self, p, prims, need_mat=False):
        A, B, r1, r2, cm, S, sr, sm = prims
        dc = _round_cone(p[:, None, :], A[None], B[None], r1[None], r2[None])
        ds = (p[:, None, :] - S[None]).norm(dim=-1) - sr[None]
        d = torch.cat([dc, ds], 1)
        k = self.k
        smin = -k * torch.logsumexp(-d / k, dim=1)
        if need_mat:
            mats = torch.cat([cm, sm])[d.argmin(1)]
            return smin, mats
        return smin

    @torch.no_grad()
    def render(self, P, G, eye, R, f, floor_z):
        """P (24,3), G (24,3,3) torch on device; eye (3,), R (3,3) rows = right, up, forward."""
        dev, H, W = self.dev, self.H, self.W
        eye = torch.as_tensor(eye, dtype=torch.float32, device=dev)
        R = torch.as_tensor(R, dtype=torch.float32, device=dev)
        vs, us = torch.meshgrid(torch.arange(H, device=dev, dtype=torch.float32),
                                torch.arange(W, device=dev, dtype=torch.float32), indexing="ij")
        dcam = torch.stack([(us + 0.5 - W / 2) / f, -(vs + 0.5 - H / 2) / f, torch.ones_like(us)], -1)
        d = torch.nn.functional.normalize(dcam @ R, dim=-1).reshape(-1, 3)
        N = d.shape[0]
        img = torch.zeros(N, 3, device=dev)
        sky = torch.linspace(0.30, 0.12, H, device=dev).repeat_interleave(W)
        img[:] = torch.stack([sky * 0.9, sky * 0.95, sky * 1.15], -1)
        prims = self._prims(P, G)
        center = P.mean(0)
        radius = float((P - center).norm(dim=-1).max()) + 0.2
        # floor
        t_floor = torch.where(d[:, 2] < -1e-4, (floor_z - eye[2]) / d[:, 2], torch.full_like(d[:, 2], float("inf")))
        # body: sphere tracing inside the bounding sphere
        oc = eye - center
        bq = (d * oc).sum(-1)
        cq = (oc * oc).sum() - radius * radius
        disc = bq * bq - cq
        active = torch.nonzero(disc > 0).squeeze(-1)
        t_body = torch.full((N,), float("inf"), device=dev)
        normal = torch.zeros(N, 3, device=dev)
        mat = torch.zeros(N, dtype=torch.long, device=dev)
        if active.numel():
            sq = torch.sqrt(disc[active])
            t = (-bq[active] - sq).clamp_min(0.0)
            tmax = torch.minimum(-bq[active] + sq, t_floor[active])
            da = d[active]
            hit = torch.zeros_like(t, dtype=torch.bool)
            alive = torch.ones_like(hit)
            for _ in range(self.steps):
                idx = torch.nonzero(alive).squeeze(-1)
                if idx.numel() == 0:
                    break
                p = eye + da[idx] * t[idx, None]
                dist = self._sdf(p, prims)
                t[idx] += dist
                h = dist < 1e-3
                hit[idx[h]] = True
                alive[idx[h | (t[idx] > tmax[idx])]] = False
            hidx = torch.nonzero(hit).squeeze(-1)
            if hidx.numel():
                gi = active[hidx]
                p = eye + d[gi] * t[hidx, None]
                e = 1e-3
                ks = torch.tensor([[1, -1, -1], [-1, -1, 1], [-1, 1, -1], [1, 1, 1]], dtype=torch.float32, device=dev)
                n = sum(k * self._sdf(p + e * k, prims)[:, None] for k in ks)
                normal[gi] = torch.nn.functional.normalize(n, dim=-1)
                _, m = self._sdf(p, prims, need_mat=True)
                mat[gi] = m
                t_body[gi] = t[hidx]
        body = torch.isfinite(t_body) & (t_body <= t_floor)
        floor = torch.isfinite(t_floor) & ~body
        # floor shading with checkerboard + soft shadow
        fidx = torch.nonzero(floor).squeeze(-1)
        if fidx.numel():
            fp = eye + d[fidx] * t_floor[fidx, None]
            chk = ((torch.floor(fp[:, 0] / 0.5) + torch.floor(fp[:, 1] / 0.5)) % 2 == 0).float()
            base = 0.42 + 0.12 * chk
            shadow = torch.ones_like(base)
            near = torch.nonzero((fp[:, :2] - center[:2]).norm(dim=-1) < radius + 1.8).squeeze(-1)
            if near.numel():
                sp = fp[near] + self.light * 0.02
                tt = torch.full((near.numel(),), 0.02, device=dev)
                res = torch.ones_like(tt)
                for _ in range(self.shadow_steps):
                    dist = self._sdf(sp + self.light * tt[:, None], prims)
                    res = torch.minimum(res, (10.0 * dist / tt).clamp(0, 1))
                    tt = tt + dist.clamp(0.01, 0.25)
                shadow[near] = 0.35 + 0.65 * res
            dist_cam = (fp[:, :2] - center[:2]).norm(dim=-1)
            fade = torch.exp(-(dist_cam / 6.0) ** 2)
            col = (base * shadow)[:, None] * torch.tensor([1.0, 1.0, 1.03], device=dev)
            img[fidx] = col * fade[:, None] + img[fidx] * (1 - fade[:, None])
        bidx = torch.nonzero(body).squeeze(-1)
        if bidx.numel():
            n = normal[bidx]
            alb = MATERIAL_COLORS.to(dev)[mat[bidx]]
            diff = (n @ self.light).clamp_min(0)
            hemi = 0.5 + 0.5 * n[:, 2]
            v = -d[bidx]
            hvec = torch.nn.functional.normalize(v + self.light, dim=-1)
            spec = ((n * hvec).sum(-1).clamp_min(0) ** 32) * 0.25
            rim = (1 - (n * v).sum(-1).clamp(0, 1)) ** 3 * 0.25
            col = alb * (0.25 * hemi + 0.85 * diff)[:, None] + (spec + rim)[:, None]
            img[bidx] = col
        img = img.clamp(0, 1) ** (1 / 1.1)
        return (img.reshape(H, W, 3) * 255).byte().cpu().numpy()


def render_mannequin_video(out_path, motion, audio=None, start_frame=0, n_frames=None, overlay=None, width=640,
                           height=640, fps=su.FPS, device="cuda", font_path=None, progress=None, steps=64):
    rot = motion["rotmat"].float()
    pos = motion["root_pos"].float()
    T = rot.shape[0]
    n = T - start_frame if n_frames is None else min(n_frames, T - start_frame)
    sl = slice(start_frame, start_frame + n)
    P_all, G_all = su.fk(rot[sl], pos[sl], return_global=True)
    cam = Camera(P_all, G_all[:, 0], width, height, fps=fps)
    floor_z = cam.floor_z - 0.035  # feet are joint centres; lower the floor to the sole
    rend = MannequinRenderer(width, height, device, steps=steps)
    font_path = font_path or find_font()
    wr = FFmpegWriter(out_path, width, height, fps, audio, start_frame / fps)
    try:
        for t in range(n):
            eye, R = cam.pose(t)
            img = rend.render(P_all[t].to(rend.dev), G_all[t].to(rend.dev), eye, R, cam.f, floor_z)
            if overlay is not None:
                img = _draw_text(img, overlay(t), font_path)
            wr.write(img)
            if progress and t % 150 == 0:
                progress(t, n)
    finally:
        wr.close()
    return out_path

In [ ]:
%%writefile /content/m2d_pkg/m2d/viz.py
"""Plots and diagnostics shown in the notebook."""
import numpy as np
import torch

from . import smpl_utils as su


def _plt():
    import matplotlib
    import matplotlib.pyplot as plt
    matplotlib.rcParams["axes.unicode_minus"] = False
    return plt


# ---------------------------------------------------------------------------- audio / alignment
def rms_envelope(path, sr=16000, hop=0.02):
    import librosa
    y, _ = librosa.load(path, sr=sr, mono=True)
    h = int(hop * sr)
    rms = librosa.feature.rms(y=y, frame_length=2 * h, hop_length=h)[0]
    return np.arange(len(rms)) * hop, rms


def plot_alignment(vocals_path, lines, words=None, figsize=(16, 3.2)):
    plt = _plt()
    t, rms = rms_envelope(vocals_path)
    fig, ax = plt.subplots(figsize=figsize)
    ax.plot(t, rms / (rms.max() + 1e-9), color="0.6", lw=0.6, label="vocal energy")
    for ln in lines:
        if ln.get("start") is None:
            continue
        bad = ln.get("score", 0) < -2.5
        ax.axvspan(ln["start"], ln["end"], color="tab:red" if bad else "tab:blue", alpha=0.25)
        ax.text(ln["start"], 1.02, str(ln["index"]), fontsize=8, color="tab:red" if bad else "tab:blue")
    if words:
        for w in words:
            if w.get("start") is not None:
                ax.axvline(w["start"], color="tab:green", lw=0.4, alpha=0.6)
    ax.set_xlim(0, t[-1])
    ax.set_ylim(0, 1.12)
    ax.set_xlabel("seconds")
    ax.set_title("lyric lines on the vocal track (number = line index, red = low confidence; green = word starts)")
    plt.tight_layout()
    plt.show()


def plot_beats(song_path, beats, downbeats, grid=None, start=0.0, duration=30.0, figsize=(16, 3)):
    import librosa
    plt = _plt()
    y, sr = librosa.load(song_path, sr=11025, mono=True, offset=start, duration=duration)
    t = start + np.arange(len(y)) / sr
    fig, ax = plt.subplots(figsize=figsize)
    ax.plot(t, y, color="0.75", lw=0.3)
    for b in beats:
        if start <= b <= start + duration:
            ax.axvline(b, color="tab:blue", lw=0.8, alpha=0.6)
    for d in downbeats:
        if start <= d <= start + duration:
            ax.axvline(d, color="tab:red", lw=1.6)
    if grid:
        for e in grid["eights"]:
            if start <= e["start"] <= start + duration:
                ax.text(e["start"], ax.get_ylim()[1] * 0.9, f"E{e['index']}", color="tab:purple", fontsize=9)
    ax.set_xlim(start, start + duration)
    ax.set_title("beats (blue), downbeats (red), start of each eight (E#)")
    plt.tight_layout()
    plt.show()


def click_track(song_path, grid, start=0.0, duration=30.0, sr=22050, gain=0.6):
    """Song + clicks on every count (count 1 of each eight high and loud, '&' soft)."""
    import librosa
    y, _ = librosa.load(song_path, sr=sr, mono=True, offset=start, duration=duration)
    out = 0.6 * y.copy()

    def click(t, freq, amp):
        i = int((t - start) * sr)
        n = int(0.04 * sr)
        if 0 <= i < len(out) - n:
            tt = np.arange(n) / sr
            out[i:i + n] += amp * np.sin(2 * np.pi * freq * tt) * np.exp(-tt * 60)

    for e in grid["eights"]:
        for name, ct in e["counts"].items():
            if name == "1":
                click(ct, 1760, gain)
            elif name.endswith("&"):
                click(ct, 660, gain * 0.25)
            else:
                click(ct, 1100, gain * 0.6)
    return np.clip(out, -1, 1), sr


# ---------------------------------------------------------------------------- dance diagnostics
def plot_speed_windows(stats, labels, window_s=5.0, figsize=(14, 3)):
    plt = _plt()
    fig, ax = plt.subplots(figsize=figsize)
    for st, lb in zip(stats, labels):
        x = np.arange(len(st["speed_per_window"])) * window_s
        ax.plot(x, st["speed_per_window"], marker="o", ms=3, label=f"{lb} (mean {st['mean_speed']:.2f} m/s)")
    ax.set_xlabel("seconds")
    ax.set_ylabel("mean joint speed [m/s]")
    ax.set_title("motion energy per 5 s window (a decay over time suggests the sequence is too long for one pass)")
    ax.legend()
    plt.tight_layout()
    plt.show()


def plot_gesture_debug(r, fps=su.FPS):
    """Candidate speed curves + IK error per joint + target/IK skeleton overlay at the key pose."""
    plt = _plt()
    conv = r["debug"]["conv"]
    fig = plt.figure(figsize=(17, 4.2))
    ax = fig.add_subplot(1, 3, 1)
    for i, sp in enumerate(r["debug"]["speeds"]):
        ax.plot(np.arange(len(sp)) / fps, sp, label=f"candidate {i}", lw=2 if i == r["info"]["chosen"] else 0.8)
    k = r["info"]["key_frame_in_candidate"]
    ax.axvline(k / fps, color="k", ls="--", label="key pose")
    ax.axvspan((k - r["info"]["pre"]) / fps, (k + r["info"]["post"]) / fps, color="orange", alpha=0.2, label="cut")
    ax.set_xlabel("seconds")
    ax.set_ylabel("end-effector speed")
    ax.set_title(f"gesture #{r['id']}: key pose detection")
    ax.legend(fontsize=7)
    ax = fig.add_subplot(1, 3, 2)
    e0 = conv["errors_analytic"]["pos_err_m"].mean(0) * 100
    e1 = conv["errors"]["pos_err_m"].mean(0) * 100
    x = np.arange(len(e1))
    ax.bar(x - 0.2, e0, 0.4, label="analytic IK")
    ax.bar(x + 0.2, e1, 0.4, label="refined IK")
    ax.set_xticks(x)
    ax.set_xticklabels([su.JOINT_NAMES[j] for j in range(len(e1))], rotation=90, fontsize=7)
    ax.set_ylabel("mean position error [cm]")
    ax.set_title("conversion error per joint (vs. retargeted target)")
    ax.legend(fontsize=8)
    ax = fig.add_subplot(1, 3, 3, projection="3d")
    tgt = conv["errors"]["target_positions"][k]
    fkp = conv["errors"]["fk_positions"][k]
    for c in range(1, 22):
        p = su.SMPL_PARENTS[c]
        ax.plot(*[[tgt[p, i], tgt[c, i]] for i in (0, 2, 1)], color="tab:green", lw=3, alpha=0.6)
        ax.plot(*[[fkp[p, i], fkp[c, i]] for i in (0, 2, 1)], color="tab:red", lw=1.2)
    ax.set_title("key pose: target (green) vs converted SMPL (red)")
    lim = np.abs(tgt - tgt[0]).max() + 0.1
    ax.set_xlim(tgt[0, 0] - lim, tgt[0, 0] + lim)
    ax.set_ylim(tgt[0, 2] - lim, tgt[0, 2] + lim)
    ax.set_zlim(tgt[0, 1] - lim, tgt[0, 1] + lim)
    plt.tight_layout()
    plt.show()


def plot_mask_timeline(mask, regions, windows, fps=su.FPS, figsize=(16, 3.4)):
    """Rows: contacts / root / upper-body joints / lower-body joints / other joints."""
    plt = _plt()
    groups = [("contacts", list(range(0, 4))), ("root pos+rot", list(range(4, 7)) + su.joint_channels([0])),
              ("upper body", su.joint_channels(su.UPPER_JOINTS_DEFAULT)),
              ("lower body", su.joint_channels([1, 2, 4, 5, 7, 8, 10, 11])),
              ("head/wrists/hands", su.joint_channels([15, 20, 21, 22, 23]))]
    T = mask.shape[0]
    img = np.stack([mask[:, ch].float().mean(1).numpy() for _, ch in groups])
    gest = np.zeros_like(img)
    for r in regions:
        for gi, (_, ch) in enumerate(groups):
            if set(ch) & set(r["channels"]):
                gest[gi, r["start"]:r["end"]] = 1
    rgb = np.ones((len(groups), T, 3))
    rgb[img < 0.5] = (0.95, 0.55, 0.2)          # re-generated
    rgb[(img >= 0.5) & (gest > 0)] = (0.2, 0.6, 0.25)  # fixed to the gesture
    rgb[(img >= 0.5) & (gest == 0)] = (0.8, 0.85, 0.95)  # fixed to the base dance
    fig, ax = plt.subplots(figsize=figsize)
    ax.imshow(rgb, aspect="auto", extent=[0, T / fps, len(groups) - 0.5, -0.5], interpolation="nearest")
    for s, e in windows:
        ax.axvspan(s / fps, e / fps, ymin=0, ymax=1, fill=False, ec="k", lw=1.2)
    for r in regions:
        ax.axvline(r["hit"] / fps, color="k", lw=1)
        ax.text(r["hit"] / fps, -0.6, f"#{r['id']}", fontsize=8, ha="center")
    ax.set_yticks(range(len(groups)))
    ax.set_yticklabels([g for g, _ in groups])
    ax.set_xlabel("seconds")
    ax.set_title("mask: green = gesture (fixed), blue = base dance (fixed), orange = re-generated; boxes = windows")
    plt.tight_layout()
    plt.show()


def rotation_angle_deg(Ra, Rb):
    rel = Ra.transpose(-1, -2) @ Rb
    cos = ((rel.diagonal(dim1=-2, dim2=-1).sum(-1) - 1) / 2).clamp(-1, 1)
    return torch.rad2deg(torch.acos(cos))


def gesture_adherence(final_rot, target_rot, regions, joints_by_scope):
    """Mean angular deviation (deg) between the final dance and the pasted gesture on its joints."""
    rows = []
    for r in regions:
        js = joints_by_scope[r["scope"]]
        a = rotation_angle_deg(final_rot[r["start"]:r["end"]][:, js], target_rot[r["start"]:r["end"]][:, js])
        h = min(max(r["hit"], r["start"]), r["end"] - 1)
        ah = rotation_angle_deg(final_rot[h, js], target_rot[h, js])
        rows.append({"id": r["id"], "scope": r["scope"], "mean_dev_deg": float(a.mean()),
                     "hit_dev_deg": float(ah.mean()), "max_dev_deg": float(a.max())})
    return rows


def boundary_jumps(positions, windows, fps=su.FPS):
    """Joint speed right at each window border vs. the median speed (should be similar)."""
    v = (positions[1:] - positions[:-1]).norm(dim=-1).mean(-1) * fps
    med = float(v.median())
    out = []
    for s, e in windows:
        for b in (s, e):
            if 1 <= b < len(v):
                out.append({"frame": int(b), "speed": float(v[b - 1]), "median_speed": med})
    return out

In [ ]:
# @title 1-7. 共通設定と補助関数（ランタイム再起動後は 1-1 とこのセルを実行し直す）
import os, sys, json, gc, time, math, shutil, subprocess, importlib, base64
ROOT, WORK, MODELS, REPOS, PKG = "/content", "/content/work", "/content/models", "/content/repos", "/content/m2d_pkg"
MGPT_SITE, FD_REPO, MGPT_REPO = f"{ROOT}/mgpt_site", f"{REPOS}/FlowerDance", f"{REPOS}/MotionGPT3"
for k, v in {"TORCH_HOME": f"{MODELS}/torch", "HF_HOME": f"{MODELS}/huggingface", "NLTK_DATA": f"{MODELS}/nltk",
             "TOKENIZERS_PARALLELISM": "false"}.items():
    os.environ[k] = v
if PKG not in sys.path:
    sys.path.insert(0, PKG)
# サブプロセスからも m2d と PyTorch3D の代替が見えるようにする
import site
with open(os.path.join(site.getsitepackages()[0], "m2d_pkg.pth"), "w") as f:
    f.write(PKG + "\n")

import numpy as np
import pandas as pd
import librosa
import torch
from IPython.display import Audio, HTML, display

import m2d
from m2d import (smpl_utils as su, mamba_fallback as mf, flowerdance_runner as fr, audio_analysis as aa,
                 lyrics_align as la, choreo_llm as cl, motion_ik as ik, gesture_pipeline as gp,
                 gesture_embed as ge, export_render as er, viz)
for _m in (su, mf, fr, aa, la, cl, ik, gp, ge, er, viz):
    importlib.reload(_m)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
ENV_STATE_PATH = f"{WORK}/env_state.json"

def env_state():
    return json.load(open(ENV_STATE_PATH)) if os.path.exists(ENV_STATE_PATH) else {}

def update_env_state(**kw):
    st = env_state(); st.update(kw); json.dump(st, open(ENV_STATE_PATH, "w"), indent=1)

def save_json(name, obj):
    with open(f"{WORK}/{name}", "w", encoding="utf-8") as f:
        json.dump(obj, f, ensure_ascii=False, indent=1, default=float)

def load_json(name):
    with open(f"{WORK}/{name}", encoding="utf-8") as f:
        return json.load(f)

def save_pt(name, obj):
    torch.save(obj, f"{WORK}/{name}")

def load_pt(name):
    return torch.load(f"{WORK}/{name}", map_location="cpu", weights_only=False)

def free_gpu():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

def show_video(path, width=480, max_mb=60):
    size = os.path.getsize(path) / 2**20
    if size > max_mb:
        print(f"{path} ({size:.0f} MB) は大きいのでここでは表示しません。ダウンロードして確認してください。")
        return
    b64 = base64.b64encode(open(path, "rb").read()).decode()
    display(HTML(f'<video width="{width}" controls><source src="data:video/mp4;base64,{b64}" type="video/mp4"></video>'))

def mgpt_env():
    env = dict(os.environ)
    env["PYTHONPATH"] = os.pathsep.join([MGPT_SITE] + [p for p in env.get("PYTHONPATH", "").split(os.pathsep) if p])
    return env

def run_streaming(cmd, env=None, cwd=None):
    proc = subprocess.Popen(cmd, env=env, cwd=cwd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    for line in proc.stdout:
        print(line, end="")
    if proc.wait() != 0:
        raise RuntimeError(f"command failed ({proc.returncode}): {' '.join(cmd[:4])} ...")

MAMBA_BACKEND = env_state().get("MAMBA_BACKEND", "cuda")
print("torch", torch.__version__, "| device", DEVICE, "| MAMBA_BACKEND", MAMBA_BACKEND)
print("m2d", m2d.__version__, "| modules OK")

## 2. モデルのダウンロード（`/content/models` に保存）

| モデル | 取得元 | 目安 |
|---|---|---|
| FlowerDance | Hugging Face `xlt99/FlowerDance`（`train-3700.pt`） | 1GB 未満 |
| Demucs htdemucs | Hugging Face（demucs 4.1 が自動取得） | 約 0.3GB |
| wav2vec2（日本語） | `jonatasgrosman/wav2vec2-large-xlsr-53-japanese`（WhisperX の既定） | 約 1.2GB |
| beat_this `final0` | 作者の公開ストレージ。落ちているときは同一ファイルのミラー（GitHub Release / Hugging Face）から取得し、SHA-256 で元ファイルと同一か確認 | 0.1GB 未満 |
| MotionGPT3 | Hugging Face `OpenMotionLab/motiongpt3`（失敗時は作者の Google Drive 公開リンク） | 約 1〜2GB |
| GPT-2（MotionGPT3 の言語部分） | `openai-community/gpt2` | 約 0.5GB |

合計で約 3〜4GB です。SMPL のモデルファイル（要登録）は不要です（FlowerDance は骨格オフセットをコードに持っています）。

In [ ]:
# @title 2-1. FlowerDance / Demucs / beat_this / wav2vec2 / NLTK
from huggingface_hub import hf_hub_download, snapshot_download, list_repo_files

HF_TOKEN = None
try:
    from google.colab import userdata
    HF_TOKEN = userdata.get("HF_TOKEN")
except Exception:
    pass

FD_CKPT = hf_hub_download("xlt99/FlowerDance", "train-3700.pt", local_dir=f"{MODELS}/flowerdance", token=HF_TOKEN)
print("FlowerDance:", FD_CKPT)

from demucs.pretrained import get_model as demucs_get_model
_ = demucs_get_model("htdemucs"); del _
print("Demucs htdemucs: OK")

# 作者の公開サーバー（cloud.cp.jku.at）は 503 を返すことがあるため、作者サーバー → ミラー（GitHub Release、
# Hugging Face）の順に取得し、SHA-256 が元ファイルと一致したものだけを使う。
# それでも取得できなければ他のモデルのダウンロードは続け、4-4 でもう一度試す。
try:
    print("beat_this final0: OK", aa.ensure_beat_this_checkpoint("final0"))
except RuntimeError as e:
    print("[warn]", e)

ALIGN_MODEL_JA = "jonatasgrosman/wav2vec2-large-xlsr-53-japanese"
snapshot_download(ALIGN_MODEL_JA, token=HF_TOKEN)
print("wav2vec2 (ja): OK")

import nltk
nltk.download("punkt_tab", download_dir=os.environ["NLTK_DATA"], quiet=True)
update_env_state(FD_CKPT=FD_CKPT)

In [ ]:
# @title 2-2. MotionGPT3 と GPT-2
MGPT_HF_REPO = "OpenMotionLab/motiongpt3"
MGPT_GDRIVE_ID = "1Wvx5PGJjVKPRvjcl8firChw1UVjUj36l"  # MotionGPT3 の prepare スクリプトが指す公開ファイル
MGPT_CKPT = ""
try:
    files = list_repo_files(MGPT_HF_REPO, token=HF_TOKEN)
    print("files in", MGPT_HF_REPO, ":", files)
    cands = [f for f in files if f.endswith((".ckpt", ".pt", ".pth"))]
    cands.sort(key=lambda f: (os.path.basename(f) != "motiongpt3.ckpt", "motiongpt3" not in f.lower(),
                              not f.endswith(".ckpt"), len(f)))
    if cands:
        MGPT_CKPT = hf_hub_download(MGPT_HF_REPO, cands[0], local_dir=f"{MODELS}/motiongpt3", token=HF_TOKEN)
except Exception as e:
    print("[warn] Hugging Face から取得できませんでした:", repr(e))
if not MGPT_CKPT:
    import gdown
    MGPT_CKPT = f"{MODELS}/motiongpt3/motiongpt3.ckpt"
    os.makedirs(os.path.dirname(MGPT_CKPT), exist_ok=True)
    gdown.download(id=MGPT_GDRIVE_ID, output=MGPT_CKPT, quiet=False)
assert os.path.getsize(MGPT_CKPT) > 10 * 2**20, f"MotionGPT3 のチェックポイントが不正です: {MGPT_CKPT}"
print("MotionGPT3:", MGPT_CKPT, f"({os.path.getsize(MGPT_CKPT) / 2**30:.2f} GB)")

GPT2_DIR = snapshot_download("openai-community/gpt2", local_dir=f"{MODELS}/gpt2", token=HF_TOKEN,
                             allow_patterns=["config.json", "generation_config.json", "model.safetensors",
                                             "tokenizer.json", "tokenizer_config.json", "vocab.json", "merges.txt"])
# MotionGPT3 の scripts/gen_mot_gpt.py と同じ変換（deps/mot-gpt2 を作る）
run_streaming([sys.executable, f"{PKG}/m2d/mgpt_runner.py", "prepare", "--repo", MGPT_REPO, "--gpt2", GPT2_DIR],
              env=mgpt_env())
update_env_state(MGPT_CKPT=MGPT_CKPT)
print(subprocess.run(["du", "-sh", MODELS], capture_output=True, text=True).stdout)

In [ ]:
# @title 2-3. インストールの確認（Mamba のカーネル、FlowerDance の重みの strict 読み込み）
FD_CKPT = env_state()["FD_CKPT"]
fr.patch_transformers_for_mamba()
if MAMBA_BACKEND == "cuda":
    try:
        from mamba_ssm import Mamba as CudaMamba
        torch.manual_seed(0)
        m_cuda = CudaMamba(d_model=512, d_state=16, d_conv=4, expand=2).to(DEVICE).eval()
        m_ref = mf.Mamba(512).to(DEVICE).eval()
        m_ref.load_state_dict(m_cuda.state_dict(), strict=True)
        x = torch.randn(1, 1200, 512, device=DEVICE)
        with torch.no_grad():
            a, b = m_cuda(x), m_ref(x)
        rel = ((a - b).abs().max() / b.abs().max()).item()
        print(f"Mamba CUDA kernel: OK (純 PyTorch 参照実装との相対誤差 {rel:.2e})")
        assert rel < 1e-2, "CUDA カーネルの結果が参照実装と一致しません"
        del m_cuda, m_ref, x, a, b
    except Exception as e:
        print("[warn] Mamba の CUDA カーネルが使えません:", repr(e))
        if not globals().get("ALLOW_TORCH_MAMBA_FALLBACK", True):
            raise
        MAMBA_BACKEND = "torch"
        print("→ 純 PyTorch 版の Mamba に切り替えます")
update_env_state(MAMBA_BACKEND=MAMBA_BACKEND)

fd = fr.FlowerDance(FD_REPO, FD_CKPT, DEVICE, MAMBA_BACKEND)  # 内部で load_state_dict(strict=True)
print(f"FlowerDance: strict 読み込み OK（{fd.n_params / 1e6:.1f}M パラメータ, Mamba={fd.mamba_backend}）")
t0 = time.time()
smoke = fd.sample(np.zeros((150, 35), np.float32), "Hiphop", n_steps=4, seed=0)
assert torch.isfinite(smoke).all()
print(f"FlowerDance: 試し生成 OK（5 秒分, {time.time() - t0:.1f}s）")
fd.free(); del fd, smoke; free_gpu()

## 3. 入力

- **音源**: アップロード / Colab 内のパス / URL のいずれか。`SONG_START_SEC`・`SONG_DURATION_SEC` で一部だけを使うこともできます（試運転に便利です）。
- **ジャンル**: FlowerDance が学習した 16 ジャンルから選びます。
- **既定値**: 音源は `https://enabify.net/uploads/daten.mp3`（先頭 0〜120 秒）、ジャンルは `Hiphop`、歌詞はこの曲のものが入っています。
- **歌詞**: 1 行 = 1 フレーズで貼り付けます。`[サビ]` `【Aメロ】` のような括弧だけの行はセクション名として扱います。
- **OpenRouter**: シークレット `OPENROUTER_API_KEY` を読み込み、モデル名が存在するか確認します。

In [ ]:
# @title 3-1. 音源とジャンル
AUDIO_SOURCE = "url"  # @param ["upload", "path", "url"]
AUDIO_PATH_OR_URL = "https://enabify.net/uploads/daten.mp3"  # @param {type:"string"}
SONG_START_SEC = 0.0  # @param {type:"number"}
SONG_DURATION_SEC = 120.0  # @param {type:"number"}
# SONG_DURATION_SEC = 0 で最後まで。区間を切り出したときは、区間外の歌詞の行は 4-2 で自動的に「未整列」になります。
GENRE = "Hiphop"  # @param ["Dai", "ShenYun", "Wei", "Korean", "Urban", "Hiphop", "Popping", "Miao", "HanTang", "Breaking", "Kun", "Locking", "Jazz", "Choreography", "Chinese", "DunHuang"]
SEED = 2026  # @param {type:"integer"}

os.makedirs(f"{WORK}/input", exist_ok=True)
if AUDIO_SOURCE == "upload":
    from google.colab import files
    up = files.upload()
    if not up:
        raise SystemExit("ファイルが選ばれていません")
    src = f"{WORK}/input/{next(iter(up))}"
    with open(src, "wb") as f:
        f.write(up[next(iter(up))])
elif AUDIO_SOURCE == "url":
    import requests
    from urllib.parse import urlparse
    name = os.path.basename(urlparse(AUDIO_PATH_OR_URL).path) or "download_audio"
    src = f"{WORK}/input/{name}"
    with requests.get(AUDIO_PATH_OR_URL, stream=True, timeout=120) as r:
        r.raise_for_status()
        with open(src, "wb") as f:
            for chunk in r.iter_content(1 << 20):
                f.write(chunk)
    print(f"downloaded {AUDIO_PATH_OR_URL} ({os.path.getsize(src) / 2**20:.1f} MB)")
else:
    src = AUDIO_PATH_OR_URL
    assert os.path.exists(src), f"ファイルがありません: {src}"

SONG = f"{WORK}/input/song.wav"  # 以降はこの 44.1kHz ステレオ WAV を使う
cmd = ["ffmpeg", "-y", "-loglevel", "error"]
if SONG_START_SEC > 0:
    cmd += ["-ss", str(SONG_START_SEC)]
cmd += ["-i", src]
if SONG_DURATION_SEC > 0:
    cmd += ["-t", str(SONG_DURATION_SEC)]
cmd += ["-ac", "2", "-ar", "44100", SONG]
subprocess.run(cmd, check=True)
import librosa
DURATION = librosa.get_duration(path=SONG)
FULL_DURATION = float(subprocess.run(["ffprobe", "-v", "error", "-show_entries", "format=duration", "-of",
                                      "default=nw=1:nk=1", src], capture_output=True, text=True).stdout.strip() or 0)
fr.genre_id(GENRE)  # ジャンル名の確認
save_json("inputs.json", {"song": SONG, "source": src, "start": SONG_START_SEC, "duration": DURATION,
                          "full_duration": FULL_DURATION, "genre": GENRE, "seed": SEED,
                          "cut_start": SONG_START_SEC > 0,
                          "cut_end": FULL_DURATION > 0 and SONG_START_SEC + DURATION < FULL_DURATION - 0.5})
print(f"song: {SONG}  ({DURATION:.1f} s of {FULL_DURATION:.1f} s), genre: {GENRE}")
display(Audio(SONG))

In [ ]:
# @title 3-2. 歌詞（下の LYRICS を書き換えるか、LYRICS_FILE に .txt のパスを入れる）
LYRICS_LANGUAGE = "ja"  # @param ["ja", "en"]
LYRICS_FILE = ""  # @param {type:"string"}

LYRICS = """
あの日林檎の木の下共犯で
踏み越えてみた境界線
すぐに black out あ、お怒りで
追い立てられてこんな次元へ
俺とした事が 俺如きですら
俺に限っていや、まさかね
君のせいにした 秘密を手にした
イチジクの葉が右左

あっという間、目が醒める
かと思えば空に落ちて行く
なんというか、やめられぬ
もうひと齧り

Again, fallin', falling, 螺旋状に
堕ちてゆく摩天楼に
今 fallin', falling, 二人ぼっち
気づかないカーテンコールにも
Yeah, yeah, yeah, yeah, yeah どこまでも
Yeah, yeah, yeah, yeah, yeah この身任せて
いつの間に傷が埋まってくyeah

そこじゃ何から何までごった煮で
鬼も仏もおんなじ目
全て shut out どうかしてる？
俺らハナから大真面目 ayy
ふりほどいて来たうしろ指ですら
むしろ追い風さまだ足んねぇ
蛇に睨まれた歴史の徒花
エデンにはまだ空室あり

あっという間染められる
かと思えば熱が醒めていく
ちょっと待ってその前に
もうひと雫

Again, fallin', falling, 螺旋状に
堕ちてゆく摩天楼に
今 fallin', falling, 二人ぼっち
気づかないカーテンコールにも
Yeah, yeah, yeah, yeah, yeah どこまでも
Yeah, yeah, yeah, yeah, yeah この身任せて
いつの間に傷が埋まってくyeah

Again, fallin', falling, 螺旋状に
堕ちてゆく摩天楼に
今 fallin', falling, 二人ぼっち
気づかないカーテンコールにも
Yeah, yeah, yeah, yeah, yeah どこまでも
Yeah, yeah, yeah, yeah, yeah この身任せて
いつの間に傷が埋まってく

この血が冷めないうちに飲み干して
眩暈するほど気取って
取り留めない出会いに色付けて
この目が醒めないうちに憑り込んで
狭い空ごと突き抜けて falling
蜜の味二人ハマってく yeah
"""

if LYRICS_FILE:
    LYRICS = open(LYRICS_FILE, encoding="utf-8").read()
parsed = la.parse_lyrics(LYRICS)
save_json("lyrics_input.json", {"text": LYRICS, "language": LYRICS_LANGUAGE})
display(pd.DataFrame(parsed))

In [ ]:
# @title 3-3. OpenRouter の API キーとモデル
OPENROUTER_MODEL = "anthropic/claude-sonnet-4.5"  # @param {type:"string"}
from google.colab import userdata
OPENROUTER_API_KEY = userdata.get("OPENROUTER_API_KEY")
models = cl.list_models(OPENROUTER_API_KEY)
if OPENROUTER_MODEL not in models:
    near = [m for m in models if any(k in m for k in ("claude", "gpt", "gemini"))][:30]
    raise SystemExit(f"OpenRouter にモデル {OPENROUTER_MODEL!r} がありません。候補: {near}")
print("OpenRouter: OK,", OPENROUTER_MODEL)

## 4. 工程 1〜4：音楽と歌詞の解析

### 4-1. 工程 1：Demucs（htdemucs）でボーカルを分離

In [ ]:
# @title 4-1. ボーカル分離
SONG = load_json("inputs.json")["song"]
VOCALS, ACCOMP = aa.separate_vocals(SONG, f"{WORK}/stems", model_name="htdemucs", device=DEVICE)
free_gpu()
print("vocals:", VOCALS)
y_v, sr_v = librosa.load(VOCALS, sr=22050, mono=True, duration=30)
display(Audio(y_v, rate=sr_v))

### 4-2. 工程 2：歌詞とボーカルの強制整列

1. 曲全体の wav2vec2 の出力（CTC の確率）を 20 秒ずつ計算し、歌詞全体を 1 回の Viterbi 整列で当てはめて**行のおおよその位置**を決めます。
2. 各行の区間（前後 0.4 秒の余裕つき）で **WhisperX の `align`** を実行し、文字・単語の時刻を精密化します。
3. 日本語は、整列モデルの語彙にない漢字を pykakasi でひらがなに直して整列し、元の文字に対応付け直します。単語の区切りは pykakasi の分割です。

歌唱は話し声より精度が落ちるため、結果を一覧と図で確認し、必要なら 4-3 で補正してください（`score` が低い行は赤で表示）。

In [ ]:
# @title 4-2. 整列の実行
lyr = load_json("lyrics_input.json"); inp = load_json("inputs.json")
VOCALS = f"{WORK}/stems/vocals.wav"
# 曲の一部だけを処理しているときは、区間の前後で歌われる行を無理に当てはめず「未整列」にする
lines_raw, words_raw, align_info = la.align_lyrics(VOCALS, lyr["text"], lyr["language"], DEVICE, refine=True,
                                                   free_start=inp.get("cut_start", False),
                                                   free_end=inp.get("cut_end", False))
free_gpu()
save_json("align_raw.json", {"lines": lines_raw, "words": words_raw, "info": align_info})
print(align_info)
df_lines = pd.DataFrame([{"line": l["index"], "section": l.get("section"), "start": l.get("start"), "end": l.get("end"),
                          "score": round(l.get("score", float("nan")), 2), "text": l["text"]} for l in lines_raw])
display(df_lines.style.apply(lambda r: ["background-color: #fdd" if (r["score"] < -2.5 or pd.isna(r["start"])) else ""
                                        for _ in r], axis=1))
viz.plot_alignment(VOCALS, lines_raw, words_raw)

In [ ]:
# @title 4-2b. 行ごとに聴いて確認（ドロップダウンで行を選ぶ）
import ipywidgets as W
SONG, VOCALS = load_json("inputs.json")["song"], f"{WORK}/stems/vocals.wav"
_raw = load_json("align_raw.json"); lines_raw, words_raw = _raw["lines"], _raw["words"]
y_full, sr_full = librosa.load(VOCALS, sr=22050, mono=True)
y_mix, _ = librosa.load(SONG, sr=22050, mono=True)
out = W.Output()
opts = [(f"{l['index']}: {l['text'][:30]}", l["index"]) for l in lines_raw if l.get("start") is not None]
dd = W.Dropdown(options=opts, description="line")
src_sel = W.ToggleButtons(options=["vocals", "mix"], value="vocals")

def _play(*_):
    with out:
        out.clear_output()
        l = lines_raw[dd.value]
        a, b = int(max(0, l["start"] - 0.3) * sr_full), int((l["end"] + 0.3) * sr_full)
        y = y_full if src_sel.value == "vocals" else y_mix
        ws = [w for w in words_raw if w["line"] == l["index"]]
        print(f"{l['start']:.2f}-{l['end']:.2f}s  " + " ".join(f"{w['text']}@{w['start']:.2f}" for w in ws if w.get("start") is not None))
        display(Audio(y[a:b], rate=sr_full, autoplay=True))

dd.observe(_play, "value"); src_sel.observe(_play, "value")
display(W.HBox([dd, src_sel]), out)
if opts:
    _play()

In [ ]:
# @title 4-3. タイミングの手動補正（不要ならそのまま実行）
# 行番号: (開始秒, 終了秒)。行の中の単語は比例して伸縮します。
LINE_TIME_OVERRIDES = {
    # 3: (45.20, 48.90),
}
GLOBAL_OFFSET_SEC = 0.0  # 全体を一律にずらす（秒）
EDITED_LINES_CSV = ""    # 下で書き出した lyrics_lines.csv の start/end を編集してアップロードした場合、そのパス

raw = load_json("align_raw.json")
lines, words = raw["lines"], raw["words"]
overrides = dict(LINE_TIME_OVERRIDES)
if EDITED_LINES_CSV:
    ed = pd.read_csv(EDITED_LINES_CSV)
    for _, r in ed.iterrows():
        overrides[int(r["line"])] = (float(r["start"]), float(r["end"]))
lines, words = la.apply_overrides(lines, words, overrides, GLOBAL_OFFSET_SEC)
save_json("lyrics_lines.json", lines)
save_json("lyrics_words.json", words)
pd.DataFrame([{"line": l["index"], "start": l.get("start"), "end": l.get("end"), "text": l["text"]} for l in lines]
             ).to_csv(f"{WORK}/lyrics_lines.csv", index=False)
pd.DataFrame(words).to_csv(f"{WORK}/lyrics_words.csv", index=False)
print("saved: lyrics_lines.json / lyrics_words.json (CSV: lyrics_lines.csv, lyrics_words.csv)")
display(pd.DataFrame([{"line": l["index"], "start": l.get("start"), "end": l.get("end"), "manual": l.get("manual", False),
                       "text": l["text"]} for l in lines]))

### 4-4. 工程 3：beat_this でビートとダウンビートを検出

In [ ]:
# @title 4-4. ビート検出
BEAT_TRACKER = "beat_this"  # @param ["beat_this", "librosa"]
# "librosa" は beat_this の重みがどうしても取得できないときの最終手段（ダウンビートの精度が下がります）
SONG = load_json("inputs.json")["song"]; lines = load_json("lyrics_lines.json")
if BEAT_TRACKER == "beat_this":
    beats, downbeats = aa.detect_beats(SONG, checkpoint="final0", device=DEVICE)
else:
    print("⚠ librosa でビートを検出します（beat_this の代替）。4-5 のクリック音で 8 カウントを必ず確認してください。")
    beats, downbeats = aa.detect_beats_librosa(SONG)
free_gpu()
save_json("beats.json", {"beats": beats.tolist(), "downbeats": downbeats.tolist()})
print(f"beats: {len(beats)}, downbeats: {len(downbeats)}, tempo ≈ {aa.tempo_from_beats(beats):.1f} BPM")
first_vocal = min([l["start"] for l in lines if l.get("start") is not None] or [0.0])
viz.plot_beats(SONG, beats, downbeats, start=max(0.0, first_vocal - 5), duration=30)

### 4-5. 工程 4：8 カウントのグリッド

- 4/4 拍子の 2 小節 = 8 拍を 1 単位（エイト）とし、ダウンビート（小節の 1 拍目）から数え始めます。裏拍は「&」です。
- どちらの小節を「1」にするか（偶数小節 / 奇数小節）は、**歌詞の行の頭がカウント 1（またはその直前の 8・8&）に来る**ように自動で選びます。`GRID_PHASE` で固定もできます。
- 変拍子や検出漏れでカウント 1・5 以外にダウンビートが来た場合は、そこで新しいエイトを始めます（表の `irregular`）。
- クリック音つきで聴いて確認できます（高い音 = 各エイトの 1、中くらい = 各拍、小さい音 = 裏拍）。

In [ ]:
# @title 4-5. 8 カウントの作成と確認
GRID_PHASE = "auto"  # @param ["auto", "0", "1"]
CLICK_PREVIEW_START_SEC = -1.0  # @param {type:"number"}
CLICK_PREVIEW_SEC = 30.0  # @param {type:"number"}

bj = load_json("beats.json"); SONG = load_json("inputs.json")["song"]
lines = load_json("lyrics_lines.json"); words = load_json("lyrics_words.json")
DURATION = load_json("inputs.json")["duration"]
grid = aa.build_grid(bj["beats"], bj["downbeats"], DURATION,
                     line_starts=[l["start"] for l in lines if l.get("start") is not None],
                     phase=GRID_PHASE if GRID_PHASE == "auto" else int(GRID_PHASE))
grid = aa.attach_lyrics(grid, lines, words)
save_json("grid.json", grid)
save_json("lyrics_lines.json", lines)
print(f"{len(grid['eights'])} eights, {grid['bpm']:.1f} BPM, phase={grid['phase']} (scores {grid['phase_scores']}), "
      f"pickup beats before the first eight: {grid['pickup_beats']}")
display(pd.DataFrame(aa.grid_table(grid)))
start = CLICK_PREVIEW_START_SEC if CLICK_PREVIEW_START_SEC >= 0 else max(0.0, (lines[0].get("start") or 0) - 4)
y_click, sr_click = viz.click_track(SONG, grid, start=start, duration=CLICK_PREVIEW_SEC)
display(Audio(y_click, rate=sr_click))

## 5. 工程 5：LLM で振付台本を作る

入力は歌詞・単語のタイミング・8 カウント表・ジャンルです。LLM は、意味の強い箇所（サビなど）を優先して**一部の行だけ**にジェスチャーを置き、各ジェスチャーについて次を返します。

- `motion_description`: MotionGPT3 向けの英語の動作記述（HumanML3D の書き方: "a person raises both arms ..."）
- `eight` / `start_count` / `hit_count` / `end_count`: 配置スロットと「決め」の拍
- `scope`: `upper`（腕・手・上体で表す）または `full`（跳ぶ・しゃがむ・回るなど全身）

JSON として読めない・スロットが存在しないなどの場合は、理由を添えて自動で再試行します。間隔が近すぎるもの・上限を超えたものは優先度の低い順に外します。

In [ ]:
# @title 5-1. 振付台本の生成
MAX_GESTURES = 0  # @param {type:"integer"}
# 0 = 自動（エイト数 ÷ 5、3〜12 個）
MIN_GAP_BEATS = 8  # @param {type:"integer"}
LLM_TEMPERATURE = 0.7  # @param {type:"number"}
EXTRA_INSTRUCTIONS = ""  # @param {type:"string"}

grid = load_json("grid.json"); lines = load_json("lyrics_lines.json")
GENRE = load_json("inputs.json")["genre"]
if "OPENROUTER_API_KEY" not in globals():  # ランタイム再起動後
    from google.colab import userdata
    OPENROUTER_API_KEY = userdata.get("OPENROUTER_API_KEY")
    OPENROUTER_MODEL = globals().get("OPENROUTER_MODEL", "anthropic/claude-sonnet-4.5")
max_g = MAX_GESTURES if MAX_GESTURES > 0 else int(np.clip(len(grid["eights"]) // 5, 3, 12))
script = cl.generate_script(lines, grid, GENRE, OPENROUTER_API_KEY, OPENROUTER_MODEL, aa.count_time,
                            max_gestures=max_g, min_gap_beats=MIN_GAP_BEATS, temperature=LLM_TEMPERATURE,
                            extra_instructions=EXTRA_INSTRUCTIONS)
save_json("script.json", script)
print("concept:", script["concept"])
print(f"{len(script['gestures'])} gestures (max {max_g}), LLM attempts: {script['attempts']}")
display(pd.DataFrame([{k: g[k] for k in ("id", "hit_time", "eight", "start_count", "hit_count", "end_count", "scope",
                                          "priority", "lyric", "meaning", "motion_description")}
                      for g in script["gestures"]]))

In [ ]:
# @title 5-2. 台本の手直し（任意）
# /content/work/script.json を左のファイル一覧から開いて編集・保存したあと、このセルを実行すると再検証します。
# 例: motion_description を書き換える、scope を変える、eight / hit_count を動かす、不要な要素を消す。
grid = load_json("grid.json"); lines = load_json("lyrics_lines.json")
obj = load_json("script.json")
gestures, warns = cl.validate_script(obj, grid, lines, max_gestures=100, min_gap_beats=globals().get("MIN_GAP_BEATS", 8),
                                     count_time=aa.count_time)
for w in warns:
    print("[note]", w)
obj["gestures"] = gestures
save_json("script.json", obj)
display(pd.DataFrame([{k: g[k] for k in ("id", "hit_time", "eight", "hit_count", "scope", "lyric", "motion_description")}
                      for g in gestures]))

## 6. 工程 6：FlowerDance で曲全体のベースダンスを生成（1 回目）

- 音楽特徴量は FlowerDance の `inference.py` と同じ関数（librosa の 35 次元）で、**元のミックス音源**から計算します。
- FlowerDance の既定の生成長は 32 秒（学習時の系列長は 40 秒）です。Mamba には位置の上限がないので曲全体を 1 回で生成できますが、学習より長い系列で質が落ちる可能性があります。
  - `BASE_MODE="whole"`: 1 回で生成
  - `BASE_MODE="chunked"`: 32 秒ずつ生成し、各区間の先頭（前の区間との重なり）を**厳密マスク**で前の結果に固定して続きを生成（境界がつながる）
  - `BASE_MODE="auto"`: 40 秒以内なら whole、それより長ければ chunked
- `RUN_LENGTH_CHECK=True`（既定）では、曲が 40 秒より長いとき、最初に「一度に生成」と「区間分割」を同じ区間（`LENGTH_CHECK_SEC`）で比べ、時間・メモリ・動きの量（5 秒ごとの関節速度）・足滑りを表示します。確認が済んだら `False` にして構いません。

In [ ]:
# @title 6-1. ベースダンスの生成
BASE_MODE = "auto"  # @param ["auto", "whole", "chunked"]
BASE_STEPS = 21  # @param {type:"integer"}
CHUNK_WINDOW_SEC = 32.0  # @param {type:"number"}
CHUNK_OVERLAP_SEC = 4.0  # @param {type:"number"}
CHUNK_INPAINT_STEPS = 51  # @param {type:"integer"}
RUN_LENGTH_CHECK = True  # @param {type:"boolean"}
LENGTH_CHECK_SEC = 120.0  # @param {type:"number"}

inp = load_json("inputs.json"); SONG, GENRE, SEED = inp["song"], inp["genre"], inp["seed"]
FD_CKPT = env_state()["FD_CKPT"]; MAMBA_BACKEND = env_state().get("MAMBA_BACKEND", "cuda")
fd = fr.FlowerDance(FD_REPO, FD_CKPT, DEVICE, MAMBA_BACKEND)
feat = fd.music_features(SONG)
cond = feat["features"]
save_pt("cond.pt", {"cond": cond, "fps": feat["fps"], "bpm_librosa": feat["bpm_librosa"]})
print(f"music features: {cond.shape} ({cond.shape[0] / su.FPS:.1f} s)")
win, ovl = int(CHUNK_WINDOW_SEC * su.FPS), int(CHUNK_OVERLAP_SEC * su.FPS)

if RUN_LENGTH_CHECK and len(cond) > fr.TRAIN_HORIZON:
    seg = cond[: int(min(len(cond), LENGTH_CHECK_SEC * su.FPS))]
    res = {}
    for mode in ("whole", "chunked"):
        if torch.cuda.is_available():
            torch.cuda.reset_peak_memory_stats()
        xn, info = fd.sample_long(seg, GENRE, mode=mode, n_steps=BASE_STEPS, seed=SEED, window=win, overlap=ovl,
                                  inpaint_steps=CHUNK_INPAINT_STEPS)
        st = fr.motion_stats(fd.decode(xn)["positions"])
        res[mode] = st
        peak = torch.cuda.max_memory_allocated() / 2**30 if torch.cuda.is_available() else float("nan")
        print(f"{mode:8s}: {info['seconds']:.1f}s, peak VRAM {peak:.2f} GB, "
              f"mean speed {st['mean_speed']:.2f} m/s, foot skate {st['foot_skate_mps']:.3f} m/s")
    viz.plot_speed_windows([res["whole"], res["chunked"]], ["whole", "chunked"])
    print("→ whole の後半で動きが小さくなる・崩れる場合は BASE_MODE='chunked' を使ってください。")

base_norm, info = fd.sample_long(cond, GENRE, mode=BASE_MODE, n_steps=BASE_STEPS, seed=SEED, window=win, overlap=ovl,
                                 inpaint_steps=CHUNK_INPAINT_STEPS)
base = fd.decode(base_norm)
stats = fr.motion_stats(base["positions"])
print(f"base dance: mode={info['mode']}, {info['seconds']:.1f}s, windows={info['windows']}")
print(f"mean speed {stats['mean_speed']:.2f} m/s, foot skate {stats['foot_skate_mps']:.3f} m/s, root travel {stats['root_travel_m']:.1f} m")
if stats["up_axis_cos"] < 0.8:
    print(f"[warn] 体の上方向が +z と一致しません (cos={stats['up_axis_cos']:.2f})。ジェスチャーの埋め込みは z-up を前提にしています。")
save_pt("base.pt", {"base_norm": base_norm, "info": info})
er.export_smpl_pkl(base, f"{WORK}/base_motion_zup.pkl")
viz.plot_speed_windows([stats], ["base"])
fd.free(); del fd; free_gpu()

In [ ]:
# @title 6-2. ベースダンスの骨格プレビュー（任意）
PREVIEW_BASE_START_SEC = 0.0  # @param {type:"number"}
PREVIEW_BASE_SEC = 20.0  # @param {type:"number"}
SONG = load_json("inputs.json")["song"]
codec = fr.FlowerDance(FD_REPO, env_state()["FD_CKPT"], "cpu", load_model=False)  # 正規化器だけ読む
base = codec.decode(load_pt("base.pt")["base_norm"])
p = er.render_skeleton_video(f"{WORK}/preview_base.mp4", base["positions"], base["rotmat"][:, 0], audio=SONG,
                             start_frame=int(PREVIEW_BASE_START_SEC * su.FPS), n_frames=int(PREVIEW_BASE_SEC * su.FPS),
                             width=480, height=480)
show_video(p)

## 7. 工程 7〜8：歌詞ジェスチャーの生成と埋め込み

### 工程 7：MotionGPT3 でジェスチャーを生成し、「決め」を切り出す
1. 同じ記述から `N_CANDIDATES` 個の候補を生成します（MotionGPT3 は別プロセス・専用ライブラリで実行）。
2. 手首・肘（`full` は足首と腰も）の速度の**極小点**のうち、直前に大きな動きがあり腕が伸びているものを「決め」とし、候補間で最も明確なものを選びます。
3. 決めの前後（`PRE_BEATS` / `POST_BEATS` 拍、既定 1 拍）を切り出します。
4. **形式変換（最大の難所）**: HumanML3D（22 関節の位置、y-up、20fps）→ FlowerDance（SMPL 24 関節の回転＋ルート位置、30fps）。
   - 20fps → 30fps は関節位置の 3 次スプライン補間
   - 逆運動学: 骨盤と胸は複数の骨方向への Kabsch 当てはめ、それ以外は骨方向への最小回転で初期解 → 骨方向誤差・肘と膝の蝶番（ヒンジ）制約・逆曲がり防止・時間方向の滑らかさで最適化
   - 骨の長さの違いは方向だけを合わせることで吸収し、ルート移動は脚の長さの比で拡大縮小
   - 手首・手・頭は位置情報がないため、ベースダンスのものを残します
5. 可動域を少し誇張（`EXAGGERATION`、候補の最初のフレーム＝自然立ちを基準に回転量を拡大）してダンスらしさを補います。

変換前後の関節位置の誤差は 7-3 で図示します。

In [ ]:
# @title 7-1. MotionGPT3 でジェスチャー候補を生成
N_CANDIDATES = 4  # @param {type:"integer"}
GESTURE_GEN_SEC = 4.0  # @param {type:"number"}

script = load_json("script.json"); SEED = load_json("inputs.json")["seed"]
MGPT_CKPT = env_state()["MGPT_CKPT"]
GEST_DIR = f"{WORK}/gestures_raw"
os.makedirs(GEST_DIR, exist_ok=True)
gp.write_requests(script["gestures"], f"{WORK}/gesture_requests.json", n_candidates=N_CANDIDATES,
                  seconds=GESTURE_GEN_SEC)
run_streaming([sys.executable, f"{PKG}/m2d/mgpt_runner.py", "generate", "--repo", MGPT_REPO, "--ckpt", MGPT_CKPT,
               "--requests", f"{WORK}/gesture_requests.json", "--out", GEST_DIR, "--seed", str(SEED)], env=mgpt_env())
free_gpu()
print(sorted(os.listdir(GEST_DIR)))

In [ ]:
# @title 7-2. 決めの検出・切り出し・SMPL への変換・誇張
PRE_BEATS = 1.0  # @param {type:"number"}
POST_BEATS = 1.0  # @param {type:"number"}
EXAGGERATION = 1.15  # @param {type:"number"}
IK_ITERS = 300  # @param {type:"integer"}

script = load_json("script.json"); grid = load_json("grid.json")
GEST_DIR = f"{WORK}/gestures_raw"
processed = []
for g in script["gestures"]:
    J, fps = gp.load_candidates(GEST_DIR, g["id"])
    bp = grid["eights"][g["eight"]]["beat_period"]
    r = gp.process_gesture(g, J, bp, pre_beats=PRE_BEATS, post_beats=POST_BEATS, exaggeration=EXAGGERATION,
                           ik_iters=IK_ITERS, device=DEVICE)
    r["label"] = g["lyric"][:20] + " / " + g["motion_description"][:40]
    processed.append(r)
save_pt("gestures.pt", [{k: v for k, v in r.items() if k != "debug"} for r in processed])
display(pd.DataFrame([{"id": r["id"], "scope": r["scope"], "hit_s": round(r["hit_frame"] / su.FPS, 2),
                       "candidate": r["info"]["chosen"], "key_in_cand": r["info"]["key_frame_in_candidate"],
                       "clip_frames": r["rotmat"].shape[0], "IK angle mean°": round(r["info"]["ik_bone_angle_deg_mean"], 2),
                       "IK angle max°": round(r["info"]["ik_bone_angle_deg_max"], 2),
                       "pos err mean cm": round(r["info"]["ik_pos_err_cm_mean"], 2),
                       "pos err max cm": round(r["info"]["ik_pos_err_cm_max"], 2),
                       "mirrored": r["info"]["mirrored"], "label": r["label"]} for r in processed]))

In [ ]:
# @title 7-3. 変換の確認（決めの検出・関節ごとの誤差・変換前後の骨格）
# 7-2 を同じセッションで実行した直後に使います（詳細な途中結果はメモリ上にだけあります）。
INSPECT_GESTURE_ID = 0  # @param {type:"integer"}
r = next(x for x in processed if x["id"] == INSPECT_GESTURE_ID)
viz.plot_gesture_debug(r)
conv = r["debug"]["conv"]
print(f"gesture #{r['id']}: analytic IK mean error {conv['errors_analytic']['pos_err_m'].mean() * 100:.2f} cm "
      f"→ refined {conv['errors']['pos_err_m'].mean() * 100:.2f} cm (scale {conv['scale']:.3f}, mirrored {conv['mirrored']})")
# 変換後の候補全体（切り出し前）を骨格動画で確認
R_full = r["debug"]["rotmat_full"]
rot0, pos0 = su.yup_to_zup(R_full[:, 0], conv["root_pos"])
Rz = R_full.clone(); Rz[:, 0] = rot0
p = er.render_skeleton_video(f"{WORK}/preview_gesture_{r['id']}.mp4", su.fk(Rz, pos0), Rz[:, 0], width=360, height=360)
show_video(p, width=360)

### 工程 8：ジェスチャーを拍に合わせてベースダンスに埋め込む

- 決めのフレームが台本の `hit_count` の時刻にぴったり来るように置きます。
- `upper`: 上半身（背骨・首・鎖骨・肩・肘）の回転チャンネルだけを差し替えます。
- `full`: 全関節を差し替えます。ルートは `FULL_ROOT_MODE` で選びます。
  - `"base"`: ルートの位置と向きをベースダンスのものにする（計画書どおり）
  - `"hybrid"`（既定）: **立ち位置（水平位置）と向きはベースダンス**に合わせつつ、ジェスチャー自身の高さの変化（跳ぶ・しゃがむ）、体の傾き、決めからの回転量（回る）を上乗せする
- マスク（工程 9 で使う）:
  - ジェスチャー区間の差し替えたチャンネル → 指定（固定）
  - ジェスチャーの前後 `TRANSITION_BEATS` 拍 → 生成し直す（つなぎと予備動作）
  - それ以外 → ベースダンスを指定（固定）
- `GESTURE_MASK_MODE`: `"soft"` は FlowerDance 標準の**時間減衰ソフトマスク**（生成の終盤ほど拘束が弱まる）、`"strict"` は減衰しない厳密マスク。ジェスチャーが薄まる場合は `"strict"` に変える**1 行の変更**で切り替えられます。
- マスク付き生成は、ジェスチャーの周囲だけの「ウィンドウ」ごとに行います。ウィンドウの両端 `EDGE_LOCK_FRAMES` フレームは常に厳密に固定するので、ウィンドウの外のベースダンスと継ぎ目なくつながります。

In [ ]:
# @title 7-4. 埋め込みとマスクの作成（工程 8）
FULL_ROOT_MODE = "hybrid"  # @param ["hybrid", "base"]
TRANSITION_BEATS = 1.5  # @param {type:"number"}
TRANSITION_REGEN = "replaced"  # @param ["replaced", "all"]
GESTURE_MASK_MODE = "soft"  # @param ["soft", "strict"]
BASE_MASK_MODE = "soft"  # @param ["soft", "strict"]
CONTEXT_SEC = 3.0  # @param {type:"number"}
MIN_WINDOW_SEC = 8.0  # @param {type:"number"}
EDGE_LOCK_FRAMES = 10  # @param {type:"integer"}

grid = load_json("grid.json")
codec = fr.FlowerDance(FD_REPO, env_state()["FD_CKPT"], "cpu", load_model=False)
base_norm = load_pt("base.pt")["base_norm"]
base = codec.decode(base_norm)
gestures = load_pt("gestures.pt")
T = base_norm.shape[0]
trans = int(round(TRANSITION_BEATS * grid["beat_period"] * su.FPS))
assert EDGE_LOCK_FRAMES <= CONTEXT_SEC * su.FPS, "EDGE_LOCK_FRAMES は CONTEXT_SEC 以下にしてください"
assembled, mask, decay, regions = ge.assemble(base, gestures, trans, full_root_mode=FULL_ROOT_MODE,
                                              gesture_mask_mode=GESTURE_MASK_MODE, base_mask_mode=BASE_MASK_MODE,
                                              transition_regen=TRANSITION_REGEN)
labels = {g["id"]: g.get("label", "") for g in gestures}
for r in regions:
    r["label"] = labels.get(r["id"], "")
windows = ge.edit_windows(regions, T, int(CONTEXT_SEC * su.FPS), int(MIN_WINDOW_SEC * su.FPS), EDGE_LOCK_FRAMES)
mask_w, decay_w = ge.lock_window_edges(mask, decay, windows, EDGE_LOCK_FRAMES, T)
edited_norm, clip_frac = ge.edited_vector(codec, base_norm, assembled)
print(f"{len(regions)} gestures placed, transition {trans} frames, {len(windows)} windows: "
      + ", ".join(f"{s / su.FPS:.1f}-{e / su.FPS:.1f}s" for s, e in windows))
if clip_frac > 0.05:
    print(f"[note] gesture values at the edge of FlowerDance's training range: {clip_frac * 100:.1f}% "
          "(they are clipped by the normaliser; lower EXAGGERATION if the poses look flattened)")
save_pt("edit_inputs.pt", {"assembled": assembled, "mask": mask_w, "decay": decay_w, "regions": regions,
                           "windows": windows, "edited_norm": edited_norm,
                           "settings": {"FULL_ROOT_MODE": FULL_ROOT_MODE, "TRANSITION_BEATS": TRANSITION_BEATS,
                                        "GESTURE_MASK_MODE": GESTURE_MASK_MODE, "BASE_MASK_MODE": BASE_MASK_MODE,
                                        "TRANSITION_REGEN": TRANSITION_REGEN}})
viz.plot_mask_timeline(mask_w, regions, windows)

## 8. 工程 9：FlowerDance のマスク付き生成でつなぎを作り直す（2 回目）

付属の `inpaint.py` はデータセットの曲用のデモなので、`inference.py` の特徴量計算と `FlowMatching.inpaint` と同じ計算を組み合わせて自前で実装しています（`m2d/flowerdance_runner.py` の `inpaint`）。
要素ごとに「時間減衰ソフトマスク / 厳密マスク」を選べるようにしてあり、ウィンドウの端は常に厳密に固定します。

生成後、各ジェスチャーが完成モーションにどれだけ残っているか（差し替えた関節の回転のずれ）を表示します。ずれが大きい（目安 20° 以上）ときは 8-1 で `GESTURE_MASK_MODE = "strict"` にして 7-4 と 8-1 をやり直してください。

In [ ]:
# @title 8-1. マスク付き生成
INPAINT_STEPS = 101  # @param {type:"integer"}

inp = load_json("inputs.json"); GENRE, SEED = inp["genre"], inp["seed"]
ed = load_pt("edit_inputs.pt"); cond = load_pt("cond.pt")["cond"]; base_norm = load_pt("base.pt")["base_norm"]
fd = fr.FlowerDance(FD_REPO, env_state()["FD_CKPT"], DEVICE, env_state().get("MAMBA_BACKEND", "cuda"))
t0 = time.time()
final_norm = ge.run_edit(fd, base_norm, ed["edited_norm"], ed["mask"], ed["decay"], ed["windows"], cond, GENRE,
                         n_steps=INPAINT_STEPS, seed=SEED + 7)
print(f"masked generation: {time.time() - t0:.1f}s")
final = fd.decode(final_norm)
fd.free(); del fd; free_gpu()
save_pt("final.pt", {"final_norm": final_norm})

joints_by_scope = {"upper": su.UPPER_JOINTS_DEFAULT, "full": su.FULL_JOINTS_DEFAULT}
adh = viz.gesture_adherence(final["rotmat"], ed["assembled"]["rotmat"], ed["regions"], joints_by_scope)
display(pd.DataFrame(adh))
if adh and np.mean([a["mean_dev_deg"] for a in adh]) > 20:
    print("→ ジェスチャーが薄まっています。7-4 で GESTURE_MASK_MODE = 'strict' にしてやり直すと、形がそのまま残ります。")
jumps = viz.boundary_jumps(final["positions"], ed["windows"])
print("speed at window borders (m/s):", [round(j["speed"], 2) for j in jumps], "/ median", round(jumps[0]["median_speed"], 2) if jumps else "-")
st_b = fr.motion_stats(fr.FlowerDance(FD_REPO, env_state()["FD_CKPT"], "cpu", load_model=False).decode(base_norm)["positions"])
st_f = fr.motion_stats(final["positions"])
viz.plot_speed_windows([st_b, st_f], ["base", "final"])

## 9. 工程 10：書き出し・プレビュー・ダウンロード

| ファイル | 内容 |
|---|---|
| `motion_zup.pkl` | **FlowerDance の推論出力と同じ形式**: `smpl_poses` (T, 72 軸角), `smpl_trans` (T, 3), `full_pose` (T, 24, 3)。z-up、30fps |
| `motion_yup_aistpp.pkl` | SMPL 本来の y-up に戻したもの（AIST++ 形式: `smpl_poses`, `smpl_trans`, `smpl_scaling`）＋ `full_pose` |
| `motion.bvh` | SMPL 骨格の BVH（y-up、cm）。Blender などで読み込めます |
| `joints_zup.npy` | 24 関節の位置 (T, 24, 3) |
| `base_motion_zup.pkl` | 比較用：ジェスチャーを入れる前のベースダンス |
| `script.json` ほか | 振付台本、歌詞の時刻、8 カウント、ビート、設定 |
| `preview_skeleton.mp4` | 骨格アニメーション（音声つき、薄い骨格 = ベースダンス） |

In [ ]:
# @title 9-1. 書き出しと骨格アニメーション
PREVIEW_START_SEC = 0.0  # @param {type:"number"}
PREVIEW_DURATION_SEC = 0.0  # @param {type:"number"}
SHOW_BASE_GHOST = True  # @param {type:"boolean"}

OUT = f"{WORK}/outputs"
os.makedirs(OUT, exist_ok=True)
codec = fr.FlowerDance(FD_REPO, env_state()["FD_CKPT"], "cpu", load_model=False)
final = codec.decode(load_pt("final.pt")["final_norm"])
base = codec.decode(load_pt("base.pt")["base_norm"])
ed = load_pt("edit_inputs.pt")
er.export_smpl_pkl(final, f"{OUT}/motion_zup.pkl", "zup")
er.export_smpl_pkl(final, f"{OUT}/motion_yup_aistpp.pkl", "yup")
er.export_bvh(final, f"{OUT}/motion.bvh")
np.save(f"{OUT}/joints_zup.npy", final["positions"].numpy().astype(np.float32))
er.export_smpl_pkl(base, f"{OUT}/base_motion_zup.pkl", "zup")
for name in ("script.json", "lyrics_lines.json", "lyrics_words.json", "grid.json", "beats.json", "inputs.json",
             "lyrics_lines.csv", "lyrics_words.csv"):
    if os.path.exists(f"{WORK}/{name}"):
        shutil.copy(f"{WORK}/{name}", f"{OUT}/{name}")
json.dump({"settings": ed["settings"], "regions": [{k: v for k, v in r.items() if k != "channels"} for r in ed["regions"]],
           "windows": ed["windows"]}, open(f"{OUT}/edit_summary.json", "w"), ensure_ascii=False, indent=1, default=float)

lines = load_json("lyrics_lines.json"); grid = load_json("grid.json")
s0 = int(PREVIEW_START_SEC * su.FPS)
n = None if PREVIEW_DURATION_SEC <= 0 else int(PREVIEW_DURATION_SEC * su.FPS)
overlay = er.make_overlay_fn(grid, lines, ed["regions"], start_frame=s0)
t0 = time.time()
p = er.render_skeleton_video(f"{OUT}/preview_skeleton.mp4", final["positions"], final["rotmat"][:, 0],
                             audio=load_json("inputs.json")["song"], start_frame=s0, n_frames=n, overlay=overlay,
                             ghost_positions=base["positions"] if SHOW_BASE_GHOST else None,
                             progress=lambda i, k: print(f"  {i}/{k} frames"))
print(f"skeleton preview: {time.time() - t0:.0f}s")
show_video(p)

In [ ]:
# @title 9-2. 成果物を ZIP でダウンロード
zip_path = shutil.make_archive(f"{WORK}/music2dance_outputs", "zip", f"{WORK}/outputs")
print(zip_path, f"{os.path.getsize(zip_path) / 2**20:.1f} MB")
from google.colab import files
files.download(zip_path)

## 10. 工程 11：プレビュー動画（マネキンでダンスを描写）

関節位置からマネキン（丸みのある円錐と球を滑らかにつないだ人形）を作り、PyTorch の GPU レイマーチングで描画します（追加のレンダラーや SMPL のモデルファイルは不要）。床の市松模様と柔らかい影、歌詞・カウント・ジェスチャー名の表示つきです。

In [ ]:
# @title 10-1. マネキン動画
MANNEQUIN_RES = 640  # @param {type:"integer"}
MANNEQUIN_START_SEC = 0.0  # @param {type:"number"}
MANNEQUIN_DURATION_SEC = 0.0  # @param {type:"number"}
SHOW_OVERLAY = True  # @param {type:"boolean"}

codec = fr.FlowerDance(FD_REPO, env_state()["FD_CKPT"], "cpu", load_model=False)
final = codec.decode(load_pt("final.pt")["final_norm"])
ed = load_pt("edit_inputs.pt"); lines = load_json("lyrics_lines.json"); grid = load_json("grid.json")
s0 = int(MANNEQUIN_START_SEC * su.FPS)
n = None if MANNEQUIN_DURATION_SEC <= 0 else int(MANNEQUIN_DURATION_SEC * su.FPS)
overlay = er.make_overlay_fn(grid, lines, ed["regions"], start_frame=s0) if SHOW_OVERLAY else None
t0 = time.time()
p = er.render_mannequin_video(f"{WORK}/outputs/preview_mannequin.mp4", final, audio=load_json("inputs.json")["song"],
                              start_frame=s0, n_frames=n, overlay=overlay, width=MANNEQUIN_RES, height=MANNEQUIN_RES,
                              device=DEVICE, progress=lambda i, k: print(f"  {i}/{k} frames"))
free_gpu()
print(f"mannequin preview: {time.time() - t0:.0f}s")
show_video(p)

In [ ]:
# @title 10-2. マネキン動画を含めて ZIP をダウンロード
zip_path = shutil.make_archive(f"{WORK}/music2dance_outputs", "zip", f"{WORK}/outputs")
print(zip_path, f"{os.path.getsize(zip_path) / 2**20:.1f} MB")
from google.colab import files
files.download(zip_path)

## 困ったとき

| 症状 | 対処 |
|---|---|
| 1-3 で「ビルド済みホイールが見つかりません」 | Colab の Python / PyTorch の組み合わせにホイールがありません。自動で純 PyTorch 版 Mamba に切り替わります（生成が遅くなるだけで結果は同等）。 |
| 6-1 で GPU メモリ不足 | `BASE_MODE="chunked"`、または `CHUNK_WINDOW_SEC` を小さくします。 |
| 曲の後半で動きが小さくなる | `RUN_LENGTH_CHECK=True` で比較し、`BASE_MODE="chunked"` を使います。 |
| 歌詞のタイミングがずれる | 4-2b で聴いて確認し、4-3 の `LINE_TIME_OVERRIDES` で行の開始・終了を直します。 |
| 8 カウントの「1」がずれている | 4-5 で `GRID_PHASE` を `0` / `1` に固定します。 |
| ジェスチャーが薄い・見えない | 7-4 で `GESTURE_MASK_MODE="strict"`、または `EXAGGERATION` を上げます。 |
| ジェスチャーが不自然・跳ねる | 7-3 で変換結果を確認し、5-2 で `motion_description` を具体的に書き直すか、`scope` を `upper` にします。 |
| 2-1 / 4-4 で beat_this の重みが取得できない | 作者サーバーとミラー 2 か所がすべて失敗した場合です。時間をおいて再実行するか、表示されたいずれかの URL から `final0.ckpt` をブラウザで取得して `/content/models/torch/hub/checkpoints/beat_this-final0.ckpt` にアップロードします。最終手段は 4-4 の `BEAT_TRACKER="librosa"` です。 |
| OpenRouter のモデルが無い | 3-3 に表示される候補から `OPENROUTER_MODEL` を選び直します。 |
| MotionGPT3 のダウンロードに失敗 | Hugging Face の `OpenMotionLab/motiongpt3` から `.ckpt` を手動で取得し、`/content/models/motiongpt3/` に置いてから 2-2 を再実行します。 |

**ライセンス**: FlowerDance・MotionGPT3・Demucs・beat_this・WhisperX・wav2vec2 の各モデル／コードはそれぞれのライセンスに従ってください。同梱した PyTorch3D の回転変換は BSD ライセンスです。